In [ ]:
# National models of the thesis (Case Study, Model Development): AR, VAR, AR + Copula and VAR + Copula for the
# national hourly wind speed, air temperature and irradiance of Germany (box 47-55 N, 5-16 E), fitted to ERA5 2000-2024
#
# Sections:
#   1-4      imports, settings, data loading, climatology and anomalies
#   5        model AR: one autoregression per variable, residuals bootstrapped per variable
#   6        model VAR: one vector autoregression across the three variables, Gaussian innovations
#   7        model AR + Copula: seasonal autoregressions and margins, lag-1 copula of irradiance
#   8        Monte Carlo ensembles of each model
#   9-17     error diagnostics by season and tail, confidence envelopes, Dunkelflaute validation and its causes
#   18-21    spatial disaggregation of wind (north and south)
#   22-23    significance tests and Monte Carlo convergence
#   24-25    Dunkelflaute events per year; temporal and spatial dependence of all three variables
#   26-31    spatial disaggregation of temperature and irradiance, spatial copulas
#   32-36    further compound scenarios and regional granularity (1, 2, 4 and 16 regions)
#   37-40    spatial vine copula (4 regions) and cross-variable tail dependence
#   41       persistence of Dunkelflaute episodes
#   42-43    variants D and E of the wind reconstruction
#   44       model VAR + Copula: seasonal VAR with copula innovations
#
# Needs: ERA5 / PECD 4.2 files in data/ERA-5-2000-2013, data/ERA-5-2014-2025 and data/ERA-5-extreme-years (README.md);
#        the 16-cell series of analysis/load_4x4_grid.py and, for Sections 36-37, the regional simulations of
#        analysis/simulate_4_16_regions.py and analysis/simulate_4region_vine.py;
#        vectorized_simulation.py and vectorized_solar_temporal.py (same folder)
# Run: jupyter nbconvert --to notebook --execute --inplace ar_var_copula_extremes.ipynb   (about 1 hour)

In [ ]:
# 1. Imports

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from scipy.stats import rankdata, norm, t
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.ar_model import ar_select_order
from statsmodels.tsa.api import VAR

import pyvinecopulib as pv

pd.set_option("display.width", 160)


In [ ]:
# 2. Settings: data folders, training period 2000-2024, test year 2025, extreme test year 1963,
#    transformations of wind and irradiance

base_dir = "./data/ERA-5-2014-2025"

# 2000-2013 in a separate folder, with a different filename suffix
pre2014_dir = "./data/ERA-5-2000-2013"
pre2014_suffix = ".area-subset.75.45.18.-31"
PRE2014_CUTOFF = 2014

train_years = range(2000, 2025)   # 2000-2024
test_year = 2025

extreme_year = 1963               # out-of-sample test year (outside 2000-2024): coldest winter on record in
                                   # Germany, with a persistent blocking anticyclone
extreme_year_dir = "./data/ERA-5-extreme-years"
extreme_year_suffix = ".area-subset.75.45.18.-31"  # different filename suffix

daily_mean_cache_path = "./data/ar_var_copula_daily_means_cache.npz"

variables = ["wind", "temp", "solar"]
eps = 1e-6

# wind in Box-Cox space (WIND_LAMBDA), imported from vectorized_simulation.py so that both use the same value
import sys
sys.path.insert(0, ".")
from vectorized_simulation import WIND_LAMBDA, wind_fwd, wind_inv

# irradiance in log space, log(solar + SOLAR_LOG_EPS); SOLAR_LOG_EPS keeps the logarithm finite at night
SOLAR_LOG_EPS = 1.0  # W/m2

REGION = {
    "lat_min": 47,
    "lat_max": 55,
    "lon_min": 5,
    "lon_max": 16,
}

AR_MAX_LAG = 168
N_TRAJECTORIES = 300
BASE_SEED = 42

SEASONS = ["winter", "spring", "summer", "autumn"]


def get_season(month):
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"


In [ ]:
# 3. Data loading: national means of 10 m wind, 2 m temperature and irradiance over the study box

def load_year(var_code, var_name, year, mbc_or_noc, source_dir=None, filename_suffix=""):

    source_dir = base_dir if source_dir is None else source_dir

    filename = (
        f"H_ERA5_ECMW_T639_{var_code}_Pecd_025d_"
        f"S{year}01010000_E{year}12312300_INS_MAP_01h_COM_"
        f"{mbc_or_noc}_org_NA_NA---_NA---_NA---_PECD4.2_fv1{filename_suffix}.nc"
    )

    path = os.path.join(source_dir, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with xr.open_dataset(path) as ds:

        data = ds[var_name]

        latitude = data["latitude"].values

        lat_slice = (
            slice(REGION["lat_max"], REGION["lat_min"])
            if latitude[0] > latitude[-1]
            else slice(REGION["lat_min"], REGION["lat_max"])
        )

        ts = (
            data
            .sel(latitude=lat_slice, longitude=slice(REGION["lon_min"], REGION["lon_max"]))
            .mean(dim=["latitude", "longitude"])
            .load()
        )

    return ts


wind_list, temp_list, solar_list = [], [], []

for year in [*train_years, test_year]:
    print(f"Loading {year}")
    if year < PRE2014_CUTOFF:
        src_dir, suffix = pre2014_dir, pre2014_suffix
    else:
        src_dir, suffix = base_dir, ""
    wind_list.append(load_year("WS-_0010m", "ws10", year, "mbc", source_dir=src_dir, filename_suffix=suffix))
    temp_list.append(load_year("TA-_0002m", "t2m", year, "noc", source_dir=src_dir, filename_suffix=suffix))
    solar_list.append(load_year("GHI_0000m", "ssrd", year, "noc", source_dir=src_dir, filename_suffix=suffix))

wind_ts = xr.concat(wind_list, dim="time")
temp_ts = xr.concat(temp_list, dim="time")
solar_ts = xr.concat(solar_list, dim="time")

df_all = pd.DataFrame(
    {"wind": wind_ts.values, "temp": temp_ts.values, "solar": solar_ts.values},
    index=pd.to_datetime(wind_ts["time"].values),
).sort_index()

df_train = df_all[df_all.index.year < test_year].copy()
df_test = df_all[df_all.index.year == test_year].copy()

print(f"Loading extreme year {extreme_year} (out-of-sample, from {extreme_year_dir})")

wind_extreme = load_year("WS-_0010m", "ws10", extreme_year, "mbc",
                          source_dir=extreme_year_dir, filename_suffix=extreme_year_suffix)
temp_extreme = load_year("TA-_0002m", "t2m", extreme_year, "noc",
                          source_dir=extreme_year_dir, filename_suffix=extreme_year_suffix)
solar_extreme = load_year("GHI_0000m", "ssrd", extreme_year, "noc",
                           source_dir=extreme_year_dir, filename_suffix=extreme_year_suffix)

df_extreme = pd.DataFrame(
    {"wind": wind_extreme.values, "temp": temp_extreme.values, "solar": solar_extreme.values},
    index=pd.to_datetime(wind_extreme["time"].values),
).sort_index()

print("Training:", df_train.shape, df_train.index.min(), df_train.index.max())
print("Test:", df_test.shape, df_test.index.min(), df_test.index.max())
print("Extreme year:", df_extreme.shape, df_extreme.index.min(), df_extreme.index.max(), "(out-of-sample)")


In [ ]:
# 4. Climatology and anomalies: linear trend and smoothed (month, day, hour) climatology per variable, anomalies
#    for the model fits, and one offset per training year for the year bootstrap

CLIMATOLOGY_SMOOTHING_WINDOW_DAYS = 15

df_train["month"] = df_train.index.month
df_train["day"] = df_train.index.day
df_train["hour"] = df_train.index.hour

# irradiance in log space for the model fits (df_train_solar_log); df_train stays in W/m2 for the comparisons
# with the observed data

df_train_solar_log = df_train.copy()
df_train_solar_log["solar"] = np.log(df_train["solar"] + SOLAR_LOG_EPS)
df_train_solar_log["wind"] = wind_fwd(df_train["wind"].to_numpy())


def build_smoothed_climatology(df_train, variables, window_days=15):

    raw = df_train.groupby(["month", "day", "hour"])[variables].mean()

    days_in_month = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    calendar_order = [(m, d) for m in range(1, 13) for d in range(1, days_in_month[m - 1] + 1)]

    smoothed_blocks = []

    for hour in range(24):
        vals = raw.xs(hour, level="hour").reindex(calendar_order)[variables].to_numpy(dtype=float)
        padded = np.concatenate([vals[-window_days:], vals, vals[:window_days]], axis=0)
        smoothed_vals = (
            pd.DataFrame(padded)
            .rolling(2 * window_days + 1, center=True, min_periods=1)
            .mean()
            .to_numpy()[window_days:-window_days]
        )
        block = pd.DataFrame(smoothed_vals, columns=variables)
        block["month"] = [m for m, d in calendar_order]
        block["day"] = [d for m, d in calendar_order]
        block["hour"] = hour
        smoothed_blocks.append(block)

    return pd.concat(smoothed_blocks, ignore_index=True).set_index(["month", "day", "hour"])[variables]


def map_climatology(index, climatology):
    cal = pd.DataFrame({"month": index.month, "day": index.day, "hour": index.hour}, index=index)
    mapped = cal.merge(climatology, left_on=["month", "day", "hour"], right_index=True, how="left")[variables]
    mapped.index = index
    return mapped


# linear trend per variable, fitted to a first, roughly deseasonalized anomaly; then the smoothed (month, day,
# hour) climatology of the detrended series (for irradiance in log space); used for the fits only

def fit_linear_trend(df_train, variables, window_days=15):

    rough_climatology = build_smoothed_climatology(df_train, variables, window_days=window_days)
    rough_mean = map_climatology(df_train.index, rough_climatology)
    rough_anom = df_train[variables] - rough_mean

    t_years = (df_train.index - df_train.index[0]).total_seconds() / (365.25 * 24 * 3600)

    trend_coefs = {}
    for col in variables:
        slope, intercept = np.polyfit(t_years, rough_anom[col].to_numpy(), 1)
        trend_coefs[col] = {"slope": slope, "intercept": intercept, "t0": df_train.index[0]}
        print(f"  {col}: trend = {slope:+.4f} /year ({slope*10:+.3f} /decade)")

    return trend_coefs


def trend_value(index, trend_coefs, variables):
    out = pd.DataFrame(index=index)
    for col in variables:
        t0 = trend_coefs[col]["t0"]
        t_years = (index - t0).total_seconds() / (365.25 * 24 * 3600)
        out[col] = trend_coefs[col]["intercept"] + trend_coefs[col]["slope"] * t_years
    return out


print("Fitted linear trends (per variable, on roughly-deseasonalized anomalies; solar is log-space, i.e. /decade means proportional not absolute):")
trend_coefs = fit_linear_trend(df_train_solar_log, variables, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)

df_train_detrended = df_train_solar_log[variables] - trend_value(df_train_solar_log.index, trend_coefs, variables)
df_train_detrended["month"] = df_train_solar_log.index.month
df_train_detrended["day"] = df_train_solar_log.index.day
df_train_detrended["hour"] = df_train_solar_log.index.hour

climatology = build_smoothed_climatology(df_train_detrended, variables, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)

train_mean = map_climatology(df_train_solar_log.index, climatology) + trend_value(df_train_solar_log.index, trend_coefs, variables)
df_anom = df_train_solar_log[variables] - train_mean

print()
print(df_anom.describe())
print("\n(solar column above is in log-space - small, roughly symmetric values are expected, not physical W/m2)")

max_jump = np.diff(climatology.xs(12, level="hour")["temp"].to_numpy()).__abs__().max()
print(f"\nSanity check - max day-to-day jump in smoothed hour=12 temp climatology: {max_jump:.3f} K")

# year bootstrap: the mean anomaly of each training year (wind, temperature and irradiance together) is one
# offset; each trajectory draws one training year and keeps its offset for the whole simulated year (the
# same draw for all models)

year_baseline_offset = df_anom.groupby(df_anom.index.year).mean()
sim_calendar_mean = map_climatology(df_test.index, climatology)

print("\nEmpirical year-baseline offsets (each training year's own deviation from trend+shape):")
print(year_baseline_offset.describe().loc[["min", "mean", "max"]])
print(f"\nTemp offset range across {len(year_baseline_offset)} training years: "
      f"{year_baseline_offset['temp'].min():+.2f} to {year_baseline_offset['temp'].max():+.2f} K "
      f"(span {year_baseline_offset['temp'].max() - year_baseline_offset['temp'].min():.2f} K) "
      "- this is the interannual spread the bootstrap injects into the ensemble.")


In [ ]:
# 5. Model AR: one autoregression per variable (order by BIC), residuals bootstrapped per variable

def fit_ar_models(df_anom, variables, maxlag=168):

    models = {}
    residuals = pd.DataFrame(index=df_anom.index)

    for col in variables:
        print(f"Selecting AR model for {col}")
        selection = ar_select_order(df_anom[col].dropna(), maxlag=maxlag, ic="bic", old_names=False)
        model = selection.model.fit()
        models[col] = model
        residuals[col] = model.resid
        print("  selected lags:", list(model.model._lags))

    return models, residuals.dropna()


def reconstruct_simulation(resid_sim, models, test_mean, train_anom, variables):

    sim_anom = pd.DataFrame(index=test_mean.index, columns=variables, dtype=float)

    for col in variables:
        model = models[col]
        params = model.params
        lags = list(model.model._lags)
        train_history = train_anom[col].dropna().to_numpy(dtype=float)

        intercept = float(params["const"]) if "const" in params.index else 0.0
        n_test = len(test_mean)
        simulated = np.zeros(n_test, dtype=float)

        for i in range(n_test):
            value = intercept
            for lag in lags:
                coefficient = float(params[f"{col}.L{lag}"])
                lagged_value = simulated[i - lag] if i - lag >= 0 else train_history[i - lag]
                value += coefficient * lagged_value
            value += float(resid_sim[col].iloc[i])
            simulated[i] = value

        sim_anom[col] = simulated

    sim_phys = sim_anom + test_mean[variables]
    sim_phys["wind"] = wind_inv(sim_phys["wind"].to_numpy())
    # irradiance back from log space with exp(); the clip only guards against floating-point noise
    sim_phys["solar"] = np.exp(sim_phys["solar"]) - SOLAR_LOG_EPS
    sim_phys["solar"] = sim_phys["solar"].clip(lower=0)

    return sim_phys


def simulate_ar_independent(models, residuals, target_mean, train_anom, variables, seed):

    rng = np.random.default_rng(seed)
    n = len(target_mean)

    resid_sim = pd.DataFrame(index=target_mean.index, columns=variables, dtype=float)
    for col in variables:
        pool = residuals[col].dropna().to_numpy()
        resid_sim[col] = rng.choice(pool, size=n, replace=True)

    return reconstruct_simulation(
        resid_sim=resid_sim, models=models, test_mean=target_mean, train_anom=train_anom, variables=variables
    )


ar_models, ar_residuals = fit_ar_models(df_anom=df_anom, variables=variables, maxlag=AR_MAX_LAG)


In [ ]:
# 6. Model VAR: one vector autoregression across the three variables for the whole year, Gaussian innovations

def fit_var_model(df_anom_train, variables, maxlags=168, criterion="bic", trend="c"):

    train = df_anom_train[variables].replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    model = VAR(train)
    selection = model.select_order(maxlags=maxlags, trend=trend)

    selected_lag = max(1, int(getattr(selection, criterion)))
    results = model.fit(maxlags=selected_lag, trend=trend)

    print(f"Selected VAR order by {criterion.upper()}: {selected_lag}")
    print(f"Stable VAR: {results.is_stable(verbose=False)}")

    return results, selection


def simulate_var_path(var_results, n_steps, index, seed=42, burn_in=500):

    if not var_results.is_stable(verbose=False):
        raise ValueError("The fitted VAR is not stable.")

    rng = np.random.default_rng(seed)

    p = var_results.k_ar
    k = var_results.neqs
    coefs = var_results.coefs
    intercept = var_results.intercept
    sigma_u = np.asarray(var_results.sigma_u)

    total_steps = n_steps + burn_in
    history = np.asarray(var_results.endog[-p:], dtype=float).copy()
    simulated = np.empty((total_steps, k))

    for step in range(total_steps):
        value = intercept.copy()
        for lag in range(1, p + 1):
            value += coefs[lag - 1] @ history[-lag]

        innovation = rng.multivariate_normal(mean=np.zeros(k), cov=sigma_u)
        new_value = value + innovation
        simulated[step] = new_value

        history = np.vstack([history, new_value])[-p:]

    simulated = simulated[burn_in:]

    return pd.DataFrame(simulated, index=index, columns=var_results.names)


def simulate_var_physical(var_results, target_mean, variables, seed, burn_in=500):

    sim_anom = simulate_var_path(
        var_results=var_results, n_steps=len(target_mean), index=target_mean.index, seed=seed, burn_in=burn_in
    )

    sim_phys = sim_anom[variables] + target_mean[variables]
    sim_phys["wind"] = wind_inv(sim_phys["wind"].to_numpy())
    # irradiance back from log space with exp()
    sim_phys["solar"] = np.exp(sim_phys["solar"]) - SOLAR_LOG_EPS
    sim_phys["solar"] = sim_phys["solar"].clip(lower=0)

    return sim_phys


var_results, var_order_selection = fit_var_model(
    df_anom_train=df_anom, variables=variables, maxlags=AR_MAX_LAG, criterion="bic"
)


In [ ]:
# 7. Model AR + Copula: seasonal autoregressions, seasonal empirical margins and a seasonal R-vine copula across the
#    variables; lag-1 temporal copula of irradiance; hour-of-day scale of the wind innovations

def season_block_ids(index, season):

    season_labels = pd.Series(index.month.map(get_season), index=index).to_numpy()
    is_season = season_labels == season
    block_id = np.full(len(index), -1, dtype=int)
    current_block = -1
    prev_in_season = False

    for i in range(len(index)):
        if is_season[i]:
            if not prev_in_season:
                current_block += 1
            block_id[i] = current_block
        prev_in_season = is_season[i]

    return block_id


def fit_ar_model_seasonal_one(y, block_id, col, maxlag):

    n = len(y)
    y_arr = y.to_numpy(dtype=float)
    lag_cols = []

    for lag in range(1, maxlag + 1):
        shifted_val = np.roll(y_arr, lag)
        shifted_block = np.roll(block_id, lag)
        valid = (block_id == shifted_block) & (block_id != -1) & (np.arange(n) >= lag)
        lag_cols.append(np.where(valid, shifted_val, np.nan))

    X_full = np.column_stack(lag_cols)
    target_valid = (block_id != -1) & ~np.isnan(X_full).any(axis=1)
    y_fit = y_arr[target_valid]
    X_fit = X_full[target_valid]
    design = np.column_stack([np.ones(len(y_fit)), X_fit])

    Q, R = np.linalg.qr(design)
    Qty = Q.T @ y_fit
    rss = np.maximum(np.sum(y_fit**2) - np.cumsum(Qty**2), 1e-12)
    n_obs = len(y_fit)
    bic = n_obs * np.log(rss / n_obs) + np.arange(1, maxlag + 2) * np.log(n_obs)
    best_k = int(np.argmin(bic)) + 1
    best_p = best_k - 1
    beta = np.linalg.solve(R[:best_k, :best_k], Qty[:best_k])

    params = pd.Series(beta, index=["const"] + [f"{col}.L{lag}" for lag in range(1, best_p + 1)])
    fitted = design[:, :best_k] @ beta
    resid = pd.Series(y_fit - fitted, index=y.index[target_valid])

    return {"params": params, "lags": list(range(1, best_p + 1)), "resid": resid}


def fit_ar_models_seasonal(df_anom, variables, seasons, maxlag=168):

    models_seasonal = {season: {} for season in seasons}
    residuals_by_season = {}

    for season in seasons:
        print(f"Fitting seasonal AR models for {season}")
        block_id = season_block_ids(df_anom.index, season)
        season_resid = pd.DataFrame(index=df_anom.index)

        for col in variables:
            fit = fit_ar_model_seasonal_one(y=df_anom[col], block_id=block_id, col=col, maxlag=maxlag)
            models_seasonal[season][col] = fit
            season_resid[col] = fit["resid"]
            print(f"  {col}: selected AR order {len(fit['lags'])}")

        residuals_by_season[season] = season_resid.dropna()

    residuals_seasonal_ar = pd.concat(
        [df.assign(season=season) for season, df in residuals_by_season.items()]
    ).sort_index()

    return models_seasonal, residuals_seasonal_ar


def reconstruct_simulation_seasonal(resid_sim, models_seasonal, target_mean, train_anom, variables):

    sim_anom = pd.DataFrame(index=target_mean.index, columns=variables, dtype=float)
    season_labels = target_mean.index.month.map(get_season).to_numpy()

    for col in variables:
        train_history = train_anom[col].dropna().to_numpy(dtype=float)
        n = len(target_mean)
        simulated = np.zeros(n, dtype=float)

        for i in range(n):
            model = models_seasonal[season_labels[i]][col]
            params = model["params"]
            lags = model["lags"]
            value = float(params["const"])

            for lag in lags:
                coefficient = float(params[f"{col}.L{lag}"])
                lagged_value = simulated[i - lag] if i - lag >= 0 else train_history[i - lag]
                value += coefficient * lagged_value

            value += float(resid_sim[col].iloc[i])
            simulated[i] = value

        sim_anom[col] = simulated

    sim_phys = sim_anom + target_mean[variables]
    sim_phys["wind"] = wind_inv(sim_phys["wind"].to_numpy())
    # irradiance back from log space with exp()
    sim_phys["solar"] = np.exp(sim_phys["solar"]) - SOLAR_LOG_EPS
    sim_phys["solar"] = sim_phys["solar"].clip(lower=0)

    return sim_phys


def seasonal_uniforms(residuals, variables, eps=1e-6):

    seasonal_U = {}

    for season in SEASONS:
        season_data = residuals[residuals["season"] == season][variables]
        U = pd.DataFrame(index=season_data.index)
        for col in variables:
            n = len(season_data[col])
            U[col] = np.clip(rankdata(season_data[col], method="average") / (n + 1), eps, 1 - eps)
        seasonal_U[season] = U

    return seasonal_U


def fit_margins(residuals, margin_config):

    params = {}

    for col in variables:
        x = residuals[col].dropna().to_numpy()
        kind = margin_config[col]

        if kind == "empirical":
            params[col] = {"kind": "empirical", "data": np.sort(x)}
        elif kind == "normal":
            loc, scale = norm.fit(x)
            params[col] = {"kind": "normal", "loc": loc, "scale": scale}
        elif kind == "student_t":
            df_value, loc, scale = t.fit(x)
            params[col] = {"kind": "student_t", "df": df_value, "loc": loc, "scale": scale}
        else:
            raise ValueError(f"Unknown margin {kind}")

    return params


def apply_margins(U_sim, margin_params, eps=1e-6):

    resid_sim = pd.DataFrame(index=U_sim.index)

    for col in variables:
        p = margin_params[col]
        u = np.clip(U_sim[col].to_numpy(), eps, 1 - eps)

        if p["kind"] == "empirical":
            resid_sim[col] = np.quantile(p["data"], u)
        elif p["kind"] == "normal":
            resid_sim[col] = norm.ppf(u, loc=p["loc"], scale=p["scale"])
        elif p["kind"] == "student_t":
            resid_sim[col] = t.ppf(u, df=p["df"], loc=p["loc"], scale=p["scale"])

    return resid_sim


def apply_margins_seasonal(U_sim, margin_params_seasonal, variables, eps=1e-6):

    resid_sim = pd.DataFrame(index=U_sim.index, columns=variables, dtype=float)
    seasons = pd.Series(U_sim.index.month.map(get_season), index=U_sim.index)

    for season in SEASONS:
        mask = seasons == season
        if mask.sum() == 0:
            continue
        resid_sim.loc[mask, variables] = apply_margins(
            U_sim=U_sim.loc[mask], margin_params=margin_params_seasonal[season], eps=eps
        ).to_numpy()

    return resid_sim


def fit_seasonal_vines(U_seasonal, candidate_families):

    seasonal_vines = {}

    for season, U in U_seasonal.items():
        print(f"Fitting vine for {season}")
        controls = pv.FitControlsVinecop(
            family_set=candidate_families, selection_criterion="bic", parametric_method="mle",
            allow_rotations=True, num_threads=1,
        )
        vine = pv.Vinecop.from_data(data=np.asfortranarray(U[variables].to_numpy(dtype=float)), controls=controls)
        seasonal_vines[season] = vine

    return seasonal_vines


def simulate_seasonal_copulas(seasonal_vines, target_index, variables, seed):

    U_sim = pd.DataFrame(index=target_index, columns=variables, dtype=float)
    seasons = pd.Series(target_index.month.map(get_season), index=target_index)
    seed_offsets = {"winter": 0, "spring": 1000, "summer": 2000, "autumn": 3000}

    for season in SEASONS:
        mask = seasons == season
        n_samples = int(mask.sum())
        simulated = seasonal_vines[season].simulate(n=n_samples, seeds=[int(seed + seed_offsets[season])])
        U_sim.loc[mask, variables] = simulated

    return U_sim.astype(float)


# margins of the copula: empirical (quantile function of the training residuals) for all three variables
margin_config = {"wind": "empirical", "temp": "empirical", "solar": "empirical"}
candidate_families = [pv.BicopFamily.gaussian, pv.BicopFamily.student, pv.BicopFamily.clayton,
                      pv.BicopFamily.gumbel, pv.BicopFamily.frank]

models_seasonal, residuals_seasonal_ar = fit_ar_models_seasonal(
    df_anom=df_anom, variables=variables, seasons=SEASONS, maxlag=AR_MAX_LAG
)

U_seasonal = seasonal_uniforms(residuals=residuals_seasonal_ar, variables=variables, eps=eps)

margin_params_seasonal = {}
for season in SEASONS:
    season_residuals = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, variables]
    margin_params_seasonal[season] = fit_margins(residuals=season_residuals, margin_config=margin_config)

seasonal_vines = fit_seasonal_vines(U_seasonal=U_seasonal, candidate_families=candidate_families)

# simulation: wind and temperature residuals from a seasonal empirical bootstrap; irradiance residuals from a
# sequential lag-1 temporal copula per season, fitted to (residual[t-1], residual[t]) and drawn forward
# with the conditional inverse h-function (hinv2); its parameter is scaled by SOLAR_TEMPORAL_SHRINK so that
# the reconstructed series reproduces the observed lag-1 tail dependence. seasonal_vines and
# fit_seasonal_vines are used by Section 13

TEMPORAL_CANDIDATE_FAMILIES = [pv.BicopFamily.gaussian, pv.BicopFamily.student, pv.BicopFamily.clayton,
                                pv.BicopFamily.gumbel, pv.BicopFamily.frank, pv.BicopFamily.bb1]

# calibrated to the lag-1 tail dependence of the observed irradiance anomalies (reconstructed series)
SOLAR_TEMPORAL_SHRINK = {"winter": 2.20, "spring": 1.00, "summer": 2.20, "autumn": 0.80}


def fit_solar_temporal_copulas(residuals_seasonal_ar, seasons, shrink_by_season):

    def lag1_pairs(col, season):
        sub = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, col].dropna().sort_index()
        idx = sub.index
        elapsed = (idx[1:] - idx[:-1]).total_seconds() / 3600
        valid = (np.asarray(elapsed) == 1.0)
        return sub.to_numpy()[:-1][valid], sub.to_numpy()[1:][valid]

    copulas = {}
    for season in seasons:
        x_prev, x_curr = lag1_pairs("solar", season)
        n = len(x_prev)
        u_prev = rankdata(x_prev, method="average") / (n + 1)
        u_curr = rankdata(x_curr, method="average") / (n + 1)
        data = np.asfortranarray(np.column_stack([u_prev, u_curr]))
        controls = pv.FitControlsBicop(family_set=TEMPORAL_CANDIDATE_FAMILIES, selection_criterion="bic",
                                        parametric_method="mle")
        bicop = pv.Bicop()
        bicop.select(data=data, controls=controls)

        base_params = np.asarray(bicop.parameters).astype(float).copy()
        shrunk_params = base_params * shrink_by_season[season]
        lb = np.asarray(bicop.parameters_lower_bounds).reshape(shrunk_params.shape)
        ub = np.asarray(bicop.parameters_upper_bounds).reshape(shrunk_params.shape)
        shrunk_params = np.clip(shrunk_params, lb, ub)
        copulas[season] = pv.Bicop(family=bicop.family, rotation=bicop.rotation, parameters=shrunk_params)

    return copulas


# the quantile lookup runs once per season after the loop: the pool is sorted and mapped with np.interp
# (identical to np.quantile with linear interpolation)
_SOLAR_POOL_SORTED = {}


def simulate_solar_temporal_sequence(solar_temporal_copulas, solar_pool_by_season, season_labels, seed):

    rng = np.random.default_rng(seed)
    n = len(season_labels)
    u = np.empty(n)
    ws = rng.uniform(size=n)
    prev_season = None

    for i in range(n):
        season = season_labels[i]
        if season != prev_season:
            u[i] = ws[i]
        else:
            u[i] = float(np.asarray(solar_temporal_copulas[season].hinv2(np.array([[ws[i], u[i - 1]]])))[0])
        prev_season = season

    out = np.empty(n)
    u_clipped = np.clip(u, eps, 1 - eps)
    for season in set(season_labels):
        key = (id(solar_pool_by_season), season)
        if key not in _SOLAR_POOL_SORTED:
            sp = np.sort(solar_pool_by_season[season])
            _SOLAR_POOL_SORTED[key] = (sp, np.linspace(0.0, 1.0, len(sp)))
        sp, grid = _SOLAR_POOL_SORTED[key]
        mask = season_labels == season
        out[mask] = np.interp(u_clipped[mask], grid, sp)

    return out


solar_temporal_copulas = fit_solar_temporal_copulas(
    residuals_seasonal_ar=residuals_seasonal_ar, seasons=SEASONS, shrink_by_season=SOLAR_TEMPORAL_SHRINK
)
solar_pool_by_season = {
    season: residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, "solar"].dropna().to_numpy()
    for season in SEASONS
}
for season in SEASONS:
    print(f"  solar temporal copula, {season}: family={solar_temporal_copulas[season].family} "
          f"tau={solar_temporal_copulas[season].tau:+.3f} (shrink={SOLAR_TEMPORAL_SHRINK[season]})")

# hour-of-day scale of the wind innovations per season: the observed standard deviation of wind by hour of day
# relative to its mean, with a strength per season (WIND_HOUR_SCALE_STRENGTH) calibrated so that the
# reconstructed series reproduces the observed hour-of-day amplitude
WIND_HOUR_SCALE_STRENGTH = {"winter": 13.0, "spring": 3.7, "summer": 3.2, "autumn": 4.9}


def compute_wind_hour_scale(df_anom_local, seasons, get_season_fn):
    scale = {}
    for season in seasons:
        sub = df_anom_local.loc[df_anom_local.index.month.map(get_season_fn) == season, "wind"]
        std_by_hour = sub.groupby(sub.index.hour).std()
        scale[season] = (std_by_hour / std_by_hour.mean()).to_numpy()  # index 0..23
    return scale


wind_hour_scale_raw = compute_wind_hour_scale(df_anom, SEASONS, get_season)


def wind_scale_array_for_index(target_index):
    T = len(target_index)
    season_labels = np.array([get_season(m) for m in target_index.month])
    hour_labels = target_index.hour.to_numpy()
    out = np.empty(T)
    for season in SEASONS:
        mask = season_labels == season
        raw = wind_hour_scale_raw[season][hour_labels[mask]]
        out[mask] = 1.0 + WIND_HOUR_SCALE_STRENGTH[season] * (raw - 1.0)
    return out




In [ ]:
# 8. Monte Carlo ensembles of the three models: each trajectory draws one training year's offset (Section 4), the
#    same draw for all models; simulation batched across the trajectories (vectorized_simulation.py)

import sys
sys.path.insert(0, ".")
from vectorized_simulation import (
    simulate_ar_independent_batch, simulate_var_physical_batch,
    reconstruct_simulation_seasonal_batch,
)

seeds = [BASE_SEED + r for r in range(N_TRAJECTORIES)]
T = len(sim_calendar_mean)
target_index = sim_calendar_mean.index
sim_calendar_arr = {col: sim_calendar_mean[col].to_numpy() for col in variables}

drawn_years = {}
target_mean_batch = {col: np.empty((N_TRAJECTORIES, T)) for col in variables}
for j, seed in enumerate(seeds):
    rng = np.random.default_rng(seed + 1_000_000)
    year = int(rng.choice(year_baseline_offset.index.to_numpy()))
    drawn_years[j] = year
    offset = year_baseline_offset.loc[year]
    for col in variables:
        target_mean_batch[col][j] = sim_calendar_arr[col] + offset[col]


def unbatch(batch_dict, index, variables):
    n = next(iter(batch_dict.values())).shape[0]
    return {j: pd.DataFrame({col: batch_dict[col][j] for col in variables}, index=index) for j in range(n)}


# hour-of-day scale of the wind innovations (Section 7), applied in all three models
wind_scale_array = wind_scale_array_for_index(target_index)

print("Simulating AR ensemble (vectorized)...")
ar_batch = simulate_ar_independent_batch(ar_models, ar_residuals, target_mean_batch, df_anom, variables, seeds,
                                          wind_scale=wind_scale_array)
ar_trajectories = unbatch(ar_batch, target_index, variables)

print("Simulating VAR ensemble (vectorized)...")
var_batch = simulate_var_physical_batch(var_results, target_mean_batch, variables, seeds, burn_in=500,
                                         wind_scale=wind_scale_array)
var_trajectories = unbatch(var_batch, target_index, variables)

# model AR + Copula: wind and temperature from the seasonal bootstrap, irradiance from the lag-1 temporal
# copula (Section 7)
print("Simulating AR+Copula ensemble (seasonal bootstrap for wind/temp, calibrated temporal copula for solar + vectorized reconstruction)...")
season_labels_target = np.array([get_season(m) for m in target_index.month])
resid_sim_batch_cop = {col: np.empty((N_TRAJECTORIES, T)) for col in variables}
for j, seed in enumerate(seeds):
    rng_boot = np.random.default_rng(seed)
    for col in ["wind", "temp"]:
        for season in SEASONS:
            mask = season_labels_target == season
            pool = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, col].dropna().to_numpy()
            resid_sim_batch_cop[col][j, mask] = rng_boot.choice(pool, size=int(mask.sum()), replace=True)
    resid_sim_batch_cop["wind"][j] = resid_sim_batch_cop["wind"][j] * wind_scale_array
    resid_sim_batch_cop["solar"][j] = simulate_solar_temporal_sequence(
        solar_temporal_copulas=solar_temporal_copulas, solar_pool_by_season=solar_pool_by_season,
        season_labels=season_labels_target, seed=seed + 5_000_000
    )

arcop_batch = reconstruct_simulation_seasonal_batch(
    resid_sim_batch_cop, models_seasonal, target_mean_batch, target_index, df_anom, variables, get_season
)
arcop_trajectories = unbatch(arcop_batch, target_index, variables)

drawn_years_counts = pd.Series(drawn_years).value_counts().sort_index()
print("\nHistorical years bootstrap-sampled across the ensemble :")
print(drawn_years_counts.to_string())

print("\nDone:", len(ar_trajectories), len(var_trajectories), len(arcop_trajectories), "trajectories per arm")


In [ ]:
# 9. Error diagnostics by season and by tail

def error_by_season(obs, sim, variables):
    seasons_idx = pd.Series(obs.index.month.map(get_season), index=obs.index)
    rows = []
    for season in SEASONS:
        mask = seasons_idx == season
        for col in variables:
            rmse = np.sqrt(mean_squared_error(obs.loc[mask, col], sim.loc[mask, col]))
            bias = (sim.loc[mask, col] - obs.loc[mask, col]).mean()
            rows.append({"season": season, "variable": col, "rmse": rmse, "bias": bias})
    return pd.DataFrame(rows)


def error_by_tail(obs, sim, variables, n_bins=5):
    # Uses qcut with duplicates="drop" because solar is heavily zero-inflated
    # (roughly half of all hours are night, i.e. exactly 0) - several low
    # quantile edges collapse to the same value, so fewer than n_bins usable
    # bins is expected and handled, not an error.
    rows = []
    for col in variables:
        bin_id = pd.qcut(obs[col], q=n_bins, labels=False, duplicates="drop")
        n_actual_bins = int(bin_id.max()) + 1
        for b in range(n_actual_bins):
            mask = bin_id == b
            if mask.sum() == 0:
                continue
            rmse = np.sqrt(mean_squared_error(obs.loc[mask, col], sim.loc[mask, col]))
            bias = (sim.loc[mask, col] - obs.loc[mask, col]).mean()
            rows.append({
                "bin": b, "n_bins": n_actual_bins, "variable": col, "rmse": rmse, "bias": bias,
                "n": int(mask.sum()),
                "bin_min": obs.loc[mask, col].min(),
                "bin_max": obs.loc[mask, col].max(),
            })
    return pd.DataFrame(rows)


def summarize_ensemble_errors(trajectories, obs, variables, error_fn, **kwargs):
    frames = []
    for run, sim in trajectories.items():
        df = error_fn(obs, sim, variables, **kwargs)
        df["run"] = run
        frames.append(df)
    all_runs = pd.concat(frames, ignore_index=True)
    group_cols = [c for c in all_runs.columns if c not in ("run", "rmse", "bias", "n")]
    return all_runs.groupby(group_cols)[["rmse", "bias"]].mean().reset_index()


season_error_ar = summarize_ensemble_errors(ar_trajectories, df_test, variables, error_by_season)
season_error_var = summarize_ensemble_errors(var_trajectories, df_test, variables, error_by_season)
season_error_arcop = summarize_ensemble_errors(arcop_trajectories, df_test, variables, error_by_season)

tail_error_ar = summarize_ensemble_errors(ar_trajectories, df_test, variables, error_by_tail)
tail_error_var = summarize_ensemble_errors(var_trajectories, df_test, variables, error_by_tail)
tail_error_arcop = summarize_ensemble_errors(arcop_trajectories, df_test, variables, error_by_tail)

print(season_error_ar.head())


In [ ]:
# 10. Plots of the errors by season and by tail

model_colors = {"AR": "#4c72b0", "VAR": "#dd8452", "AR+Copula": "#55a868"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=False)

for ax, col in zip(axes, variables):
    width = 0.25
    x = np.arange(len(SEASONS))

    for offset, (name, df) in zip(
        [-width, 0, width],
        [("AR", season_error_ar), ("VAR", season_error_var), ("AR+Copula", season_error_arcop)],
    ):
        vals = [df[(df.season == s) & (df.variable == col)]["rmse"].iloc[0] for s in SEASONS]
        ax.bar(x + offset, vals, width=width, label=name, color=model_colors[name])

    ax.set_xticks(x)
    ax.set_xticklabels(SEASONS)
    ax.set_title(f"{col} - RMSE by season")
    ax.set_ylabel("RMSE")

axes[0].legend()
plt.tight_layout()
plt.savefig("rmse_by_season.png", dpi=120)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=False)

# bin edges from the observations, identical for the three models; irradiance has fewer bins because about
# half of its hours are 0 (night)
for ax, col in zip(axes, variables):
    width = 0.25

    ref = tail_error_ar[tail_error_ar.variable == col].sort_values("bin")
    n_actual_bins = len(ref)
    x = np.arange(n_actual_bins)
    labels = [f"{row.bin_min:.0f}-{row.bin_max:.0f}" for row in ref.itertuples()]

    for offset, (name, df) in zip(
        [-width, 0, width],
        [("AR", tail_error_ar), ("VAR", tail_error_var), ("AR+Copula", tail_error_arcop)],
    ):
        sub = df[df.variable == col].sort_values("bin")
        ax.bar(x[: len(sub)] + offset, sub["rmse"], width=width, label=name, color=model_colors[name])

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_title(f"{col} - RMSE by observed-value bin (n={n_actual_bins})")
    ax.set_ylabel("RMSE")

axes[0].legend()
plt.tight_layout()
plt.savefig("rmse_by_tail.png", dpi=120)
plt.show()


In [ ]:
# 11. Confidence envelopes: 5-95 % band of the daily means across the trajectories, with the observed extreme
#     years and their coverage

def daily_mean_stack(trajectories, variables):
    daily = {col: [] for col in variables}
    day_idx = None
    for sim in trajectories.values():
        d = sim.resample("D").mean()
        day_idx = d.index
        for col in variables:
            daily[col].append(d[col].to_numpy())
    return {col: np.vstack(daily[col]) for col in variables}, day_idx


ar_daily, day_index = daily_mean_stack(ar_trajectories, variables)
var_daily, _ = daily_mean_stack(var_trajectories, variables)
arcop_daily, _ = daily_mean_stack(arcop_trajectories, variables)

# Cache the (expensive to produce) daily-mean trajectory arrays so a future
# extreme-year swap can redo the overlay/coverage check below without
# re-fitting and re-simulating all three models again.
np.savez(
    daily_mean_cache_path,
    day_index_ordinal=np.array([d.toordinal() for d in day_index]),
    **{f"ar_{col}": ar_daily[col] for col in variables},
    **{f"var_{col}": var_daily[col] for col in variables},
    **{f"arcop_{col}": arcop_daily[col] for col in variables},
)
print(f"Cached daily-mean trajectories to {daily_mean_cache_path}")

day_of_year = day_index.dayofyear
model_daily = {"AR": ar_daily, "VAR": var_daily, "AR+Copula": arcop_daily}

# one figure per extreme year: 1963 (out of sample), 2010, 2017 and 2021 (training period)
extreme_years_to_plot = {
    "1963": df_extreme[variables],
    "2010": df_train.loc[df_train.index.year == 2010, variables],
    "2017": df_train.loc[df_train.index.year == 2017, variables],
    "2021": df_train.loc[df_train.index.year == 2021, variables],
}

all_coverage_rows = []

for year_label, year_df in extreme_years_to_plot.items():

    year_daily = year_df.resample("D").mean()
    year_day_of_year = year_daily.index.dayofyear

    fig, axes = plt.subplots(len(variables), 3, figsize=(16, 11), sharex=True)

    for i, col in enumerate(variables):
        for j, name in enumerate(["AR", "VAR", "AR+Copula"]):
            ax = axes[i, j]
            stack = model_daily[name][col]

            lo = np.percentile(stack, 5, axis=0)
            hi = np.percentile(stack, 95, axis=0)
            mid = np.percentile(stack, 50, axis=0)

            ax.fill_between(day_of_year, lo, hi, color=model_colors[name], alpha=0.25, label="5-95% envelope")
            ax.plot(day_of_year, mid, color=model_colors[name], lw=1, label="median simulated")
            ax.plot(year_day_of_year, year_daily[col].to_numpy(), color="crimson", lw=1.3, label=f"{year_label} (extreme)")

            lo_at_year = np.interp(year_day_of_year, day_of_year, lo)
            hi_at_year = np.interp(year_day_of_year, day_of_year, hi)
            inside = (
                (year_daily[col].to_numpy() >= lo_at_year)
                & (year_daily[col].to_numpy() <= hi_at_year)
            )
            coverage = inside.mean()
            all_coverage_rows.append({"extreme_year": year_label, "variable": col, "model": name, "coverage": coverage})

            ax.set_title(f"{col} / {name}  (coverage={coverage:.0%})")

            if i == 0 and j == 0:
                ax.legend(fontsize=8, loc="upper right")

    fig.suptitle(f"{year_label} vs. simulated 5-95% envelope (bootstrap-sampled year-effect ensemble, N={N_TRAJECTORIES})", y=1.01)
    plt.tight_layout()
    plt.savefig(f"envelope_overlay_{year_label}.png", dpi=120)
    plt.show()
    plt.close(fig)

coverage_df = pd.DataFrame(all_coverage_rows)
print("Coverage of each dunkelflaute-relevant extreme year in each model's 5-95% band:")
for year_label in extreme_years_to_plot:
    print(f"\n{year_label}:")
    print(coverage_df[coverage_df.extreme_year == year_label].pivot(index="variable", columns="model", values="coverage").round(3))


In [ ]:
# 12. Dunkelflaute rate: hours with wind and irradiance both in their bottom decile (irradiance threshold over the
#     daylight hours of each month), observed and simulated

DUNKELFLAUTE_QUANTILE = 0.10
DAYLIGHT_FLOOR = 10.0  # W/m2, climatological threshold for "this hour counts as daylight"

clim_month_hour = df_train.groupby(["month", "hour"])["solar"].mean()


def is_daylight(index):
    keys = list(zip(index.month, index.hour))
    return np.array([clim_month_hour.get(k, 0.0) > DAYLIGHT_FLOOR for k in keys])


train_daylight_mask = is_daylight(df_train.index)
train_daylight = df_train.loc[train_daylight_mask, variables]

wind_thresh = train_daylight["wind"].quantile(DUNKELFLAUTE_QUANTILE)
solar_thresh = train_daylight["solar"].quantile(DUNKELFLAUTE_QUANTILE)

print(f"Daylight-only dunkelflaute thresholds: wind <= {wind_thresh:.2f} m/s, solar <= {solar_thresh:.2f} W/m2")


def dunkelflaute_rate(df):
    mask = is_daylight(df.index)
    sub = df.loc[mask]
    event = (sub["wind"] <= wind_thresh) & (sub["solar"] <= solar_thresh)
    return event.mean()


def ensemble_dunkelflaute_rates(trajectories):
    return np.array([dunkelflaute_rate(sim) for sim in trajectories.values()])


# observed rates: 2010, 2017 and 2022-2023 in the training period, 1963 out of sample
periods = {
    "2000-2024 baseline": df_train[variables],
    "2017 full year": df_train.loc[df_train.index.year == 2017, variables],
    "Jan 16-25, 2017 event": df_train.loc["2017-01-16":"2017-01-25", variables],
    "Dec 2022 - Mar 2023": pd.concat([
        df_train.loc["2022-12-01":"2022-12-31", variables],
        df_train.loc["2023-01-01":"2023-03-31", variables],
    ]),
    "2010 (in-sample)": df_train.loc[df_train.index.year == 2010, variables],
    f"{extreme_year} (out-of-sample)": df_extreme[variables],
}

observed_rows = []
for name, df_period in periods.items():
    rate = dunkelflaute_rate(df_period)
    observed_rows.append({"period": name, "observed_rate": rate})

observed_df = pd.DataFrame(observed_rows)
observed_df["ratio_to_baseline"] = observed_df["observed_rate"] / observed_df.loc[0, "observed_rate"]
print()
print(observed_df.round(4))

# Simulated rate distributions (300 runs each, already hourly, already in memory)
ar_rates = ensemble_dunkelflaute_rates(ar_trajectories)
var_rates = ensemble_dunkelflaute_rates(var_trajectories)
arcop_rates = ensemble_dunkelflaute_rates(arcop_trajectories)

sim_summary = pd.DataFrame({
    "model": ["AR", "VAR", "AR+Copula"],
    "mean_rate": [ar_rates.mean(), var_rates.mean(), arcop_rates.mean()],
    "p5": [np.percentile(ar_rates, 5), np.percentile(var_rates, 5), np.percentile(arcop_rates, 5)],
    "p95": [np.percentile(ar_rates, 95), np.percentile(var_rates, 95), np.percentile(arcop_rates, 95)],
})
print()
print("Simulated dunkelflaute-rate distribution (bootstrap-sampled year-effect ensemble, 300 runs each):")
print(sim_summary.round(4))


In [ ]:
# 12b. Documented Dunkelflaute periods against the simulated distribution of the rate

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

sim_rates = {"AR": ar_rates, "VAR": var_rates, "AR+Copula": arcop_rates}
period_colors = {
    "2000-2024 baseline": "0.4",
    "2017 full year": "#8172b3",
    "Jan 16-25, 2017 event": "crimson",
    "Dec 2022 - Mar 2023": "#937860",
    "2010 (in-sample)": "#da8bc3",
    f"{extreme_year} (out-of-sample)": "#4c72b0",
}

for ax, name in zip(axes, ["AR", "VAR", "AR+Copula"]):
    ax.hist(sim_rates[name], bins=20, color=model_colors[name], alpha=0.5, label="simulated (200 runs)")
    for _, row in observed_df.iterrows():
        ax.axvline(row["observed_rate"], color=period_colors[row["period"]], lw=1.5,
                   label=row["period"] if name == "AR" else None)
    ax.set_title(name)
    ax.set_xlabel("joint low-wind + low-solar rate\n(share of daylight hours)")

axes[0].set_ylabel("count across 200 runs")
axes[0].legend(fontsize=7, loc="upper right")
plt.tight_layout()
plt.savefig("dunkelflaute_validation.png", dpi=120)
plt.show()


In [ ]:
# 13. Decomposition of the Dunkelflaute rate into the marginal rates and a dependence multiplier (joint rate /
#     product of the marginal rates), observed and simulated; tail dependence of the fitted copulas in rank space

def marginal_rates(df):
    mask = is_daylight(df.index)
    sub = df.loc[mask]
    p_low_wind = (sub["wind"] <= wind_thresh).mean()
    p_low_solar = (sub["solar"] <= solar_thresh).mean()
    return p_low_wind, p_low_solar


def pooled_trajectories(trajectories, variables):
    return pd.concat(trajectories.values(), axis=0)[variables]


# real data: how much does true dependence inflate the joint rate?
real_p_wind, real_p_solar = marginal_rates(df_train)
real_independence_rate = real_p_wind * real_p_solar
real_observed_rate = dunkelflaute_rate(df_train)
real_dependence_multiplier = real_observed_rate / real_independence_rate

print("REAL DATA (2000-2024):")
print(f"  P(low wind)  = {real_p_wind:.4f}")
print(f"  P(low solar) = {real_p_solar:.4f}")
print(f"  independence-implied joint rate = {real_independence_rate:.4f}")
print(f"  actual observed joint rate      = {real_observed_rate:.4f}")
print(f"  dependence multiplier (actual / independence) = {real_dependence_multiplier:.2f}x")
print()

# per-model simulated: same decomposition
model_trajectories = {"AR": ar_trajectories, "VAR": var_trajectories, "AR+Copula": arcop_trajectories}
decomposition_rows = []

for name, trajectories in model_trajectories.items():
    pooled = pooled_trajectories(trajectories, variables)
    p_wind, p_solar = marginal_rates(pooled)
    independence_rate = p_wind * p_solar
    actual_rate = dunkelflaute_rate(pooled)
    dependence_multiplier = actual_rate / independence_rate

    decomposition_rows.append({
        "model": name,
        "P(low wind)": p_wind,
        "P(low solar)": p_solar,
        "independence-implied rate": independence_rate,
        "actual simulated rate": actual_rate,
        "dependence multiplier": dependence_multiplier,
    })

    print(f"{name}:")
    print(f"  P(low wind)  = {p_wind:.4f}  (real: {real_p_wind:.4f}, ratio {p_wind / real_p_wind:.2f}x)")
    print(f"  P(low solar) = {p_solar:.4f}  (real: {real_p_solar:.4f}, ratio {p_solar / real_p_solar:.2f}x)")
    print(f"  independence-implied joint rate = {independence_rate:.4f}")
    print(f"  actual simulated joint rate      = {actual_rate:.4f}")
    print(f"  dependence multiplier = {dependence_multiplier:.2f}x  (real: {real_dependence_multiplier:.2f}x)")
    print()

decomposition_df = pd.DataFrame(decomposition_rows)
print(decomposition_df.round(4))

# tail dependence of the fitted vines alone: sample (U_wind, U_temp, U_solar) in rank space and compare
# P(U_wind <= q, U_solar <= q) with q^2 (season-relative quantiles)

DUNKEL_Q = DUNKELFLAUTE_QUANTILE  # 0.10, matches the dunkelflaute definition
N_COPULA_CHECK = 200_000
copula_check_seeds = {"winter": 7000, "spring": 7001, "summer": 7002, "autumn": 7003}

wind_idx = variables.index("wind")
solar_idx = variables.index("solar")

print(f"\nDirect copula tail-dependence check (q={DUNKEL_Q}, independence baseline = {DUNKEL_Q**2:.4f}):")
copula_tail_rows = []
for season in SEASONS:
    U = seasonal_vines[season].simulate(n=N_COPULA_CHECK, seeds=[copula_check_seeds[season]])
    u_wind = np.asarray(U)[:, wind_idx]
    u_solar = np.asarray(U)[:, solar_idx]
    joint_tail_rate = ((u_wind <= DUNKEL_Q) & (u_solar <= DUNKEL_Q)).mean()
    multiplier = joint_tail_rate / (DUNKEL_Q ** 2)
    copula_tail_rows.append({"season": season, "copula joint-tail rate": joint_tail_rate, "vs independence": multiplier})
    print(f"  {season:<8} joint-tail rate = {joint_tail_rate:.4f}  ({multiplier:.2f}x independence)")

copula_tail_df = pd.DataFrame(copula_tail_rows)
print()
print(copula_tail_df.round(4))


In [ ]:
# 14. Wind margin of the model AR + Copula: (a) seasonal AR with a seasonal empirical bootstrap, no copula;
#     (b) seasonal AR + copula with an empirical wind margin; low-tail quantile of a Student-t margin against the
#     empirical residuals

season_labels_target = pd.Series(target_index.month.map(get_season), index=target_index).to_numpy()

# 14a: seasonal AR dynamics, no copula - draw straight from each
#     season's own empirical residual pool (same mechanism as AR/Section 5,
# just season-specific pools and season-specific AR fits)

resid_sim_batch_seasonalboot = {col: np.empty((N_TRAJECTORIES, T)) for col in variables}
for j, seed in enumerate(seeds):
    rng = np.random.default_rng(seed + 2_000_000)
    for season in SEASONS:
        mask = season_labels_target == season
        n_season = int(mask.sum())
        for col in variables:
            pool = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, col].dropna().to_numpy()
            resid_sim_batch_seasonalboot[col][j, mask] = rng.choice(pool, size=n_season, replace=True)

print("Reconstructing 14a (seasonal AR, empirical bootstrap, no copula)...")
seasonalboot_batch = reconstruct_simulation_seasonal_batch(
    resid_sim_batch_seasonalboot, models_seasonal, target_mean_batch, target_index, df_anom, variables, get_season
)
seasonalboot_trajectories = unbatch(seasonalboot_batch, target_index, variables)

# 14b: seasonal AR + copula, wind's margin swapped to empirical

margin_config_wind_empirical = {"wind": "empirical", "temp": "student_t", "solar": "empirical"}
margin_params_seasonal_windemp = {}
for season in SEASONS:
    season_residuals = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, variables]
    margin_params_seasonal_windemp[season] = fit_margins(residuals=season_residuals, margin_config=margin_config_wind_empirical)

resid_sim_batch_windemp = {col: np.empty((N_TRAJECTORIES, T)) for col in variables}
for j, seed in enumerate(seeds):
    U_sim = simulate_seasonal_copulas(seasonal_vines=seasonal_vines, target_index=target_index, variables=variables, seed=seed)
    resid_sim = apply_margins_seasonal(U_sim=U_sim, margin_params_seasonal=margin_params_seasonal_windemp, variables=variables, eps=eps)
    for col in variables:
        resid_sim_batch_windemp[col][j] = resid_sim[col].to_numpy()

print("Reconstructing 14b (seasonal AR + copula, wind margin -> empirical)...")
windemp_batch = reconstruct_simulation_seasonal_batch(
    resid_sim_batch_windemp, models_seasonal, target_mean_batch, target_index, df_anom, variables, get_season
)
windemp_trajectories = unbatch(windemp_batch, target_index, variables)

# compare P(low wind) / P(low solar) across all variants

variant_trajectories = {
    "AR (S13 baseline, pooled+bootstrap)": ar_trajectories,
    "VAR (S13 baseline)": var_trajectories,
    "14a seasonal AR + bootstrap (no copula)": seasonalboot_trajectories,
    "14b seasonal AR + copula, wind->empirical": windemp_trajectories,
    "AR+Copula (S13 baseline, current)": arcop_trajectories,
}

print(f"\nP(low wind) across variants (real: {real_p_wind:.4f}):")
wind_marginal_rows = []
for name, trajectories in variant_trajectories.items():
    pooled = pooled_trajectories(trajectories, variables)
    p_wind, p_solar = marginal_rates(pooled)
    wind_marginal_rows.append({
        "variant": name, "P(low wind)": p_wind, "wind ratio to real": p_wind / real_p_wind,
        "P(low solar)": p_solar, "solar ratio to real": p_solar / real_p_solar,
    })
    print(f"  {name:<42} P(low wind)={p_wind:.4f} ({p_wind/real_p_wind:.2f}x)   P(low solar)={p_solar:.4f} ({p_solar/real_p_solar:.2f}x)")

wind_marginal_df = pd.DataFrame(wind_marginal_rows)
print()
print(wind_marginal_df.round(4))

# does the Student-t margin's own low-tail quantile sit further out
#     than the empirical residuals it was fit to? direct check on the fit
# itself, independent of any simulation

print(f"\nWind residual low-tail quantile (q={DUNKELFLAUTE_QUANTILE}): empirical vs. a student_t fit, by season")
print("(fitted here, independent of the margin_config of Section 7):")
for season in SEASONS:
    season_wind_resid = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, "wind"].dropna()
    empirical_q = np.quantile(season_wind_resid, DUNKELFLAUTE_QUANTILE)
    df_value, loc, scale = t.fit(season_wind_resid.to_numpy())
    fitted_q = t.ppf(DUNKELFLAUTE_QUANTILE, df=df_value, loc=loc, scale=scale)
    print(f"  {season:<8} empirical q{int(DUNKELFLAUTE_QUANTILE*100)}={empirical_q:+.4f}   student_t-fitted={fitted_q:+.4f}   diff={fitted_q-empirical_q:+.4f}")


In [ ]:
# 15. Irradiance in log space against an additive reconstruction clipped at zero: width of the envelope and
#     coverage of 1963
#
# map_climatology and fit_linear_trend read the global `variables`, which is set temporarily to irradiance alone

_variables_backup = variables
variables = ["solar"]
try:
    print("Fitting the linear-space solar trend + climatology...")
    trend_coefs_solar_lin = fit_linear_trend(df_train, ["solar"], window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)

    df_train_detrended_lin = df_train[["solar"]] - trend_value(df_train.index, trend_coefs_solar_lin, ["solar"])
    df_train_detrended_lin["month"] = df_train.index.month
    df_train_detrended_lin["day"] = df_train.index.day
    df_train_detrended_lin["hour"] = df_train.index.hour

    climatology_solar_lin = build_smoothed_climatology(df_train_detrended_lin, ["solar"], window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
    train_mean_solar_lin = map_climatology(df_train.index, climatology_solar_lin) + trend_value(df_train.index, trend_coefs_solar_lin, ["solar"])
    df_anom_solar_lin = df_train[["solar"]] - train_mean_solar_lin
    sim_calendar_mean_solar_lin = map_climatology(df_test.index, climatology_solar_lin)
finally:
    variables = _variables_backup

year_baseline_offset_solar_lin = df_anom_solar_lin.groupby(df_anom_solar_lin.index.year).mean()

print("Fitting the AR model for solar in linear space...")
solar_lin_selection = ar_select_order(df_anom_solar_lin["solar"].dropna(), maxlag=AR_MAX_LAG, ic="bic", old_names=False)
solar_lin_model = solar_lin_selection.model.fit()
solar_lin_resid_pool = solar_lin_model.resid.dropna().to_numpy()
print("  selected lags:", list(solar_lin_model.model._lags))


def reconstruct_solar_old(resid_sim, model, target_mean_arr, train_history):
    params = model.params
    lags = list(model.model._lags)
    intercept = float(params["const"]) if "const" in params.index else 0.0
    n = len(target_mean_arr)
    simulated = np.zeros(n, dtype=float)
    for i in range(n):
        value = intercept
        for lag in lags:
            coefficient = float(params[f"solar.L{lag}"])
            lagged_value = simulated[i - lag] if i - lag >= 0 else train_history[i - lag]
            value += coefficient * lagged_value
        value += float(resid_sim[i])
        simulated[i] = value
    sim_phys = simulated + target_mean_arr
    return np.clip(sim_phys, 0, None)  # additive reconstruction, clipped at 0


solar_lin_train_history = df_anom_solar_lin["solar"].dropna().to_numpy()
sim_calendar_solar_lin_arr = sim_calendar_mean_solar_lin["solar"].to_numpy()

print(f"Simulating {N_TRAJECTORIES} OLD-style solar trajectories...")
old_solar_daily = []
day_idx_old = None
old_solar_clip_fraction = []
for j, seed in enumerate(seeds):
    rng = np.random.default_rng(seed + 3_000_000)
    year = drawn_years[j]  # reuse the SAME year draw as every other arm this run
    offset = year_baseline_offset_solar_lin.loc[year, "solar"]
    target_mean_j = sim_calendar_solar_lin_arr + offset
    resid_sim_j = rng.choice(solar_lin_resid_pool, size=T, replace=True)
    sim_phys_j = reconstruct_solar_old(resid_sim_j, solar_lin_model, target_mean_j, solar_lin_train_history)
    old_solar_clip_fraction.append((sim_phys_j == 0).mean())
    s = pd.Series(sim_phys_j, index=target_index).resample("D").mean()
    day_idx_old = s.index
    old_solar_daily.append(s.to_numpy())

old_solar_daily = np.vstack(old_solar_daily)
print(f"\nOLD (linear, additive+clip) solar: {np.mean(old_solar_clip_fraction):.1%} of hours clipped to exactly 0 (mean across {N_TRAJECTORIES} trajectories)")

# log-space irradiance daily means (Section 11)
new_solar_daily = ar_daily["solar"]

old_width = np.percentile(old_solar_daily, 95, axis=0) - np.percentile(old_solar_daily, 5, axis=0)
new_width = np.percentile(new_solar_daily, 95, axis=0) - np.percentile(new_solar_daily, 5, axis=0)

real_daily_solar = df_train["solar"].resample("D").mean()
real_width_by_doy = (
    real_daily_solar.groupby(real_daily_solar.index.dayofyear).quantile(0.95)
    - real_daily_solar.groupby(real_daily_solar.index.dayofyear).quantile(0.05)
)
real_width_aligned = real_width_by_doy.reindex(day_index.dayofyear).to_numpy()

print(f"\nMean 5-95% envelope width for solar daily-mean (W/m2), averaged across the year:")
print(f"  OLD (linear, additive+clip):  {old_width.mean():.1f}")
print(f"  NEW (log-space, current):     {new_width.mean():.1f}")
print(f"  REAL inter-annual spread:     {real_width_aligned.mean():.1f}  (5-95% across {df_train.index.year.nunique()} real training years, same day-of-year)")

winter_mask = day_index.month.isin([12, 1, 2])
print(f"\nSame, winter days only:")
print(f"  OLD:  {old_width[winter_mask].mean():.1f}")
print(f"  NEW:  {new_width[winter_mask].mean():.1f}")
print(f"  REAL: {real_width_aligned[winter_mask].mean():.1f}")

# coverage of 1963, additive against log space

extreme_daily_solar = df_extreme["solar"].resample("D").mean()
extreme_doy = extreme_daily_solar.index.dayofyear

old_lo = np.percentile(old_solar_daily, 5, axis=0)
old_hi = np.percentile(old_solar_daily, 95, axis=0)
new_lo = np.percentile(new_solar_daily, 5, axis=0)
new_hi = np.percentile(new_solar_daily, 95, axis=0)

old_lo_at_1963 = np.interp(extreme_doy, day_index.dayofyear, old_lo)
old_hi_at_1963 = np.interp(extreme_doy, day_index.dayofyear, old_hi)
new_lo_at_1963 = np.interp(extreme_doy, day_index.dayofyear, new_lo)
new_hi_at_1963 = np.interp(extreme_doy, day_index.dayofyear, new_hi)

old_coverage_1963 = ((extreme_daily_solar.to_numpy() >= old_lo_at_1963) & (extreme_daily_solar.to_numpy() <= old_hi_at_1963)).mean()
new_coverage_1963 = ((extreme_daily_solar.to_numpy() >= new_lo_at_1963) & (extreme_daily_solar.to_numpy() <= new_hi_at_1963)).mean()

print(f"\n1963 solar coverage: OLD (linear) = {old_coverage_1963:.1%}   NEW (log-space, current) = {new_coverage_1963:.1%}")


In [ ]:
# 16. VAR with irradiance in log space against linear space: fitted wind-irradiance innovation correlation, its
#     tail dependence in rank space and the simulated rates

var_names = list(var_results.names)
wind_i, solar_i = var_names.index("wind"), var_names.index("solar")

sigma_u_new = np.asarray(var_results.sigma_u)
corr_new = sigma_u_new[wind_i, solar_i] / np.sqrt(sigma_u_new[wind_i, wind_i] * sigma_u_new[solar_i, solar_i])

df_anom_lin_var = df_anom.copy()
df_anom_lin_var["solar"] = df_anom_solar_lin["solar"]
print("Fitting counterfactual VAR (linear-space solar)...")
var_results_lin, _ = fit_var_model(df_anom_train=df_anom_lin_var, variables=variables, maxlags=AR_MAX_LAG, criterion="bic")
sigma_u_old = np.asarray(var_results_lin.sigma_u)
corr_old = sigma_u_old[wind_i, solar_i] / np.sqrt(sigma_u_old[wind_i, wind_i] * sigma_u_old[solar_i, solar_i])

print(f"\n(a) Fitted wind-solar innovation correlation:")
print(f"  OLD (linear-space solar):        {corr_old:+.4f}")
print(f"  NEW (log-space solar, current):  {corr_new:+.4f}")

print(f"\n(b) sigma_u-only rank-space tail dependence (200k draws, AR dynamics/year-bootstrap bypassed, q={DUNKELFLAUTE_QUANTILE}):")
for label, sigma_u in [("OLD (linear-space solar)", sigma_u_old), ("NEW (log-space solar, current)", sigma_u_new)]:
    rng_direct = np.random.default_rng(20260904)
    innov = rng_direct.multivariate_normal(mean=np.zeros(len(variables)), cov=sigma_u, size=200_000)
    innov_df = pd.DataFrame(innov, columns=variables)
    q_wind = np.quantile(innov_df["wind"], DUNKELFLAUTE_QUANTILE)
    q_solar = np.quantile(innov_df["solar"], DUNKELFLAUTE_QUANTILE)
    joint = ((innov_df["wind"] <= q_wind) & (innov_df["solar"] <= q_solar)).mean()
    print(f"  {label}: joint-tail rate = {joint:.4f}  ({joint / DUNKELFLAUTE_QUANTILE**2:.2f}x independence)")


def simulate_var_physical_old_solar(var_results, target_mean, variables, seed, burn_in=500):
    sim_anom = simulate_var_path(var_results=var_results, n_steps=len(target_mean), index=target_mean.index, seed=seed, burn_in=burn_in)
    sim_phys = sim_anom[variables] + target_mean[variables]
    sim_phys["wind"] = wind_inv(sim_phys["wind"].to_numpy())
    sim_phys["solar"] = sim_phys["solar"].clip(lower=0)  # additive reconstruction, clipped at 0 (no exp)
    return sim_phys


print(f"\n(c) Simulating {N_TRAJECTORIES} full VAR trajectories with linear-space solar...")
varlin_trajectories = {}
for j, seed in enumerate(seeds):
    year = drawn_years[j]
    target_mean_j = pd.DataFrame({
        "wind": target_mean_batch["wind"][j],
        "temp": target_mean_batch["temp"][j],
        "solar": sim_calendar_solar_lin_arr + year_baseline_offset_solar_lin.loc[year, "solar"],
    }, index=target_index)
    varlin_trajectories[j] = simulate_var_physical_old_solar(var_results_lin, target_mean_j, variables, seed=seed + 4_000_000, burn_in=500)

pooled_varlin = pooled_trajectories(varlin_trajectories, variables)
p_wind_varlin, p_solar_varlin = marginal_rates(pooled_varlin)
mult_varlin = dunkelflaute_rate(pooled_varlin) / (p_wind_varlin * p_solar_varlin)

var_row = decomposition_df[decomposition_df["model"] == "VAR"].iloc[0]
print(f"\nVAR, full ensemble:")
print(f"  OLD (linear solar):   P(low wind)={p_wind_varlin:.4f} ({p_wind_varlin/real_p_wind:.2f}x)   "
      f"P(low solar)={p_solar_varlin:.4f} ({p_solar_varlin/real_p_solar:.2f}x)   dependence={mult_varlin:.2f}x")
print(f"  NEW (log solar, S13): P(low wind)={var_row['P(low wind)']:.4f} ({var_row['P(low wind)']/real_p_wind:.2f}x)   "
      f"P(low solar)={var_row['P(low solar)']:.4f} ({var_row['P(low solar)']/real_p_solar:.2f}x)   dependence={var_row['dependence multiplier']:.2f}x")


In [ ]:
# 17. Dependence that comes from the shared year-bootstrap baseline and from the common seasonal cycle of the
#     marginal rates

target_mean_wind = target_mean_batch["wind"]
target_mean_solar_phys = np.exp(target_mean_batch["solar"]) - SOLAR_LOG_EPS  # solar's target_mean is log-space (Section 4)

daylight_mask_target = is_daylight(target_index)
tm_wind_daylight = target_mean_wind[:, daylight_mask_target]
tm_solar_daylight = target_mean_solar_phys[:, daylight_mask_target]

p_wind_tm = (tm_wind_daylight <= wind_thresh).mean()
p_solar_tm = (tm_solar_daylight <= solar_thresh).mean()

print("Pure deterministic target_mean pathway (zero residual noise - climatology + year-bootstrap offset only, shared by all three arms):")
print(f"  P(low wind)  = {p_wind_tm:.4f}")
print(f"  P(low solar) = {p_solar_tm:.4f}")
if p_wind_tm == 0:
    print("  -> P(low wind) = 0: the mean with the year offset never reaches the wind threshold")
else:
    independence_tm = p_wind_tm * p_solar_tm
    joint_tm = ((tm_wind_daylight <= wind_thresh) & (tm_solar_daylight <= solar_thresh)).mean()
    multiplier_tm = joint_tm / independence_tm
    print(f"  independence-implied joint rate = {independence_tm:.4f}")
    print(f"  actual joint rate (deterministic mean only) = {joint_tm:.4f}")
    print(f"  dependence multiplier = {multiplier_tm:.2f}x")

print(f"\nFor comparison, each arm's FULL dependence multiplier (residual noise included, Section 13):")
print(decomposition_df[["model", "dependence multiplier"]].round(4).to_string(index=False))
print(f"real data: {real_dependence_multiplier:.2f}x")

# per season: P(low wind) and P(low irradiance) of the model AR (no dependence mechanism), from the
# ensembles of Section 8

ar_pooled = pooled_trajectories(ar_trajectories, variables)
ar_pooled_season = ar_pooled.index.month.map(get_season)

print(f"\nAR model (no dependence mechanism), P(low wind) / P(low solar) by season:")
season_rows = []
for season in SEASONS:
    sub = ar_pooled.loc[ar_pooled_season == season]
    daylight = is_daylight(sub.index)
    sub_day = sub.loc[daylight]
    p_w = (sub_day["wind"] <= wind_thresh).mean()
    p_s = (sub_day["solar"] <= solar_thresh).mean()
    season_rows.append({"season": season, "P(low wind)": p_w, "P(low solar)": p_s})
    print(f"  {season:<8} P(low wind)={p_w:.4f}   P(low solar)={p_s:.4f}")

season_rate_df = pd.DataFrame(season_rows)
print()
print(season_rate_df.round(4))


In [ ]:
# 18. Wind in two regions (north 51-55 N, south 47-51 N): seasonal AR per region and a Student-t copula between
#     the regional residuals per season, calibrated to the observed Kendall's tau; the regional series are cached
#     in data/ns_wind_2000-2025.pkl (FORCE_RELOAD_NS_WIND = True reloads them)

import os
import pickle

import xarray as xr

NS_WIND_CACHE = "./data/ns_wind_2000-2025.pkl"
FORCE_RELOAD_NS_WIND = False

SUBREGIONS = {
    "North": {"lat_min": 51, "lat_max": 55, "lon_min": 5, "lon_max": 16},
    "South": {"lat_min": 47, "lat_max": 51, "lon_min": 5, "lon_max": 16},
}
NS_COLS = ["North_wind", "South_wind"]


def load_year_subregion(var_code, var_name, year, mbc_or_noc, region, source_dir, filename_suffix=""):
    filename = (
        f"H_ERA5_ECMW_T639_{var_code}_Pecd_025d_"
        f"S{year}01010000_E{year}12312300_INS_MAP_01h_COM_"
        f"{mbc_or_noc}_org_NA_NA---_NA---_NA---_PECD4.2_fv1{filename_suffix}.nc"
    )
    path = os.path.join(source_dir, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with xr.open_dataset(path) as ds:
        data = ds[var_name]
        latitude = data["latitude"].values
        lat_slice = (
            slice(region["lat_max"], region["lat_min"])
            if latitude[0] > latitude[-1]
            else slice(region["lat_min"], region["lat_max"])
        )
        ts = (
            data
            .sel(latitude=lat_slice, longitude=slice(region["lon_min"], region["lon_max"]))
            .mean(dim=["latitude", "longitude"])
            .load()
        )
    return ts


if FORCE_RELOAD_NS_WIND or not os.path.exists(NS_WIND_CACHE):
    print("Loading North/South wind from raw PECD/ERA5 grids (this takes a few minutes)...")
    records = {f"{region}_wind": [] for region in SUBREGIONS}
    for year in [*train_years, test_year]:
        src_dir = base_dir if year >= PRE2014_CUTOFF else pre2014_dir
        suffix = "" if year >= PRE2014_CUTOFF else pre2014_suffix
        print(f"  Loading {year}")
        for region_name, region in SUBREGIONS.items():
            wind_ts = load_year_subregion("WS-_0010m", "ws10", year, "mbc", region, src_dir, filename_suffix=suffix)
            records[f"{region_name}_wind"].append(pd.Series(wind_ts.values, index=pd.to_datetime(wind_ts["time"].values)))
    df_ns_wind = pd.DataFrame({k: pd.concat(v).sort_index() for k, v in records.items()})
    df_ns_wind = df_ns_wind[~df_ns_wind.index.duplicated(keep="first")].sort_index()
    df_ns_wind.to_pickle(NS_WIND_CACHE)
    print(f"Cached to {NS_WIND_CACHE}")
else:
    print(f"Loading North/South wind from cache: {NS_WIND_CACHE}")
    df_ns_wind = pd.read_pickle(NS_WIND_CACHE)

df_train_ns = df_ns_wind[df_ns_wind.index.year < test_year].copy()
df_test_ns = df_ns_wind[df_ns_wind.index.year == test_year].copy()
print(f"Train: {df_train_ns.shape}  Test: {df_test_ns.shape}")


def build_smoothed_climatology_ns(df, cols, window_days=15):
    raw = df.groupby([df.index.month, df.index.day, df.index.hour])[cols].mean()
    raw.index.names = ["month", "day", "hour"]
    days_in_month = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    calendar_order = [(m, d) for m in range(1, 13) for d in range(1, days_in_month[m - 1] + 1)]
    smoothed_blocks = []
    for hour in range(24):
        vals = raw.xs(hour, level="hour").reindex(calendar_order)[cols].to_numpy(dtype=float)
        padded = np.concatenate([vals[-window_days:], vals, vals[:window_days]], axis=0)
        smoothed_vals = (
            pd.DataFrame(padded).rolling(2 * window_days + 1, center=True, min_periods=1).mean()
            .to_numpy()[window_days:-window_days]
        )
        block = pd.DataFrame(smoothed_vals, columns=cols)
        block["month"] = [m for m, d in calendar_order]
        block["day"] = [d for m, d in calendar_order]
        block["hour"] = hour
        smoothed_blocks.append(block)
    return pd.concat(smoothed_blocks, ignore_index=True).set_index(["month", "day", "hour"])[cols]


def map_climatology_ns(index, climatology, cols):
    cal = pd.DataFrame({"month": index.month, "day": index.day, "hour": index.hour}, index=index)
    mapped = cal.merge(climatology, left_on=["month", "day", "hour"], right_index=True, how="left")[cols]
    mapped.index = index
    return mapped


def fit_linear_trend_ns(df, cols, window_days=15):
    rough_climatology = build_smoothed_climatology_ns(df, cols, window_days=window_days)
    rough_mean = map_climatology_ns(df.index, rough_climatology, cols)
    rough_anom = df[cols] - rough_mean
    t_years = (df.index - df.index[0]).total_seconds() / (365.25 * 24 * 3600)
    trend_coefs = {}
    for col in cols:
        slope, intercept = np.polyfit(t_years, rough_anom[col].to_numpy(), 1)
        trend_coefs[col] = {"slope": slope, "intercept": intercept, "t0": df.index[0]}
        print(f"  {col}: trend = {slope:+.4f} /year ({slope*10:+.3f} /decade)")
    return trend_coefs


def trend_value_ns(index, trend_coefs, cols):
    out = pd.DataFrame(index=index)
    for col in cols:
        t0 = trend_coefs[col]["t0"]
        t_years = (index - t0).total_seconds() / (365.25 * 24 * 3600)
        out[col] = trend_coefs[col]["intercept"] + trend_coefs[col]["slope"] * t_years
    return out


print("\nFitting linear trends...")
trend_coefs_ns = fit_linear_trend_ns(df_train_ns, NS_COLS, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)

df_train_detrended_ns = df_train_ns[NS_COLS] - trend_value_ns(df_train_ns.index, trend_coefs_ns, NS_COLS)
climatology_ns = build_smoothed_climatology_ns(df_train_detrended_ns, NS_COLS, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)

train_mean_ns = map_climatology_ns(df_train_ns.index, climatology_ns, NS_COLS) + trend_value_ns(df_train_ns.index, trend_coefs_ns, NS_COLS)
df_anom_ns = df_train_ns[NS_COLS] - train_mean_ns

year_baseline_offset_ns = df_anom_ns.groupby(df_anom_ns.index.year).mean()
sim_calendar_mean_ns = map_climatology_ns(df_test_ns.index, climatology_ns, NS_COLS)

print("\nYear-baseline offset summary:")
print(year_baseline_offset_ns.describe().loc[["min", "mean", "max"]])

print("\nFitting seasonal AR models for North/South wind...")
models_seasonal_ns = {season: {} for season in SEASONS}
residuals_by_season_ns = {}
for season in SEASONS:
    block_id = season_block_ids(df_anom_ns.index, season)
    season_resid = pd.DataFrame(index=df_anom_ns.index)
    for col in NS_COLS:
        fit = fit_ar_model_seasonal_one(y=df_anom_ns[col], block_id=block_id, col=col, maxlag=AR_MAX_LAG)
        models_seasonal_ns[season][col] = fit
        season_resid[col] = fit["resid"]
        print(f"  {season} {col}: AR order {len(fit['lags'])}")
    residuals_by_season_ns[season] = season_resid.dropna()

residuals_seasonal_ns = pd.concat(
    [df.assign(season=season) for season, df in residuals_by_season_ns.items()]
).sort_index()

SPATIAL_CANDIDATE_FAMILIES = [pv.BicopFamily.gaussian, pv.BicopFamily.student, pv.BicopFamily.clayton,
                               pv.BicopFamily.gumbel, pv.BicopFamily.frank, pv.BicopFamily.bb1]

# copula strength per season, calibrated to the observed north-south Kendall's tau
SPATIAL_CALIBRATED_STRENGTH = {"winter": 2.48, "spring": 1.69, "summer": 1.83, "autumn": 2.02}

print("\nFitting cross-region copula per season...")
spatial_copulas = {}
for season in SEASONS:
    sub = residuals_seasonal_ns.loc[residuals_seasonal_ns["season"] == season]
    n = len(sub)
    u_north = rankdata(sub["North_wind"], method="average") / (n + 1)
    u_south = rankdata(sub["South_wind"], method="average") / (n + 1)
    data = np.asfortranarray(np.column_stack([u_north, u_south]))
    controls = pv.FitControlsBicop(family_set=SPATIAL_CANDIDATE_FAMILIES, selection_criterion="bic",
                                    parametric_method="mle", allow_rotations=True)
    bicop = pv.Bicop()
    bicop.select(data=data, controls=controls)
    base_params = np.asarray(bicop.parameters).astype(float).copy()
    shrunk_params = base_params * SPATIAL_CALIBRATED_STRENGTH[season]
    lb = np.asarray(bicop.parameters_lower_bounds).reshape(shrunk_params.shape)
    ub = np.asarray(bicop.parameters_upper_bounds).reshape(shrunk_params.shape)
    shrunk_params = np.clip(shrunk_params, lb, ub)
    spatial_copulas[season] = pv.Bicop(family=bicop.family, rotation=bicop.rotation, parameters=shrunk_params)
    print(f"  {season:<8} family={bicop.family}  base_tau={bicop.tau:+.3f}  "
          f"calibrated_tau={spatial_copulas[season].tau:+.3f}  (strength={SPATIAL_CALIBRATED_STRENGTH[season]})")

pool_by_season_ns = {
    season: residuals_seasonal_ns.loc[residuals_seasonal_ns["season"] == season, NS_COLS]
    for season in SEASONS
}


In [ ]:
# 19. Ensemble of the two-region wind model; Dunkelflaute rate with both regions in their bottom decile against
#     the rate of the national mean

def reconstruct_ns_batch(resid_sim_batch, models_seasonal, target_mean_batch, target_index, train_anom, cols, get_season_fn):
    N, T = next(iter(resid_sim_batch.values())).shape
    season_labels = target_index.month.map(get_season_fn).to_numpy()
    out = {}
    for col in cols:
        train_history = train_anom[col].dropna().to_numpy(dtype=float)
        max_lag = max((max(models_seasonal[s][col]["lags"], default=0) for s in models_seasonal), default=0)
        step_const = np.empty(T, dtype=float)
        step_coefs = []
        for t in range(T):
            model = models_seasonal[season_labels[t]][col]
            params = model["params"]
            lags = model["lags"]
            step_const[t] = float(params["const"])
            step_coefs.append({lag: float(params[f"{col}.L{lag}"]) for lag in lags})
        sim = np.empty((N, max_lag + T), dtype=float)
        if max_lag > 0:
            pre = train_history[-max_lag:]
            sim[:, :max_lag] = pre[np.newaxis, :]
        for t in range(T):
            i = t + max_lag
            value = np.full(N, step_const[t], dtype=float)
            for lag, coef in step_coefs[t].items():
                value += coef * sim[:, i - lag]
            value += resid_sim_batch[col][:, t]
            sim[:, i] = value
        phys = sim[:, max_lag:] + target_mean_batch[col]
        phys = np.clip(phys, 0, None)  # both columns are wind speeds
        out[col] = phys
    return out


seeds_ns = [BASE_SEED + r for r in range(N_TRAJECTORIES)]
target_index_ns = sim_calendar_mean_ns.index
T_ns = len(target_index_ns)
sim_calendar_arr_ns = {col: sim_calendar_mean_ns[col].to_numpy() for col in NS_COLS}
season_labels_target_ns = np.array([get_season(m) for m in target_index_ns.month])

target_mean_batch_ns = {col: np.empty((N_TRAJECTORIES, T_ns)) for col in NS_COLS}
for j, seed in enumerate(seeds_ns):
    rng = np.random.default_rng(seed + 1_000_000)
    year = int(rng.choice(year_baseline_offset_ns.index.to_numpy()))
    offset = year_baseline_offset_ns.loc[year]
    for col in NS_COLS:
        target_mean_batch_ns[col][j] = sim_calendar_arr_ns[col] + offset[col]

print(f"Simulating {N_TRAJECTORIES} North/South wind trajectories with the calibrated spatial copula...")
season_seed_offsets = {"winter": 0, "spring": 1000, "summer": 2000, "autumn": 3000}
resid_sim_batch_ns = {col: np.empty((N_TRAJECTORIES, T_ns)) for col in NS_COLS}
for j, seed in enumerate(seeds_ns):
    for season in SEASONS:
        mask = season_labels_target_ns == season
        n_season = int(mask.sum())
        bicop = spatial_copulas[season]
        u_pair = np.asarray(bicop.simulate(n=n_season, seeds=[int(seed) + season_seed_offsets[season]]))
        pool = pool_by_season_ns[season]
        n_pool = len(pool)
        north_sorted = np.sort(pool["North_wind"].to_numpy())
        south_sorted = np.sort(pool["South_wind"].to_numpy())
        north_vals = north_sorted[np.clip((u_pair[:, 0] * n_pool).astype(int), 0, n_pool - 1)]
        south_vals = south_sorted[np.clip((u_pair[:, 1] * n_pool).astype(int), 0, n_pool - 1)]
        resid_sim_batch_ns["North_wind"][j, mask] = north_vals
        resid_sim_batch_ns["South_wind"][j, mask] = south_vals

phys_batch_ns = reconstruct_ns_batch(resid_sim_batch_ns, models_seasonal_ns, target_mean_batch_ns, target_index_ns,
                                      df_anom_ns, NS_COLS, get_season)
ns_trajectories = {j: pd.DataFrame({col: phys_batch_ns[col][j] for col in NS_COLS}, index=target_index_ns)
                    for j in range(N_TRAJECTORIES)}
pooled_ns = pd.concat(ns_trajectories.values(), axis=0)
print("Done.")

north_thresh = df_train_ns["North_wind"].quantile(DUNKELFLAUTE_QUANTILE)
south_thresh = df_train_ns["South_wind"].quantile(DUNKELFLAUTE_QUANTILE)
avg_thresh_ns = ((df_train_ns["North_wind"] + df_train_ns["South_wind"]) / 2).quantile(DUNKELFLAUTE_QUANTILE)

pooled_ns["avg_wind"] = (pooled_ns["North_wind"] + pooled_ns["South_wind"]) / 2
sim_agg_low = (pooled_ns["avg_wind"] <= avg_thresh_ns).mean()
sim_both_low = ((pooled_ns["North_wind"] <= north_thresh) & (pooled_ns["South_wind"] <= south_thresh)).mean()

real_avg_ns = (df_train_ns["North_wind"] + df_train_ns["South_wind"]) / 2
real_agg_low = (real_avg_ns <= avg_thresh_ns).mean()
real_both_low = ((df_train_ns["North_wind"] <= north_thresh) & (df_train_ns["South_wind"] <= south_thresh)).mean()

print("\n" + "=" * 70)
print(f"Aggregate-average vs both-region-low wind rate (N={N_TRAJECTORIES})")
print("=" * 70)
print(f"{'':32}{'aggregate-avg':>16}{'both regions low':>20}")
print(f"{'REAL (' + str(train_years.start) + '-' + str(train_years.stop - 1) + ')':32}{real_agg_low:>16.1%}{real_both_low:>20.1%}")
print(f"{'SIMULATED (N=' + str(N_TRAJECTORIES) + ')':32}{sim_agg_low:>16.1%}{sim_both_low:>20.1%}")
print(f"\nReal ratio (both/aggregate)      = {real_both_low/real_agg_low:.2f}x")
print(f"Simulated ratio (both/aggregate) = {sim_both_low/sim_agg_low:.2f}x")


In [ ]:
# 20. Both-region wind deficits for the documented periods and 1963, observed and simulated (irradiance of the
#     model AR + Copula combined trajectory by trajectory with the two-region wind)

import os

EXTREME_DIR_1963 = "./data/ERA-5-extreme-years"
EXTREME_SUFFIX_1963 = ".area-subset.75.45.18.-31"
NS_1963_CACHE = "./data/ns_wind_1963.pkl"
FORCE_RELOAD_NS_1963 = False

if FORCE_RELOAD_NS_1963 or not os.path.exists(NS_1963_CACHE):
    print("Fetching 1963 North/South wind (not covered by the 2000-2025 cache)...")
    records_1963 = {}
    for region_name, region in SUBREGIONS.items():
        ts = load_year_subregion("WS-_0010m", "ws10", 1963, "mbc", region, EXTREME_DIR_1963,
                                  filename_suffix=EXTREME_SUFFIX_1963)
        records_1963[f"{region_name}_wind"] = pd.Series(ts.values, index=pd.to_datetime(ts["time"].values))
    df_1963_ns = pd.DataFrame(records_1963)
    df_1963_ns.to_pickle(NS_1963_CACHE)
else:
    print(f"Loading 1963 North/South wind from cache: {NS_1963_CACHE}")
    df_1963_ns = pd.read_pickle(NS_1963_CACHE)

index_overlap = df_1963_ns.index.intersection(df_extreme.index)
assert len(index_overlap) == len(df_1963_ns), "1963 N/S wind and df_extreme indices don't fully align"
print(f"1963 N/S wind: {df_1963_ns.shape}, fully aligned with df_extreme")

# check that the year-bootstrap draws of the two ensembles match trajectory by trajectory (same seeds)

years_match = list(year_baseline_offset.index) == list(year_baseline_offset_ns.index)
sample_mismatches = 0
for j in range(N_TRAJECTORIES):
    seed = BASE_SEED + j
    year_main = int(np.random.default_rng(seed + 1_000_000).choice(year_baseline_offset.index.to_numpy()))
    year_ns = int(np.random.default_rng(seed + 1_000_000).choice(year_baseline_offset_ns.index.to_numpy()))
    if year_main != year_ns:
        sample_mismatches += 1
print(f"Year-baseline-offset index alignment (main vs. spatial): {years_match}  "
      f"| trajectory-level year-draw mismatches: {sample_mismatches}/{N_TRAJECTORIES}")
assert sample_mismatches == 0, "main and spatial ensembles draw different years per trajectory - not directly combinable"

north_thresh = df_train_ns["North_wind"].quantile(DUNKELFLAUTE_QUANTILE)
south_thresh = df_train_ns["South_wind"].quantile(DUNKELFLAUTE_QUANTILE)


def both_region_rate(wind_ns_df, solar_series):
    common_idx = wind_ns_df.index.intersection(solar_series.index)
    mask = is_daylight(common_idx)
    sub_wind = wind_ns_df.loc[common_idx].loc[mask]
    sub_solar = solar_series.loc[common_idx].loc[mask]
    event = (sub_wind["North_wind"] <= north_thresh) & (sub_wind["South_wind"] <= south_thresh) & (sub_solar <= solar_thresh)
    return event.mean()


periods_ns = {
    "2000-2024 baseline": df_train_ns[NS_COLS],
    "2017 full year": df_train_ns.loc[df_train_ns.index.year == 2017, NS_COLS],
    "Jan 16-25, 2017 event": df_train_ns.loc["2017-01-16":"2017-01-25", NS_COLS],
    "Dec 2022 - Mar 2023": pd.concat([
        df_train_ns.loc["2022-12-01":"2022-12-31", NS_COLS],
        df_train_ns.loc["2023-01-01":"2023-03-31", NS_COLS],
    ]),
    "2010 (in-sample)": df_train_ns.loc[df_train_ns.index.year == 2010, NS_COLS],
    "1963 (out-of-sample)": df_1963_ns,
}
periods_agg = {
    "2000-2024 baseline": df_train[["wind", "solar"]],
    "2017 full year": df_train.loc[df_train.index.year == 2017, ["wind", "solar"]],
    "Jan 16-25, 2017 event": df_train.loc["2017-01-16":"2017-01-25", ["wind", "solar"]],
    "Dec 2022 - Mar 2023": pd.concat([
        df_train.loc["2022-12-01":"2022-12-31", ["wind", "solar"]],
        df_train.loc["2023-01-01":"2023-03-31", ["wind", "solar"]],
    ]),
    "2010 (in-sample)": df_train.loc[df_train.index.year == 2010, ["wind", "solar"]],
    "1963 (out-of-sample)": df_extreme[["wind", "solar"]],
}

print("\n" + "=" * 78)
print("Real/observed: aggregate-average (Section 12's definition) vs. both-regions-low")
print("=" * 78)
print(f"{'period':<26}{'aggregate rate':>16}{'both-region rate':>19}{'ratio':>10}")
both_region_summary_rows = []
for name, wind_period in periods_ns.items():
    solar_period = df_extreme["solar"] if name == "1963 (out-of-sample)" else df_train["solar"].reindex(wind_period.index)
    agg_rate = dunkelflaute_rate(periods_agg[name])
    both_rate = both_region_rate(wind_period, solar_period)
    ratio = both_rate / agg_rate if agg_rate > 0 else float("nan")
    both_region_summary_rows.append({"period": name, "aggregate_rate": agg_rate, "both_region_rate": both_rate, "ratio": ratio})
    print(f"{name:<26}{agg_rate:>15.2%} {both_rate:>18.2%} {ratio:>9.2f}x")
both_region_summary_df = pd.DataFrame(both_region_summary_rows)

print("\n" + "=" * 78)
print(f"Simulated (N={N_TRAJECTORIES}): aggregate vs. both-regions-low, AR+Copula's solar")
print("=" * 78)
agg_rates_sim, both_rates_sim = [], []
for j in range(N_TRAJECTORIES):
    solar_j = arcop_trajectories[j]["solar"]
    agg_wind_j = arcop_trajectories[j]["wind"]
    wind_ns_j = ns_trajectories[j]
    common_idx = wind_ns_j.index.intersection(solar_j.index)
    mask = is_daylight(common_idx)
    agg_event = (agg_wind_j.loc[common_idx].loc[mask] <= wind_thresh) & (solar_j.loc[common_idx].loc[mask] <= solar_thresh)
    both_event = ((wind_ns_j["North_wind"].loc[common_idx].loc[mask] <= north_thresh) &
                  (wind_ns_j["South_wind"].loc[common_idx].loc[mask] <= south_thresh) &
                  (solar_j.loc[common_idx].loc[mask] <= solar_thresh))
    agg_rates_sim.append(agg_event.mean())
    both_rates_sim.append(both_event.mean())
agg_rates_sim, both_rates_sim = np.array(agg_rates_sim), np.array(both_rates_sim)

print(f"{'':30}{'mean rate':>12}{'5-95% range':>24}")
print(f"{'aggregate-average wind':30}{agg_rates_sim.mean():>11.2%} "
      f"  [{np.percentile(agg_rates_sim,5):.2%}, {np.percentile(agg_rates_sim,95):.2%}]")
print(f"{'both-regions-low wind':30}{both_rates_sim.mean():>11.2%} "
      f"  [{np.percentile(both_rates_sim,5):.2%}, {np.percentile(both_rates_sim,95):.2%}]")
print(f"\nSimulated ratio (both/aggregate) = {both_rates_sim.mean()/agg_rates_sim.mean():.2f}x  "
      f"(real baseline ratio: {both_region_summary_df.loc[both_region_summary_df.period=='2000-2024 baseline', 'ratio'].iloc[0]:.2f}x)")



In [ ]:
# 21. Wind and irradiance components of December 2022 - March 2023, with January 2017 as a reference

period_mask = ((df_train.index >= "2022-12-01") & (df_train.index <= "2022-12-31")) | \
              ((df_train.index >= "2023-01-01") & (df_train.index <= "2023-03-31"))
period = df_train.loc[period_mask]
period_ns = df_train_ns.loc[period_mask]
daylight_mask_period = is_daylight(period.index)
period_day = period.loc[daylight_mask_period]
period_ns_day = period_ns.loc[daylight_mask_period]

p_solar_period = (period_day["solar"] <= solar_thresh).mean()
p_wind_agg_period = (period_day["wind"] <= wind_thresh).mean()
p_north_period = (period_ns_day["North_wind"] <= north_thresh).mean()
p_south_period = (period_ns_day["South_wind"] <= south_thresh).mean()

print("Marginal low-rate, this period vs. the 25-year baseline (baseline is ~10% by construction):")
print(f"  P(low solar)           = {p_solar_period:.1%}  (ratio {p_solar_period/0.10:.2f}x)")
print(f"  P(low wind, aggregate) = {p_wind_agg_period:.1%}  (ratio {p_wind_agg_period/0.10:.2f}x)")
print(f"  P(low North wind)      = {p_north_period:.1%}  (ratio {p_north_period/0.10:.2f}x)")
print(f"  P(low South wind)      = {p_south_period:.1%}  (ratio {p_south_period/0.10:.2f}x)")

agg_event_mask = (period_day["wind"] <= wind_thresh) & (period_day["solar"] <= solar_thresh)
event_ns = period_ns_day.loc[agg_event_mask]
n_events = len(event_ns)
both_low_frac = ((event_ns["North_wind"] <= north_thresh) & (event_ns["South_wind"] <= south_thresh)).mean()
north_low_frac = (event_ns["North_wind"] <= north_thresh).mean()
south_low_frac = (event_ns["South_wind"] <= south_thresh).mean()

print(f"\nOf the {n_events} hours that DID register as aggregate-dunkelflaute this period:")
print(f"  North also independently low: {north_low_frac:.1%}")
print(f"  South also independently low: {south_low_frac:.1%}")
print(f"  BOTH independently low (genuine system-wide): {both_low_frac:.1%}")

low_solar_daily = (period_day["solar"] <= solar_thresh).resample("D").mean()
majority_low_days = int((low_solar_daily > 0.5).sum())
is_low = (low_solar_daily > 0.5).to_numpy()
longest_run = current_run = 0
for v in is_low:
    current_run = current_run + 1 if v else 0
    longest_run = max(longest_run, current_run)
print(f"\nDays with >50% of daylight hours below the solar threshold: {majority_low_days}/{len(low_solar_daily)} "
      f"(longest consecutive run: {longest_run} days)")

# reference comparison: the known-genuine Jan 2017 event, same decomposition
period_jan17 = df_train.loc["2017-01-16":"2017-01-25"]
period_jan17_ns = df_train_ns.loc["2017-01-16":"2017-01-25"]
mask_jan17 = is_daylight(period_jan17.index)
event_jan17 = period_jan17_ns.loc[mask_jan17].loc[
    (period_jan17.loc[mask_jan17, "wind"] <= wind_thresh) & (period_jan17.loc[mask_jan17, "solar"] <= solar_thresh)
]
both_low_jan17 = ((event_jan17["North_wind"] <= north_thresh) & (event_jan17["South_wind"] <= south_thresh)).mean()
print(f"\nFor reference, Jan 16-25 2017: {both_low_jan17:.1%} of its qualifying hours had both regions low")



In [ ]:
# 22. Significance tests (paired t-test, paired bootstrap interval, Benjamini-Hochberg over all six tests):
#     (A) RMSE of the irradiance daily means with the temporal copula and with the joint draw of the seasonal
#     vine; (B) gap between the national-mean and the both-region wind deficit rate, simulated and observed

from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

all_tests_s22 = []


def paired_bootstrap_ci(diff_array, n_boot=10000, rng=None):
    rng = rng or np.random.default_rng(0)
    n = len(diff_array)
    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_means[b] = diff_array[idx].mean()
    return np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)


rng_boot_s22 = np.random.default_rng(777)

# PART A: RMSE of the irradiance daily means with the temporal copula (arcop_trajectories) and with the joint
# draw of the seasonal vine instead, from the same target_mean_batch and seeds (paired)

print("Simulating the AR+Copula ensemble with the joint draw of the seasonal vine (no temporal copula of irradiance)...")
resid_sim_batch_pre = {col: np.empty((N_TRAJECTORIES, T)) for col in variables}
for j, seed in enumerate(seeds):
    U_sim = simulate_seasonal_copulas(seasonal_vines=seasonal_vines, target_index=target_index, variables=variables, seed=seed)
    resid_sim = apply_margins_seasonal(U_sim=U_sim, margin_params_seasonal=margin_params_seasonal, variables=variables, eps=eps)
    for col in variables:
        resid_sim_batch_pre[col][j] = resid_sim[col].to_numpy()
    resid_sim_batch_pre["wind"][j] = resid_sim_batch_pre["wind"][j] * wind_scale_array
pre_batch_s22 = reconstruct_simulation_seasonal_batch(
    resid_sim_batch_pre, models_seasonal, target_mean_batch, target_index, df_anom, variables, get_season
)
pre_fix7_trajectories = unbatch(pre_batch_s22, target_index, variables)


def per_trajectory_daily(trajectories, col):
    out = {}
    day_index = None
    for j, sim in trajectories.items():
        d = sim[col].resample("D").mean()
        day_index = d.index
        out[j] = d.to_numpy()
    return out, day_index


def rmse_per_trajectory(traj_daily_dict, day_of_year, real_series):
    real_doy = real_series.index.dayofyear
    real_vals = real_series.to_numpy()
    rmses = np.empty(len(traj_daily_dict))
    for j in range(len(traj_daily_dict)):
        interp_vals = np.interp(real_doy, day_of_year, traj_daily_dict[j])
        rmses[j] = np.sqrt(np.mean((interp_vals - real_vals) ** 2))
    return rmses


post_daily_s22, day_index_s22 = per_trajectory_daily(arcop_trajectories, "solar")
pre_daily_s22, _ = per_trajectory_daily(pre_fix7_trajectories, "solar")
day_of_year_s22 = day_index_s22.dayofyear

print(f"\n{'year':<8}{'vine RMSE':>16}{'temporal RMSE':>18}{'mean diff':>14}{'t-stat':>10}{'p-value':>12}{'95% CI (diff)':>24}")
for year_label, year_df in extreme_years_to_plot.items():
    real_daily_solar = year_df["solar"].resample("D").mean()
    rmse_pre = rmse_per_trajectory(pre_daily_s22, day_of_year_s22, real_daily_solar)
    rmse_post = rmse_per_trajectory(post_daily_s22, day_of_year_s22, real_daily_solar)
    diff = rmse_pre - rmse_post  # positive: lower RMSE with the temporal copula

    tstat, pval = ttest_rel(rmse_pre, rmse_post)
    ci_lo, ci_hi = paired_bootstrap_ci(diff, rng=rng_boot_s22)
    print(f"{year_label:<8}{rmse_pre.mean():>16.2f}{rmse_post.mean():>18.2f}{diff.mean():>14.2f}"
          f"{tstat:>10.2f}{pval:>12.6f}   [{ci_lo:.2f}, {ci_hi:.2f}]")
    all_tests_s22.append({
        "claim": "solar temporal copula", "test": f"solar_rmse_{year_label}",
        "method": "paired t-test (ttest_rel)", "statistic": tstat, "p_value": pval,
        "mean_diff": diff.mean(), "ci_lo": ci_lo, "ci_hi": ci_hi,
    })

# PART B: national-mean against both-region wind deficit rate (objects of Section 19)

agg_rates_s22 = np.empty(N_TRAJECTORIES)
both_rates_s22 = np.empty(N_TRAJECTORIES)
for j, sim in ns_trajectories.items():
    avg_wind = (sim["North_wind"] + sim["South_wind"]) / 2
    agg_rates_s22[j] = (avg_wind <= avg_thresh_ns).mean()
    both_rates_s22[j] = ((sim["North_wind"] <= north_thresh) & (sim["South_wind"] <= south_thresh)).mean()

diff_sim_s22 = agg_rates_s22 - both_rates_s22
tstat_sim, pval_sim = ttest_rel(agg_rates_s22, both_rates_s22)
ci_lo_sim, ci_hi_sim = paired_bootstrap_ci(diff_sim_s22, rng=rng_boot_s22)

print(f"\nWind B1 (simulated, N={N_TRAJECTORIES} trajectories, paired per-trajectory):")
print(f"  mean agg_low_rate={agg_rates_s22.mean():.4f}  mean both_low_rate={both_rates_s22.mean():.4f}  "
      f"mean diff={diff_sim_s22.mean():.4f}")
print(f"  paired t-test: t={tstat_sim:.2f}  p={pval_sim:.6f}   95% CI on mean diff: [{ci_lo_sim:.4f}, {ci_hi_sim:.4f}]")

all_tests_s22.append({
    "claim": "spatial wind ~3x", "test": "agg_vs_both_simulated_paired",
    "method": "paired t-test (ttest_rel)", "statistic": tstat_sim, "p_value": pval_sim,
    "mean_diff": diff_sim_s22.mean(), "ci_lo": ci_lo_sim, "ci_hi": ci_hi_sim,
})

years_available_s22 = np.sort(df_train_ns.index.year.unique())
n_years_s22 = len(years_available_s22)
year_groups_s22 = {y: df_train_ns.loc[df_train_ns.index.year == y] for y in years_available_s22}

real_diff_point_s22 = real_agg_low - real_both_low

print(f"\nBlock-bootstrapping {n_years_s22} years with replacement, N=10000 reps "
      f"(thresholds AND rates both recomputed fresh on each resampled pseudo-history)...")
rng_yearboot_s22 = np.random.default_rng(2026)
diffs_real_boot_s22 = np.empty(10000)
for b in range(10000):
    sampled_years = rng_yearboot_s22.choice(years_available_s22, size=n_years_s22, replace=True)
    resampled = pd.concat([year_groups_s22[y] for y in sampled_years], axis=0)
    avg_wind_b = (resampled["North_wind"] + resampled["South_wind"]) / 2
    north_thresh_b = resampled["North_wind"].quantile(DUNKELFLAUTE_QUANTILE)
    south_thresh_b = resampled["South_wind"].quantile(DUNKELFLAUTE_QUANTILE)
    avg_thresh_b = avg_wind_b.quantile(DUNKELFLAUTE_QUANTILE)
    agg_low_b = (avg_wind_b <= avg_thresh_b).mean()
    both_low_b = ((resampled["North_wind"] <= north_thresh_b) & (resampled["South_wind"] <= south_thresh_b)).mean()
    diffs_real_boot_s22[b] = agg_low_b - both_low_b

ci_lo_real_s22, ci_hi_real_s22 = np.percentile(diffs_real_boot_s22, [2.5, 97.5])
p_boot_real_s22 = 2 * min((diffs_real_boot_s22 <= 0).mean(), (diffs_real_boot_s22 >= 0).mean())
p_boot_real_s22 = min(max(p_boot_real_s22, 1.0 / 10000), 1.0)

print(f"\nWind B2 (real data, {n_years_s22}-year block bootstrap, N=10000):")
print(f"  point estimate diff (agg_low - both_low) = {real_diff_point_s22:.4f}  "
      f"ratio={real_agg_low/real_both_low:.2f}x")
print(f"  bootstrap 95% CI on diff: [{ci_lo_real_s22:.4f}, {ci_hi_real_s22:.4f}]")
print(f"  bootstrap-percentile two-sided p-value: {p_boot_real_s22:.6f}")

all_tests_s22.append({
    "claim": "spatial wind ~3x", "test": "agg_vs_both_real_yearboot",
    "method": "bootstrap-percentile (year block bootstrap, N=25 years)", "statistic": np.nan,
    "p_value": p_boot_real_s22, "mean_diff": real_diff_point_s22, "ci_lo": ci_lo_real_s22, "ci_hi": ci_hi_real_s22,
})

# PART C: FDR correction across the full family

results_df_s22 = pd.DataFrame(all_tests_s22)
reject_s22, q_values_s22, _, _ = multipletests(results_df_s22["p_value"].to_numpy(), alpha=0.05, method="fdr_bh")
results_df_s22["q_value"] = q_values_s22
results_df_s22["significant_fdr_q<0.05"] = reject_s22

print("\n" + "=" * 100)
print("FULL FAMILY - Benjamini-Hochberg FDR correction, alpha=0.05, across all 6 tests together")
print("=" * 100)
print(results_df_s22[["claim", "test", "method", "p_value", "q_value", "significant_fdr_q<0.05",
                       "mean_diff", "ci_lo", "ci_hi"]].round(6).to_string(index=False))
print(f"\nSignificant at FDR q<0.05: {int(results_df_s22['significant_fdr_q<0.05'].sum())} / {len(results_df_s22)}")



In [ ]:
# 23. Monte Carlo convergence: 12,800 trajectories of the model AR + Copula split into independent chunks of
#     N = 100 ... 3,200; coverage and Dunkelflaute rate per chunk size (irradiance drawn with
#     vectorized_solar_temporal.py)

import sys
sys.path.insert(0, ".")
from vectorized_solar_temporal import simulate_solar_temporal_sequence_batch

N_BIG = 12800
N_LEVELS = [100, 200, 400, 800, 1600, 3200]
MAX_REPLICATES_SHOWN = 40
BASE_SEED_BIG = 20260914

seeds_big = [BASE_SEED_BIG + r for r in range(N_BIG)]
years_avail = year_baseline_offset.index.to_numpy()

print(f"Building target_mean_batch for N_BIG={N_BIG}...")
target_mean_batch_big = {col: np.empty((N_BIG, T)) for col in variables}
for j, seed in enumerate(seeds_big):
    rng = np.random.default_rng(seed + 1_000_000)
    year = int(rng.choice(years_avail))
    offset = year_baseline_offset.loc[year]
    for col in variables:
        target_mean_batch_big[col][j] = sim_calendar_arr[col] + offset[col]

print("Generating wind/temp bootstrap residuals (vectorized across N_BIG)...")
resid_sim_batch_big = {col: np.empty((N_BIG, T)) for col in variables}
rng_boot_big = np.random.default_rng(BASE_SEED_BIG + 777)
for col in ["wind", "temp"]:
    for season in SEASONS:
        mask = season_labels_target == season
        n_season = int(mask.sum())
        pool = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, col].dropna().to_numpy()
        resid_sim_batch_big[col][:, mask] = rng_boot_big.choice(pool, size=(N_BIG, n_season), replace=True)
resid_sim_batch_big["wind"] = resid_sim_batch_big["wind"] * wind_scale_array[np.newaxis, :]

print(f"Generating solar temporal-copula residuals for N_BIG={N_BIG} (vectorized across trajectories)...")
resid_sim_batch_big["solar"] = simulate_solar_temporal_sequence_batch(
    solar_temporal_copulas=solar_temporal_copulas, solar_pool_by_season=solar_pool_by_season,
    season_labels=season_labels_target, n_traj=N_BIG, seed=BASE_SEED_BIG + 5_000_000
)

print("Reconstructing physical-units trajectories (vectorized batch reconstruction)...")
phys_batch_big = reconstruct_simulation_seasonal_batch(
    resid_sim_batch_big, models_seasonal, target_mean_batch_big, target_index, df_anom, variables, get_season
)
del resid_sim_batch_big, target_mean_batch_big

print("Computing daily means + per-trajectory dunkelflaute-hour counts...")
daily_means_big = {col: phys_batch_big[col].reshape(N_BIG, 365, 24).mean(axis=2) for col in variables}

daylight_mask_target = is_daylight(target_index)
n_daylight_per_traj = int(daylight_mask_target.sum())
dunkelflaute_mask_big = (
    (phys_batch_big["wind"] <= wind_thresh) & (phys_batch_big["solar"] <= solar_thresh)
    & daylight_mask_target[np.newaxis, :]
)
dunkelflaute_hour_count_big = dunkelflaute_mask_big.sum(axis=1)
del phys_batch_big, dunkelflaute_mask_big

extreme_daily = {col: df_extreme[col].resample("D").mean() for col in variables}
day_index_daily = pd.date_range(target_index[0], periods=365, freq="D")
day_of_year_arr = day_index_daily.dayofyear


def coverage_for_chunk(daily_arr_chunk, col):
    lo = np.percentile(daily_arr_chunk, 5, axis=0)
    hi = np.percentile(daily_arr_chunk, 95, axis=0)
    real_doy = extreme_daily[col].index.dayofyear
    real_vals = extreme_daily[col].to_numpy()
    lo_at = np.interp(real_doy, day_of_year_arr, lo)
    hi_at = np.interp(real_doy, day_of_year_arr, hi)
    return float(((real_vals >= lo_at) & (real_vals <= hi_at)).mean())


print("\nSlicing into N-level replicates and computing metrics...")
mc_conv_results = []
for N in N_LEVELS:
    n_reps = min(MAX_REPLICATES_SHOWN, N_BIG // N)
    for r in range(n_reps):
        idx = slice(r * N, (r + 1) * N)
        row = {"N": N, "replicate": r}
        for col in variables:
            row[f"coverage_{col}"] = coverage_for_chunk(daily_means_big[col][idx], col)
        row["dunkelflaute_rate"] = float(dunkelflaute_hour_count_big[idx].sum()) / (N * n_daylight_per_traj)
        mc_conv_results.append(row)
    print(f"  N={N:<6} n_replicates={n_reps}")

mc_conv_df = pd.DataFrame(mc_conv_results)
print("\nSummary (mean / std across independent replicates, by N):")
print(mc_conv_df.groupby("N")[["coverage_wind", "coverage_temp", "coverage_solar", "dunkelflaute_rate"]]
      .agg(["mean", "std"]).round(5).to_string())

print("\nDrift from N=100 to N=3200 (mean across replicates):")
for col, label in [("coverage_wind", "Wind coverage"), ("coverage_temp", "Temp coverage"),
                    ("coverage_solar", "Solar coverage"), ("dunkelflaute_rate", "Dunkelflaute rate")]:
    v100 = mc_conv_df.loc[mc_conv_df["N"] == 100, col].mean()
    v3200 = mc_conv_df.loc[mc_conv_df["N"] == 3200, col].mean()
    print(f"  {label:<18} {v100*100:.2f}% -> {v3200*100:.2f}%   drift = {(v3200-v100)*100:+.2f}pp")


In [ ]:
# 23. (continued) figure of the convergence

C_ARCOP_MC = "#EB811B"
C_REAL_MC = "#23373B"

mc_panels = [
    ("coverage_wind", "Wind coverage (1963, AR+Copula)", "coverage"),
    ("coverage_temp", "Temp coverage (1963, AR+Copula)", "coverage"),
    ("coverage_solar", "Solar coverage (1963, AR+Copula)", "coverage"),
    ("dunkelflaute_rate", "Dunkelflaute rate (P(low wind AND low solar))", "rate"),
]

fig, mc_axes = plt.subplots(2, 2, figsize=(12, 9))
mc_axes = mc_axes.ravel()

for ax, (col, title, kind) in zip(mc_axes, mc_panels):
    positions = np.arange(len(N_LEVELS))
    box_data = [mc_conv_df.loc[mc_conv_df["N"] == N, col].to_numpy() for N in N_LEVELS]
    bp = ax.boxplot(box_data, positions=positions, widths=0.5, patch_artist=True,
                     showfliers=False, medianprops=dict(color=C_REAL_MC, linewidth=1.5))
    for patch in bp["boxes"]:
        patch.set_facecolor(C_ARCOP_MC)
        patch.set_alpha(0.35)
        patch.set_edgecolor(C_REAL_MC)
    for element in ["whiskers", "caps"]:
        for line in bp[element]:
            line.set_color(C_REAL_MC)

    means = [np.mean(vals) for vals in box_data]
    ax.plot(positions, means, "o-", color=C_REAL_MC, markersize=5, linewidth=1.3, label="mean across replicates")

    if kind == "coverage":
        ax.axhline(0.90, color="#B23A2E", linestyle="--", linewidth=1.1, zorder=0)
        ax.text(positions[-1], 0.905, "nominal target (90%)", color="#B23A2E", fontsize=9, ha="right")
        ax.set_ylabel("coverage")
        ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    else:
        ax.axhline(0.0113, color=C_REAL_MC, linestyle=":", linewidth=1.1, zorder=0)
        ax.text(positions[-1], 0.0113 + 0.0002, "real 2000-2024 baseline (1.13%)", color=C_REAL_MC, fontsize=9, ha="right")
        ax.set_ylabel("P(low wind AND low solar)")
        ax.yaxis.set_major_formatter(lambda v, _: f"{v:.2%}")

    ax.set_xticks(positions)
    ax.set_xticklabels([str(n) for n in N_LEVELS])
    ax.set_xlabel("N (Monte Carlo trajectories)")
    ax.set_title(title, fontsize=13)

mc_axes[0].legend(frameon=False, loc="lower right", fontsize=10)
fig.suptitle("Monte Carlo convergence: does N=300 give a stable estimate?", fontsize=15, y=1.00)
n_reps_line = ", ".join(f"N={N}: n={min(MAX_REPLICATES_SHOWN, N_BIG // N)}" for N in N_LEVELS)
fig.text(0.5, 0.005, f"Each box = independent, non-overlapping replicates sliced from one N={N_BIG} pool.  {n_reps_line}",
          ha="center", fontsize=9, color="#888888")
fig.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.savefig("mc_convergence.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)



In [ ]:
# 24. Dunkelflaute events per year, 2000-2025: a day qualifies if its daylight-mean wind and daylight-mean
#     irradiance are both in their bottom decile (thresholds from the 2000-2024 daily means); consecutive days
#     form one event; duration and intensity (mean shortfall below the thresholds) per event

df_all_real = pd.concat([df_train[variables], df_test[variables]], axis=0).sort_index()
print("Real hourly record:", df_all_real.shape, df_all_real.index.min(), "to", df_all_real.index.max())

daylight_mask_all = is_daylight(df_all_real.index)
daylight_wind_mean = df_all_real.loc[daylight_mask_all, "wind"].resample("D").mean()
daylight_solar_mean = df_all_real.loc[daylight_mask_all, "solar"].resample("D").mean()

daily_events = pd.DataFrame({"wind_mean_daylight": daylight_wind_mean, "solar_mean_daylight": daylight_solar_mean})
daily_events = daily_events.dropna(how="all")

train_days_mask = daily_events.index.year <= 2024
wind_thresh_daily = daily_events.loc[train_days_mask, "wind_mean_daylight"].quantile(DUNKELFLAUTE_QUANTILE)
solar_thresh_daily = daily_events.loc[train_days_mask, "solar_mean_daylight"].quantile(DUNKELFLAUTE_QUANTILE)
print(f"Day-level thresholds (fit on 2000-2024 training days only, daylight-mean): "
      f"wind<={wind_thresh_daily:.3f} m/s, solar<={solar_thresh_daily:.3f} W/m2")
print(f"(for comparison, Section 12's HOURLY thresholds: wind<={wind_thresh:.3f}, solar<={solar_thresh:.3f})")

daily_events["is_dark_day"] = (daily_events["wind_mean_daylight"] <= wind_thresh_daily) & \
                               (daily_events["solar_mean_daylight"] <= solar_thresh_daily)
daily_events["wind_deficit"] = np.clip((wind_thresh_daily - daily_events["wind_mean_daylight"]) / wind_thresh_daily, 0, None)
daily_events["solar_deficit"] = np.clip((solar_thresh_daily - daily_events["solar_mean_daylight"]) / solar_thresh_daily, 0, None)
daily_events["intensity"] = (daily_events["wind_deficit"] + daily_events["solar_deficit"]) / 2

n_dark_days = int(daily_events["is_dark_day"].sum())
print(f"\nDark days (daylight-mean wind AND solar both below their bottom-decile threshold): "
      f"{n_dark_days} / {len(daily_events)} days ({n_dark_days/len(daily_events):.2%})")

# cluster consecutive dark days into events
events = []
in_event = False
for date, row in daily_events.iterrows():
    if row["is_dark_day"] and not in_event:
        start, in_event, days_in_event = date, True, [date]
    elif row["is_dark_day"] and in_event:
        days_in_event.append(date)
    elif not row["is_dark_day"] and in_event:
        events.append((start, days_in_event[-1], days_in_event))
        in_event = False
if in_event:
    events.append((start, days_in_event[-1], days_in_event))

event_rows = []
for start, end, days_in_event in events:
    sub = daily_events.loc[days_in_event]
    event_rows.append({
        "start": start, "end": end, "duration_days": len(days_in_event), "intensity": sub["intensity"].mean(),
        "wind_mean": sub["wind_mean_daylight"].mean(), "solar_mean": sub["solar_mean_daylight"].mean(),
        "year": start.year,
    })
events_df = pd.DataFrame(event_rows).sort_values("start").reset_index(drop=True)
print(f"\nTotal discrete events, 2000-2025: {len(events_df)}")
print("\nMost severe events overall (by intensity):")
print(events_df.sort_values("intensity", ascending=False).head(5)
      [["start", "end", "duration_days", "intensity", "wind_mean", "solar_mean"]].to_string(index=False))

per_year = events_df.groupby("year").agg(n_events=("duration_days", "size"), total_days=("duration_days", "sum"),
                                          max_duration=("duration_days", "max"), mean_intensity=("intensity", "mean"))
per_year = per_year.reindex(range(2000, 2026), fill_value=0)
print("\nEvents per year:")
print(per_year.to_string())
print("\nYears with zero joint events:", [int(y) for y, n in per_year["n_events"].items() if n == 0])

# January 16-25, 2017 under this definition
jan2017 = daily_events.loc["2017-01-16":"2017-01-25"]
print("\nJan 16-25, 2017 under this day-level joint definition:")
print(jan2017.to_string())


In [ ]:
# 24. (continued) figure of the events per year

years_ev = per_year.index.to_numpy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.6]})

ax1.bar(years_ev, per_year["n_events"], width=0.65, color="#EB811B")
for y, n in zip(years_ev, per_year["n_events"]):
    if n > 0:
        ax1.text(y, n + 0.08, str(int(n)), ha="center", fontsize=9, color="#23373B")
ax1.set_ylabel("events / year")
ax1.set_ylim(0, per_year["n_events"].max() + 1)
ax1.set_title("Number of dunkelflaute events per year", fontsize=13, loc="left")

sizes = 40 + 90 * (events_df["duration_days"] - 1)
ax2.scatter(events_df["start"].dt.year + (events_df["start"].dt.dayofyear / 366), events_df["intensity"],
            s=sizes, color="#EB811B", edgecolor="#23373B", linewidth=0.8, alpha=0.85, zorder=3)
for _, row in events_df.iterrows():
    if row["duration_days"] >= 2:
        ax2.annotate(row["start"].strftime("%b %-d"), (row["start"].year + row["start"].dayofyear / 366, row["intensity"]),
                      xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8, color="#23373B")

ax2.set_ylabel("severity (intensity score)")
ax2.set_xlabel("year")
ax2.set_title("Severity of each event (marker size = duration in days)", fontsize=13, loc="left")
ax2.set_xticks(years_ev)
ax2.set_xticklabels([str(y) for y in years_ev], rotation=45, ha="right")
ax2.set_ylim(bottom=0)

for ax in (ax1, ax2):
    for y in years_ev:
        ax.axvline(y, color="#EEEEEE", linewidth=0.6, zorder=0)

fig.suptitle("Dunkelflaute events, 2000-2025: joint low-wind-AND-low-solar days, real ERA5 data", fontsize=14, y=0.99)
fig.text(0.5, 0.005,
          "A day qualifies if daylight-mean wind AND daylight-mean solar are BOTH below their own 2000-2024 bottom-decile "
          "threshold; consecutive qualifying\ndays are merged into one event. Severity (intensity) = mean fractional "
          "shortfall of daily-mean wind/solar below threshold, averaged over the event.",
          ha="center", fontsize=8.5, color="#888888")
fig.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.savefig("dunkelflaute_events_per_year.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)



In [ ]:
# 25. Temporal (lag-1) and spatial (north-south) Kendall's tau of the seasonal AR residuals of all three variables

from scipy.stats import kendalltau


def lag1_tau(residuals_seasonal, col, season):
    sub = residuals_seasonal.loc[residuals_seasonal["season"] == season, col].dropna().sort_index()
    idx = sub.index
    elapsed = (idx[1:] - idx[:-1]).total_seconds() / 3600
    valid = (np.asarray(elapsed) == 1.0)
    x_prev = sub.to_numpy()[:-1][valid]
    x_curr = sub.to_numpy()[1:][valid]
    tau, _ = kendalltau(x_prev, x_curr)
    return tau, int(valid.sum())


print("Part A: temporal (lag-1) tau, all three variables")
temporal_tau_all = {col: {} for col in variables}
for col in variables:
    for season in SEASONS:
        tau, n = lag1_tau(residuals_seasonal_ar, col, season)
        temporal_tau_all[col][season] = tau
    print(f"  {col:<6}", {s: round(v, 3) for s, v in temporal_tau_all[col].items()})

print("\nPart B1: spatial tau, wind (reusing Section 18's fitted residuals)")
spatial_tau_all = {"wind": {}}
for season in SEASONS:
    sub = residuals_seasonal_ns.loc[residuals_seasonal_ns["season"] == season]
    tau, _ = kendalltau(sub["North_wind"], sub["South_wind"])
    spatial_tau_all["wind"][season] = tau
print(f"  wind  ", {s: round(v, 3) for s, v in spatial_tau_all["wind"].items()})


def load_ns_variable(var_code, var_name, mbc_or_noc, out_name):
    records = {f"{region}_{out_name}": [] for region in SUBREGIONS}
    for year in train_years:
        src_dir = base_dir if year >= PRE2014_CUTOFF else pre2014_dir
        suffix = "" if year >= PRE2014_CUTOFF else pre2014_suffix
        for region_name, region in SUBREGIONS.items():
            ts = load_year_subregion(var_code, var_name, year, mbc_or_noc, region, src_dir, filename_suffix=suffix)
            records[f"{region_name}_{out_name}"].append(pd.Series(ts.values, index=pd.to_datetime(ts["time"].values)))
        print(f"  Loading {out_name} {year}")
    df = pd.DataFrame({k: pd.concat(v).sort_index() for k, v in records.items()})
    return df[~df.index.duplicated(keep="first")].sort_index()


def process_ns_variable(df_raw, cols, is_solar=False):
    df_proc = df_raw.copy()
    if is_solar:
        for c in cols:
            df_proc[c] = np.log(df_proc[c] + SOLAR_LOG_EPS)
    trend_coefs = fit_linear_trend_ns(df_proc, cols, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
    df_detrended = df_proc[cols] - trend_value_ns(df_proc.index, trend_coefs, cols)
    climatology = build_smoothed_climatology_ns(df_detrended, cols, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
    train_mean = map_climatology_ns(df_proc.index, climatology, cols) + trend_value_ns(df_proc.index, trend_coefs, cols)
    df_anom = df_proc[cols] - train_mean

    residuals_by_season = {}
    for season in SEASONS:
        block_id = season_block_ids(df_anom.index, season)
        season_resid = pd.DataFrame(index=df_anom.index)
        for col in cols:
            fit = fit_ar_model_seasonal_one(y=df_anom[col], block_id=block_id, col=col, maxlag=AR_MAX_LAG)
            season_resid[col] = fit["resid"]
        residuals_by_season[season] = season_resid.dropna()
    return pd.concat([d.assign(season=s) for s, d in residuals_by_season.items()]).sort_index()


print("\nPart B2: spatial tau, temperature (loading North/South regions)")
df_ns_temp = load_ns_variable("TA-_0002m", "t2m", "noc", "temp")
df_ns_temp.to_pickle("./data/ns_temp_2000-2024.pkl")
residuals_seasonal_ns_temp = process_ns_variable(df_ns_temp, ["North_temp", "South_temp"], is_solar=False)
for season in SEASONS:
    sub = residuals_seasonal_ns_temp.loc[residuals_seasonal_ns_temp["season"] == season]
    tau, _ = kendalltau(sub["North_temp"], sub["South_temp"])
    spatial_tau_all.setdefault("temp", {})[season] = tau
print(f"  temp  ", {s: round(v, 3) for s, v in spatial_tau_all["temp"].items()})

print("\nPart B3: spatial tau, solar (loading North/South regions, log-space)")
df_ns_solar_raw = load_ns_variable("GHI_0000m", "ssrd", "noc", "solar")
df_ns_solar_raw.to_pickle("./data/ns_solar_raw_2000-2024.pkl")
residuals_seasonal_ns_solar = process_ns_variable(df_ns_solar_raw, ["North_solar", "South_solar"], is_solar=True)
for season in SEASONS:
    sub = residuals_seasonal_ns_solar.loc[residuals_seasonal_ns_solar["season"] == season]
    tau, _ = kendalltau(sub["North_solar"], sub["South_solar"])
    spatial_tau_all.setdefault("solar", {})[season] = tau
print(f"  solar ", {s: round(v, 3) for s, v in spatial_tau_all["solar"].items()})

print("\nSummary: mean |tau| across seasons, by variable and axis")
summary_rows = []
for col in variables:
    t_vals = list(temporal_tau_all[col].values())
    s_vals = list(spatial_tau_all[col].values())
    summary_rows.append({
        "variable": col, "temporal_mean": np.mean(t_vals), "temporal_range": (min(t_vals), max(t_vals)),
        "spatial_mean": np.mean(s_vals), "spatial_range": (min(s_vals), max(s_vals)),
    })
summary_df_all = pd.DataFrame(summary_rows)
print(summary_df_all.round(4).to_string(index=False))


In [ ]:
# 25. (continued) table and figure

variables_order = ["wind", "temp", "solar"]
labels_order = ["Wind", "Temperature", "Solar"]

fig, ax = plt.subplots(figsize=(9.5, 5.2))
x = np.arange(len(variables_order))
w = 0.32

for i, col in enumerate(variables_order):
    t_vals = np.array(list(temporal_tau_all[col].values()))
    s_vals = np.array(list(spatial_tau_all[col].values()))
    ax.bar(i - w / 2, t_vals.mean(), width=w, color="#EB811B", label="Temporal (lag 1h)" if i == 0 else None)
    ax.plot([i - w / 2, i - w / 2], [t_vals.min(), t_vals.max()], color="#23373B", lw=1.4)
    ax.bar(i + w / 2, s_vals.mean(), width=w, color="#3B6E8F", label="Spatial (North-South)" if i == 0 else None)
    ax.plot([i + w / 2, i + w / 2], [s_vals.min(), s_vals.max()], color="#23373B", lw=1.4)

ax.axhline(0.08, color="#999999", linestyle="--", linewidth=1.2, zorder=0)
ax.text(2.65, 0.088, "cross-variable ceiling ($\\leq$0.08)", color="#777777", fontsize=11, ha="right")
ax.axhline(0, color="#CCCCCC", linewidth=0.8, zorder=0)
ax.set_xticks(x)
ax.set_xticklabels(labels_order)
ax.set_ylabel(r"$|\tau|$ (Kendall's rank correlation)")
ax.legend(frameon=False, loc="upper left", fontsize=13)
fig.tight_layout()
plt.savefig("all_dependence.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)



In [ ]:
# 26. Temperature in two regions: Student-t copula between the north and south AR residuals per season, its
#     strength calibrated by bisection to the observed Kendall's tau (calibrate_strength); test year 2025 cached

NS_TEMP_TEST_CACHE = "./data/ns_temp_test2025.pkl"
NS_TEMP_COLS = ["North_temp", "South_temp"]

if os.path.exists(NS_TEMP_TEST_CACHE):
    df_test_ns_temp = pd.read_pickle(NS_TEMP_TEST_CACHE)
    df_test_ns_temp.columns = NS_TEMP_COLS
else:
    print(f"Loading test year {test_year} North/South temp...")
    records = {c: [] for c in NS_TEMP_COLS}
    for region_name, region in SUBREGIONS.items():
        ts = load_year_subregion("TA-_0002m", "t2m", test_year, "noc", region, base_dir, filename_suffix="")
        records[f"{region_name}_temp"].append(pd.Series(ts.values, index=pd.to_datetime(ts["time"].values)))
    df_test_ns_temp = pd.DataFrame({k: pd.concat(v).sort_index() for k, v in records.items()})
    df_test_ns_temp = df_test_ns_temp[~df_test_ns_temp.index.duplicated(keep="first")].sort_index()
    df_test_ns_temp.to_pickle(NS_TEMP_TEST_CACHE)

df_train_ns_temp = df_ns_temp.copy()
print(f"Train: {df_train_ns_temp.shape}  Test: {df_test_ns_temp.shape}")

print("\nFitting linear trend + climatology (temp, North/South)...")
trend_coefs_ns_temp = fit_linear_trend_ns(df_train_ns_temp, NS_TEMP_COLS, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
df_train_detrended_ns_temp = df_train_ns_temp[NS_TEMP_COLS] - trend_value_ns(df_train_ns_temp.index, trend_coefs_ns_temp, NS_TEMP_COLS)
climatology_ns_temp = build_smoothed_climatology_ns(df_train_detrended_ns_temp, NS_TEMP_COLS, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
train_mean_ns_temp = map_climatology_ns(df_train_ns_temp.index, climatology_ns_temp, NS_TEMP_COLS) + trend_value_ns(df_train_ns_temp.index, trend_coefs_ns_temp, NS_TEMP_COLS)
df_anom_ns_temp = df_train_ns_temp[NS_TEMP_COLS] - train_mean_ns_temp

year_baseline_offset_ns_temp = df_anom_ns_temp.groupby(df_anom_ns_temp.index.year).mean()
sim_calendar_mean_ns_temp = map_climatology_ns(df_test_ns_temp.index, climatology_ns_temp, NS_TEMP_COLS)

print("\nFitting seasonal AR models for North/South temp...")
models_seasonal_ns_temp = {season: {} for season in SEASONS}
residuals_by_season_ns_temp_fit = {}
for season in SEASONS:
    block_id = season_block_ids(df_anom_ns_temp.index, season)
    season_resid = pd.DataFrame(index=df_anom_ns_temp.index)
    for col in NS_TEMP_COLS:
        fit = fit_ar_model_seasonal_one(y=df_anom_ns_temp[col], block_id=block_id, col=col, maxlag=AR_MAX_LAG)
        models_seasonal_ns_temp[season][col] = fit
        season_resid[col] = fit["resid"]
        print(f"  {season} {col}: AR order {len(fit['lags'])}")
    residuals_by_season_ns_temp_fit[season] = season_resid.dropna()

residuals_seasonal_ns_temp_fit = pd.concat(
    [df.assign(season=season) for season, df in residuals_by_season_ns_temp_fit.items()]
).sort_index()


def calibrate_strength(bicop, target_tau, s_lo=0.02, s_hi=8.0, tol=2e-4, max_iter=60):
    """Automatic version of Section 18's manual SPATIAL_CALIBRATED_STRENGTH
    tuning: bisects the scalar multiplier on the fitted parameter vector
    until the copula's own analytic .tau matches target_tau."""
    base_params = np.asarray(bicop.parameters).astype(float).copy()
    lb = np.asarray(bicop.parameters_lower_bounds).reshape(base_params.shape)
    ub = np.asarray(bicop.parameters_upper_bounds).reshape(base_params.shape)

    def tau_at(s):
        params = np.clip(base_params * s, lb, ub)
        return pv.Bicop(family=bicop.family, rotation=bicop.rotation, parameters=params).tau

    lo, hi = s_lo, s_hi
    tau_lo, tau_hi = tau_at(lo), tau_at(hi)
    tries = 0
    while (tau_hi - target_tau) * (tau_lo - target_tau) > 0 and hi < 1e4 and tries < 20:
        hi *= 1.6
        tau_hi = tau_at(hi)
        tries += 1
    mid, tau_mid = hi, tau_hi
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        tau_mid = tau_at(mid)
        if abs(tau_mid - target_tau) < tol:
            break
        if (tau_mid - target_tau) * (tau_lo - target_tau) <= 0:
            hi, tau_hi = mid, tau_mid
        else:
            lo, tau_lo = mid, tau_mid
    params = np.clip(base_params * mid, lb, ub)
    calibrated = pv.Bicop(family=bicop.family, rotation=bicop.rotation, parameters=params)
    return calibrated, float(mid), float(calibrated.tau)


print("\nFitting + calibrating cross-region copula per season (temp)...")
spatial_copulas_temp = {}
calib_diag_temp = {}
for season in SEASONS:
    sub = residuals_seasonal_ns_temp_fit.loc[residuals_seasonal_ns_temp_fit["season"] == season]
    target_tau = spatial_tau_all["temp"][season]
    n = len(sub)
    u_north = rankdata(sub["North_temp"], method="average") / (n + 1)
    u_south = rankdata(sub["South_temp"], method="average") / (n + 1)
    data = np.asfortranarray(np.column_stack([u_north, u_south]))
    controls = pv.FitControlsBicop(family_set=SPATIAL_CANDIDATE_FAMILIES, selection_criterion="bic",
                                    parametric_method="mle", allow_rotations=True)
    bicop = pv.Bicop()
    bicop.select(data=data, controls=controls)
    calibrated, strength, achieved_tau = calibrate_strength(bicop, target_tau)
    spatial_copulas_temp[season] = calibrated
    calib_diag_temp[season] = {"family": str(bicop.family), "base_tau": bicop.tau, "strength": strength,
                                "achieved_tau": achieved_tau, "target_tau": target_tau}
    print(f"  {season:<8} family={bicop.family}  base_tau={bicop.tau:+.3f}  "
          f"strength={strength:.3f}  achieved_tau={achieved_tau:+.3f}  target={target_tau:+.3f}")

pool_by_season_ns_temp = {
    season: residuals_seasonal_ns_temp_fit.loc[residuals_seasonal_ns_temp_fit["season"] == season, NS_TEMP_COLS]
    for season in SEASONS
}

In [ ]:
# 27. Ensemble of the two-region temperature model; rate of cold deficits with both regions in their bottom
#     decile against the national mean

seeds_ns_temp = [BASE_SEED + r for r in range(N_TRAJECTORIES)]
target_index_ns_temp = sim_calendar_mean_ns_temp.index
T_ns_temp = len(target_index_ns_temp)
sim_calendar_arr_ns_temp = {col: sim_calendar_mean_ns_temp[col].to_numpy() for col in NS_TEMP_COLS}
season_labels_target_ns_temp = np.array([get_season(m) for m in target_index_ns_temp.month])

target_mean_batch_ns_temp = {col: np.empty((N_TRAJECTORIES, T_ns_temp)) for col in NS_TEMP_COLS}
for j, seed in enumerate(seeds_ns_temp):
    rng = np.random.default_rng(seed + 1_000_000)
    year = int(rng.choice(year_baseline_offset_ns_temp.index.to_numpy()))
    offset = year_baseline_offset_ns_temp.loc[year]
    for col in NS_TEMP_COLS:
        target_mean_batch_ns_temp[col][j] = sim_calendar_arr_ns_temp[col] + offset[col]

print(f"Simulating {N_TRAJECTORIES} North/South temp trajectories with the calibrated spatial copula...")
VAR_SEED_SALT_TEMP = 50_000  # keeps temp's copula draws independent of wind's (Section 19) and solar's (Section 29)
resid_sim_batch_ns_temp = {col: np.empty((N_TRAJECTORIES, T_ns_temp)) for col in NS_TEMP_COLS}
for j, seed in enumerate(seeds_ns_temp):
    for season in SEASONS:
        mask = season_labels_target_ns_temp == season
        n_season = int(mask.sum())
        bicop = spatial_copulas_temp[season]
        u_pair = np.asarray(bicop.simulate(n=n_season, seeds=[int(seed) + VAR_SEED_SALT_TEMP + season_seed_offsets[season]]))
        pool = pool_by_season_ns_temp[season]
        n_pool = len(pool)
        north_sorted = np.sort(pool["North_temp"].to_numpy())
        south_sorted = np.sort(pool["South_temp"].to_numpy())
        north_vals = north_sorted[np.clip((u_pair[:, 0] * n_pool).astype(int), 0, n_pool - 1)]
        south_vals = south_sorted[np.clip((u_pair[:, 1] * n_pool).astype(int), 0, n_pool - 1)]
        resid_sim_batch_ns_temp["North_temp"][j, mask] = north_vals
        resid_sim_batch_ns_temp["South_temp"][j, mask] = south_vals

phys_batch_ns_temp = reconstruct_ns_batch(resid_sim_batch_ns_temp, models_seasonal_ns_temp, target_mean_batch_ns_temp,
                                           target_index_ns_temp, df_anom_ns_temp, NS_TEMP_COLS, get_season)
ns_trajectories_temp = {j: pd.DataFrame({col: phys_batch_ns_temp[col][j] for col in NS_TEMP_COLS}, index=target_index_ns_temp)
                         for j in range(N_TRAJECTORIES)}
pooled_ns_temp = pd.concat(ns_trajectories_temp.values(), axis=0)
print("Done.")

north_thresh_temp = df_train_ns_temp["North_temp"].quantile(DUNKELFLAUTE_QUANTILE)
south_thresh_temp = df_train_ns_temp["South_temp"].quantile(DUNKELFLAUTE_QUANTILE)
avg_thresh_ns_temp = ((df_train_ns_temp["North_temp"] + df_train_ns_temp["South_temp"]) / 2).quantile(DUNKELFLAUTE_QUANTILE)

pooled_ns_temp["avg_temp"] = (pooled_ns_temp["North_temp"] + pooled_ns_temp["South_temp"]) / 2
sim_agg_low_temp = (pooled_ns_temp["avg_temp"] <= avg_thresh_ns_temp).mean()
sim_both_low_temp = ((pooled_ns_temp["North_temp"] <= north_thresh_temp) & (pooled_ns_temp["South_temp"] <= south_thresh_temp)).mean()

real_avg_ns_temp = (df_train_ns_temp["North_temp"] + df_train_ns_temp["South_temp"]) / 2
real_agg_low_temp = (real_avg_ns_temp <= avg_thresh_ns_temp).mean()
real_both_low_temp = ((df_train_ns_temp["North_temp"] <= north_thresh_temp) & (df_train_ns_temp["South_temp"] <= south_thresh_temp)).mean()

print("\n" + "=" * 70)
print(f"Aggregate-average vs both-region-low temp (cold snap) rate (N={N_TRAJECTORIES})")
print("=" * 70)
print(f"{'':32}{'aggregate-avg':>16}{'both regions low':>20}")
print(f"{'REAL (' + str(train_years.start) + '-' + str(train_years.stop - 1) + ')':32}{real_agg_low_temp:>16.1%}{real_both_low_temp:>20.1%}")
print(f"{'SIMULATED (N=' + str(N_TRAJECTORIES) + ')':32}{sim_agg_low_temp:>16.1%}{sim_both_low_temp:>20.1%}")
print(f"\nReal ratio (both/aggregate)      = {real_both_low_temp/real_agg_low_temp:.2f}x")
print(f"Simulated ratio (both/aggregate) = {sim_both_low_temp/sim_agg_low_temp:.2f}x")
print("\nCalibration hit the target tau almost exactly in every season (Section 26), but the")
print("both-region joint-low rate still undershoots real - same known shape as wind's own gap")
print("(Section 19-20): a symmetric copula calibrated to the right tau doesn't automatically get")
print("the right LOWER-TAIL dependence, which is what this joint-deficit metric actually measures.")

In [ ]:
# 28. Irradiance in two regions (log space): Student-t copula between the north and south AR residuals per
#     season, calibrated to the observed Kendall's tau

NS_SOLAR_TEST_CACHE = "./data/ns_solar_test2025.pkl"
NS_SOLAR_COLS = ["North_solar", "South_solar"]

if os.path.exists(NS_SOLAR_TEST_CACHE):
    df_test_ns_solar_raw = pd.read_pickle(NS_SOLAR_TEST_CACHE)
    df_test_ns_solar_raw.columns = NS_SOLAR_COLS
else:
    print(f"Loading test year {test_year} North/South solar...")
    records = {c: [] for c in NS_SOLAR_COLS}
    for region_name, region in SUBREGIONS.items():
        ts = load_year_subregion("GHI_0000m", "ssrd", test_year, "noc", region, base_dir, filename_suffix="")
        records[f"{region_name}_solar"].append(pd.Series(ts.values, index=pd.to_datetime(ts["time"].values)))
    df_test_ns_solar_raw = pd.DataFrame({k: pd.concat(v).sort_index() for k, v in records.items()})
    df_test_ns_solar_raw = df_test_ns_solar_raw[~df_test_ns_solar_raw.index.duplicated(keep="first")].sort_index()
    df_test_ns_solar_raw.to_pickle(NS_SOLAR_TEST_CACHE)

df_train_ns_solar_raw = df_ns_solar_raw.copy()
print(f"Train: {df_train_ns_solar_raw.shape}  Test: {df_test_ns_solar_raw.shape}")

df_train_ns_solar = df_train_ns_solar_raw.copy()
df_test_ns_solar = df_test_ns_solar_raw.copy()
for c in NS_SOLAR_COLS:
    df_train_ns_solar[c] = np.log(df_train_ns_solar[c] + SOLAR_LOG_EPS)
    df_test_ns_solar[c] = np.log(df_test_ns_solar[c] + SOLAR_LOG_EPS)

print("\nFitting linear trend + climatology (solar, North/South, log-space)...")
trend_coefs_ns_solar = fit_linear_trend_ns(df_train_ns_solar, NS_SOLAR_COLS, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
df_train_detrended_ns_solar = df_train_ns_solar[NS_SOLAR_COLS] - trend_value_ns(df_train_ns_solar.index, trend_coefs_ns_solar, NS_SOLAR_COLS)
climatology_ns_solar = build_smoothed_climatology_ns(df_train_detrended_ns_solar, NS_SOLAR_COLS, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
train_mean_ns_solar = map_climatology_ns(df_train_ns_solar.index, climatology_ns_solar, NS_SOLAR_COLS) + trend_value_ns(df_train_ns_solar.index, trend_coefs_ns_solar, NS_SOLAR_COLS)
df_anom_ns_solar = df_train_ns_solar[NS_SOLAR_COLS] - train_mean_ns_solar

year_baseline_offset_ns_solar = df_anom_ns_solar.groupby(df_anom_ns_solar.index.year).mean()
sim_calendar_mean_ns_solar = map_climatology_ns(df_test_ns_solar.index, climatology_ns_solar, NS_SOLAR_COLS)

print("\nFitting seasonal AR models for North/South solar (log-space)...")
models_seasonal_ns_solar = {season: {} for season in SEASONS}
residuals_by_season_ns_solar_fit = {}
for season in SEASONS:
    block_id = season_block_ids(df_anom_ns_solar.index, season)
    season_resid = pd.DataFrame(index=df_anom_ns_solar.index)
    for col in NS_SOLAR_COLS:
        fit = fit_ar_model_seasonal_one(y=df_anom_ns_solar[col], block_id=block_id, col=col, maxlag=AR_MAX_LAG)
        models_seasonal_ns_solar[season][col] = fit
        season_resid[col] = fit["resid"]
        print(f"  {season} {col}: AR order {len(fit['lags'])}")
    residuals_by_season_ns_solar_fit[season] = season_resid.dropna()

residuals_seasonal_ns_solar_fit = pd.concat(
    [df.assign(season=season) for season, df in residuals_by_season_ns_solar_fit.items()]
).sort_index()

print("\nFitting + calibrating cross-region copula per season (solar)...")
spatial_copulas_solar = {}
calib_diag_solar = {}
for season in SEASONS:
    sub = residuals_seasonal_ns_solar_fit.loc[residuals_seasonal_ns_solar_fit["season"] == season]
    target_tau = spatial_tau_all["solar"][season]
    n = len(sub)
    u_north = rankdata(sub["North_solar"], method="average") / (n + 1)
    u_south = rankdata(sub["South_solar"], method="average") / (n + 1)
    data = np.asfortranarray(np.column_stack([u_north, u_south]))
    controls = pv.FitControlsBicop(family_set=SPATIAL_CANDIDATE_FAMILIES, selection_criterion="bic",
                                    parametric_method="mle", allow_rotations=True)
    bicop = pv.Bicop()
    bicop.select(data=data, controls=controls)
    calibrated, strength, achieved_tau = calibrate_strength(bicop, target_tau)
    spatial_copulas_solar[season] = calibrated
    calib_diag_solar[season] = {"family": str(bicop.family), "base_tau": bicop.tau, "strength": strength,
                                 "achieved_tau": achieved_tau, "target_tau": target_tau}
    print(f"  {season:<8} family={bicop.family}  base_tau={bicop.tau:+.3f}  "
          f"strength={strength:.3f}  achieved_tau={achieved_tau:+.3f}  target={target_tau:+.3f}")

pool_by_season_ns_solar = {
    season: residuals_seasonal_ns_solar_fit.loc[residuals_seasonal_ns_solar_fit["season"] == season, NS_SOLAR_COLS]
    for season in SEASONS
}

In [ ]:
# 29. Ensemble of the two-region irradiance model; rate of joint low irradiance (threshold over the daylight
#     hours of each month), observed and simulated

seeds_ns_solar = [BASE_SEED + r for r in range(N_TRAJECTORIES)]
target_index_ns_solar = sim_calendar_mean_ns_solar.index
T_ns_solar = len(target_index_ns_solar)
sim_calendar_arr_ns_solar = {col: sim_calendar_mean_ns_solar[col].to_numpy() for col in NS_SOLAR_COLS}
season_labels_target_ns_solar = np.array([get_season(m) for m in target_index_ns_solar.month])

target_mean_batch_ns_solar = {col: np.empty((N_TRAJECTORIES, T_ns_solar)) for col in NS_SOLAR_COLS}
for j, seed in enumerate(seeds_ns_solar):
    rng = np.random.default_rng(seed + 1_000_000)
    year = int(rng.choice(year_baseline_offset_ns_solar.index.to_numpy()))
    offset = year_baseline_offset_ns_solar.loc[year]
    for col in NS_SOLAR_COLS:
        target_mean_batch_ns_solar[col][j] = sim_calendar_arr_ns_solar[col] + offset[col]

print(f"Simulating {N_TRAJECTORIES} North/South solar trajectories with the calibrated spatial copula...")
VAR_SEED_SALT_SOLAR = 90_000  # keeps solar's copula draws independent of wind's/temp's
resid_sim_batch_ns_solar = {col: np.empty((N_TRAJECTORIES, T_ns_solar)) for col in NS_SOLAR_COLS}
for j, seed in enumerate(seeds_ns_solar):
    for season in SEASONS:
        mask = season_labels_target_ns_solar == season
        n_season = int(mask.sum())
        bicop = spatial_copulas_solar[season]
        u_pair = np.asarray(bicop.simulate(n=n_season, seeds=[int(seed) + VAR_SEED_SALT_SOLAR + season_seed_offsets[season]]))
        pool = pool_by_season_ns_solar[season]
        n_pool = len(pool)
        north_sorted = np.sort(pool["North_solar"].to_numpy())
        south_sorted = np.sort(pool["South_solar"].to_numpy())
        north_vals = north_sorted[np.clip((u_pair[:, 0] * n_pool).astype(int), 0, n_pool - 1)]
        south_vals = south_sorted[np.clip((u_pair[:, 1] * n_pool).astype(int), 0, n_pool - 1)]
        resid_sim_batch_ns_solar["North_solar"][j, mask] = north_vals
        resid_sim_batch_ns_solar["South_solar"][j, mask] = south_vals


def reconstruct_ns_batch_log(resid_sim_batch, models_seasonal, target_mean_batch, target_index, train_anom, cols, get_season_fn):
    """Same recursion as Section 19/20's reconstruct_ns_batch, but exponentiates
    back out of log-space at the end (as in Section 5) instead of clipping a
    linear-space value directly - solar can't go negative either way, but the
    transform has to happen in the right order."""
    N, T = next(iter(resid_sim_batch.values())).shape
    season_labels = target_index.month.map(get_season_fn).to_numpy()
    out = {}
    for col in cols:
        train_history = train_anom[col].dropna().to_numpy(dtype=float)
        max_lag = max((max(models_seasonal[s][col]["lags"], default=0) for s in models_seasonal), default=0)
        step_const = np.empty(T, dtype=float)
        step_coefs = []
        for t in range(T):
            model = models_seasonal[season_labels[t]][col]
            params = model["params"]
            lags = model["lags"]
            step_const[t] = float(params["const"])
            step_coefs.append({lag: float(params[f"{col}.L{lag}"]) for lag in lags})
        sim = np.empty((N, max_lag + T), dtype=float)
        if max_lag > 0:
            pre = train_history[-max_lag:]
            sim[:, :max_lag] = pre[np.newaxis, :]
        for t in range(T):
            i = t + max_lag
            value = np.full(N, step_const[t], dtype=float)
            for lag, coef in step_coefs[t].items():
                value += coef * sim[:, i - lag]
            value += resid_sim_batch[col][:, t]
            sim[:, i] = value
        phys_log = sim[:, max_lag:] + target_mean_batch[col]
        out[col] = np.clip(np.exp(phys_log) - SOLAR_LOG_EPS, 0, None)
    return out


phys_batch_ns_solar = reconstruct_ns_batch_log(resid_sim_batch_ns_solar, models_seasonal_ns_solar, target_mean_batch_ns_solar,
                                                target_index_ns_solar, df_anom_ns_solar, NS_SOLAR_COLS, get_season)
ns_trajectories_solar = {j: pd.DataFrame({col: phys_batch_ns_solar[col][j] for col in NS_SOLAR_COLS}, index=target_index_ns_solar)
                          for j in range(N_TRAJECTORIES)}
pooled_ns_solar = pd.concat(ns_trajectories_solar.values(), axis=0)
print("Done.")

# Daylight mask, Section 12's exact convention, applied to the
# North/South regional average instead of the national aggregate
clim_month_hour_ns_solar = ((df_train_ns_solar_raw["North_solar"] + df_train_ns_solar_raw["South_solar"]) / 2).groupby(
    [df_train_ns_solar_raw.index.month, df_train_ns_solar_raw.index.hour]).mean()


def is_daylight_ns(index):
    keys = list(zip(index.month, index.hour))
    return np.array([clim_month_hour_ns_solar.get(k, 0.0) > DAYLIGHT_FLOOR for k in keys])


train_day_mask_ns_solar = is_daylight_ns(df_train_ns_solar_raw.index)
sim_day_mask_ns_solar = is_daylight_ns(pooled_ns_solar.index)

real_north_solar = df_train_ns_solar_raw.loc[train_day_mask_ns_solar, "North_solar"]
real_south_solar = df_train_ns_solar_raw.loc[train_day_mask_ns_solar, "South_solar"]
sim_pooled_day_solar = pooled_ns_solar.loc[sim_day_mask_ns_solar]

north_thresh_solar = real_north_solar.quantile(DUNKELFLAUTE_QUANTILE)
south_thresh_solar = real_south_solar.quantile(DUNKELFLAUTE_QUANTILE)
real_avg_solar = (real_north_solar + real_south_solar) / 2
avg_thresh_ns_solar = real_avg_solar.quantile(DUNKELFLAUTE_QUANTILE)

real_agg_low_solar = (real_avg_solar <= avg_thresh_ns_solar).mean()
real_both_low_solar = ((real_north_solar <= north_thresh_solar) & (real_south_solar <= south_thresh_solar)).mean()

sim_avg_solar = (sim_pooled_day_solar["North_solar"] + sim_pooled_day_solar["South_solar"]) / 2
sim_agg_low_solar = (sim_avg_solar <= avg_thresh_ns_solar).mean()
sim_both_low_solar = ((sim_pooled_day_solar["North_solar"] <= north_thresh_solar) & (sim_pooled_day_solar["South_solar"] <= south_thresh_solar)).mean()

print("\n" + "=" * 70)
print(f"Aggregate-average vs both-region-low solar rate, DAYLIGHT HOURS ONLY (N={N_TRAJECTORIES})")
print("=" * 70)
print(f"Daylight hours: {train_day_mask_ns_solar.mean():.1%} of all train hours")
print(f"{'':32}{'aggregate-avg':>16}{'both regions low':>20}")
print(f"{'REAL (' + str(train_years.start) + '-' + str(train_years.stop - 1) + ')':32}{real_agg_low_solar:>16.1%}{real_both_low_solar:>20.1%}")
print(f"{'SIMULATED (N=' + str(N_TRAJECTORIES) + ')':32}{sim_agg_low_solar:>16.1%}{sim_both_low_solar:>20.1%}")
print(f"\nReal ratio (both/aggregate)      = {real_both_low_solar/real_agg_low_solar:.2f}x")
print(f"Simulated ratio (both/aggregate) = {sim_both_low_solar/sim_agg_low_solar:.2f}x")

In [ ]:
# 30. Two-region copulas of the three variables side by side: observed and simulated both-region deficit rates
#     against the national mean

spatial_summary = pd.DataFrame([
    {"variable": "Wind",        "real_ratio": real_both_low / real_agg_low, "sim_ratio": sim_both_low / sim_agg_low},
    {"variable": "Temperature", "real_ratio": real_both_low_temp / real_agg_low_temp, "sim_ratio": sim_both_low_temp / sim_agg_low_temp},
    {"variable": "Solar",       "real_ratio": real_both_low_solar / real_agg_low_solar, "sim_ratio": sim_both_low_solar / sim_agg_low_solar},
])
print(spatial_summary.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 5.0))
x = np.arange(len(spatial_summary))
w = 0.32
ax.bar(x - w / 2, spatial_summary["real_ratio"], width=w, color="#9AA5AB", label="Real (both-region / aggregate)")
ax.bar(x + w / 2, spatial_summary["sim_ratio"], width=w, color="#00549F", label="Simulated (calibrated spatial copula)")
ax.set_xticks(x)
ax.set_xticklabels(spatial_summary["variable"])
ax.set_ylabel("both-region-low rate / aggregate-low rate")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.savefig("spatial_copula_all3_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 31. Model AR + Spatial: north and south reconstructed with the two-region copulas, averaged to national values
#     and checked with the coverage of Section 11 (the latitude row 51 N belongs to both regions)

def national_daily_from_ns(phys_batch, cols, target_index):
    national_hourly = (phys_batch[cols[0]] + phys_batch[cols[1]]) / 2
    df_nat = pd.DataFrame(national_hourly.T, index=target_index)
    daily = df_nat.resample("D").mean()
    return daily.to_numpy().T, daily.index


spatial_daily = {}
spatial_daily["wind"], day_index_spatial = national_daily_from_ns(phys_batch_ns, NS_COLS, target_index_ns)
spatial_daily["temp"], _ = national_daily_from_ns(phys_batch_ns_temp, NS_TEMP_COLS, target_index_ns_temp)
spatial_daily["solar"], _ = national_daily_from_ns(phys_batch_ns_solar, NS_SOLAR_COLS, target_index_ns_solar)

assert (day_index_spatial.dayofyear == day_index.dayofyear).all(), "calendar mismatch vs Section 11's day_index"
day_of_year_spatial = day_index_spatial.dayofyear

spatial_coverage_rows = []
for year_label, year_df in extreme_years_to_plot.items():
    year_daily = year_df.resample("D").mean()
    year_doy = year_daily.index.dayofyear
    for col in variables:
        stack = spatial_daily[col]
        lo = np.percentile(stack, 5, axis=0)
        hi = np.percentile(stack, 95, axis=0)
        lo_at_year = np.interp(year_doy, day_of_year_spatial, lo)
        hi_at_year = np.interp(year_doy, day_of_year_spatial, hi)
        real_vals = year_daily[col].to_numpy()
        inside = (real_vals >= lo_at_year) & (real_vals <= hi_at_year)
        spatial_coverage_rows.append({"extreme_year": year_label, "variable": col, "model": "AR+Spatial",
                                       "coverage": inside.mean()})

spatial_coverage_df = pd.DataFrame(spatial_coverage_rows)
combined_coverage_df = pd.concat([coverage_df, spatial_coverage_df], ignore_index=True)

print("All four arms, coverage of each extreme year (AR / VAR / AR+Copula from Section 11):")
for year_label in extreme_years_to_plot:
    print(f"\n{year_label}:")
    print(combined_coverage_df[combined_coverage_df.extreme_year == year_label]
          .pivot(index="variable", columns="model", values="coverage").round(3))

print("\nAR+Spatial vs plain AR (same national region, no spatial structure) - is spatial reconstruction")
print("even an improvement on THIS metric, independent of AR+Copula's temporal-copula effect?")
delta_rows = []
for col in variables:
    ar_vals = coverage_df[(coverage_df.variable == col) & (coverage_df.model == "AR")].set_index("extreme_year")["coverage"]
    sp_vals = spatial_coverage_df[spatial_coverage_df.variable == col].set_index("extreme_year")["coverage"]
    delta_rows.append({"variable": col, "mean_AR": ar_vals.mean(), "mean_AR+Spatial": sp_vals.mean(),
                        "delta_pp": (sp_vals.mean() - ar_vals.mean()) * 100})
print(pd.DataFrame(delta_rows).round(3).to_string(index=False))

# add the daily means of this model to the cache of Section 11
existing_cache = dict(np.load(daily_mean_cache_path))
existing_cache.update({f"spatial_{v}": spatial_daily[v] for v in variables})
np.savez(daily_mean_cache_path, **existing_cache)
print(f"\nExtended {daily_mean_cache_path} with spatial_wind/spatial_temp/spatial_solar.")

In [ ]:
# 32. Further compound scenarios in the observed record (bottom / top decile thresholds, national mean =
#     (north + south) / 2): cold Dunkelflaute, post-sunset heat peak, storm cut-out without sun, joint 3-h
#     ramp-down, oversupply

Q_SCENARIO = 0.10

df_national_new = pd.DataFrame({
    "wind": (df_ns_wind.loc[df_ns_wind.index.year <= 2024, "North_wind"] + df_ns_wind.loc[df_ns_wind.index.year <= 2024, "South_wind"]) / 2,
    "temp": (df_ns_temp["North_temp"] + df_ns_temp["South_temp"]) / 2,
    "solar": (df_ns_solar_raw["North_solar"] + df_ns_solar_raw["South_solar"]) / 2,
}).dropna()
print(f"National aggregate (derived from North/South): {df_national_new.shape}, "
      f"{df_national_new.index.min()} to {df_national_new.index.max()}")

clim_month_hour_new = df_national_new.groupby([df_national_new.index.month, df_national_new.index.hour])["solar"].mean()


def is_daylight_new(index):
    keys = list(zip(index.month, index.hour))
    return np.array([clim_month_hour_new.get(k, 0.0) > DAYLIGHT_FLOOR for k in keys])


daylight_mask_new = is_daylight_new(df_national_new.index)
df_day_new = df_national_new.loc[daylight_mask_new]
df_night_new = df_national_new.loc[~daylight_mask_new]

wind_lo_new = df_day_new["wind"].quantile(Q_SCENARIO)
solar_lo_new = df_day_new["solar"].quantile(Q_SCENARIO)
temp_lo_day_new = df_day_new["temp"].quantile(Q_SCENARIO)
temp_hi_night_new = df_night_new["temp"].quantile(1 - Q_SCENARIO)
wind_hi_new = df_day_new["wind"].quantile(1 - Q_SCENARIO)
solar_hi_new = df_day_new["solar"].quantile(1 - Q_SCENARIO)
wind_lo_calm_new = df_night_new["wind"].quantile(Q_SCENARIO)

scenario_results = []

# 1. Cold Dunkelflaute: low wind + low solar + low temp, daylight
cold_dunkel = (df_day_new["wind"] <= wind_lo_new) & (df_day_new["solar"] <= solar_lo_new) & (df_day_new["temp"] <= temp_lo_day_new)
rate1 = cold_dunkel.mean()
scenario_results.append({"scenario": "Cold Dunkelflaute", "rate": rate1, "vs_independence": rate1 / (Q_SCENARIO ** 3)})

plain_dunkel_new = (df_day_new["wind"] <= wind_lo_new) & (df_day_new["solar"] <= solar_lo_new)
frac_cold_given_dunkel_new = (plain_dunkel_new & (df_day_new["temp"] <= temp_lo_day_new)).sum() / plain_dunkel_new.sum()
print(f"Of plain wind+solar dunkelflaute hours, {frac_cold_given_dunkel_new:.1%} are ALSO cold "
      f"(bottom decile temp) - vs {Q_SCENARIO:.0%} expected under independence")

# 2. Post-Sunset Heatwave Peak: hot + calm + night
heatwave_peak = (df_night_new["temp"] >= temp_hi_night_new) & (df_night_new["wind"] <= wind_lo_calm_new)
rate2 = heatwave_peak.mean()
scenario_results.append({"scenario": "Post-Sunset Heatwave Peak", "rate": rate2, "vs_independence": rate2 / (Q_SCENARIO ** 2)})

# 3. Storm Cut-out + No Sun (10m proxy for a 25 m/s hub-height cutout)
HUB_SCALING_NEW = (100 / 10) ** 0.14
cutout_10m_new = 25.0 / HUB_SCALING_NEW
storm_cutout = (df_national_new["wind"] >= cutout_10m_new) & (df_national_new["solar"] <= solar_lo_new)
rate3 = storm_cutout.mean()
p_wind_extreme_new = (df_national_new["wind"] >= cutout_10m_new).mean()
scenario_results.append({"scenario": f"Storm Cut-out + No Sun (wind>={cutout_10m_new:.1f} m/s, 10m proxy)",
                          "rate": rate3, "vs_independence": float("nan")})
print(f"P(national-mean wind >= {cutout_10m_new:.1f} m/s) = {p_wind_extreme_new:.4%} "
      f"({int((df_national_new['wind'] >= cutout_10m_new).sum())} hours in 25 years) - "
      f"structurally invisible to a national spatial average (see Section 33)")

# 4. Joint 3h Ramp-Down: wind and solar both crash together
d3_new = pd.DataFrame({
    "wind_chg": df_national_new["wind"].diff(3),
    "solar_chg": df_day_new["solar"].reindex(df_national_new.index).diff(3),
}).dropna()
wind_chg_lo_new = d3_new["wind_chg"].quantile(Q_SCENARIO)
solar_chg_lo_new = d3_new["solar_chg"].quantile(Q_SCENARIO)
ramp = (d3_new["wind_chg"] <= wind_chg_lo_new) & (d3_new["solar_chg"] <= solar_chg_lo_new)
rate4 = ramp.mean()
scenario_results.append({"scenario": "Joint 3h Ramp-Down", "rate": rate4, "vs_independence": rate4 / (Q_SCENARIO ** 2)})

# 5. Oversupply proxy: wind AND solar both high, daylight (NO demand data - incomplete)
oversupply = (df_day_new["wind"] >= wind_hi_new) & (df_day_new["solar"] >= solar_hi_new)
rate5 = oversupply.mean()
scenario_results.append({"scenario": "Oversupply proxy (no demand data)", "rate": rate5, "vs_independence": rate5 / (Q_SCENARIO ** 2)})

scenario_results_df = pd.DataFrame(scenario_results)
print("\n" + "=" * 90)
print(f"{'Scenario':<45}{'Rate':>10}{'vs. independence':>18}")
print("=" * 90)
for _, r in scenario_results_df.iterrows():
    vs_indep_str = f"{r['vs_independence']:.2f}x" if pd.notna(r["vs_independence"]) else "n/a"
    print(f"{r['scenario']:<45}{r['rate']:>9.3%}{vs_indep_str:>18}")

In [ ]:
# 33. Regional granularity of wind (1, 2 and 4 regions): given the national mean in its bottom decile, the
#     probability that all or any sub-regions are too

QUADRANTS = {
    "NW": {"lat_min": 51, "lat_max": 55, "lon_min": 5,    "lon_max": 10.5},
    "NE": {"lat_min": 51, "lat_max": 55, "lon_min": 10.5, "lon_max": 16},
    "SW": {"lat_min": 47, "lat_max": 51, "lon_min": 5,    "lon_max": 10.5},
    "SE": {"lat_min": 47, "lat_max": 51, "lon_min": 10.5, "lon_max": 16},
}
QUAD_CACHE = "./data/ns_wind_quadrants_2000-2024.pkl"

if os.path.exists(QUAD_CACHE):
    print(f"Loading quadrant-level wind from cache: {QUAD_CACHE}")
    df_quad = pd.read_pickle(QUAD_CACHE)
else:
    print("Loading quadrant-level wind from raw grids (2000-2024)...")
    records = {f"{q}_wind": [] for q in QUADRANTS}
    for year in train_years:
        src_dir = base_dir if year >= PRE2014_CUTOFF else pre2014_dir
        suffix = "" if year >= PRE2014_CUTOFF else pre2014_suffix
        for qname, region in QUADRANTS.items():
            ts = load_year_subregion("WS-_0010m", "ws10", year, "mbc", region, src_dir, filename_suffix=suffix)
            records[f"{qname}_wind"].append(pd.Series(ts.values, index=pd.to_datetime(ts["time"].values)))
        print(f"  loaded {year}")
    df_quad = pd.DataFrame({k: pd.concat(v).sort_index() for k, v in records.items()})
    df_quad = df_quad[~df_quad.index.duplicated(keep="first")].sort_index()
    df_quad.to_pickle(QUAD_CACHE)

print(f"Quadrant wind shape: {df_quad.shape}")

df_ns_wind_2024 = df_ns_wind.loc[df_ns_wind.index.year <= 2024]
common_idx_g = df_quad.index.intersection(df_ns_wind_2024.index)
df_quad = df_quad.loc[common_idx_g]
df_ns_wind_2024 = df_ns_wind_2024.loc[common_idx_g]

national_g = (df_quad["NW_wind"] + df_quad["NE_wind"] + df_quad["SW_wind"] + df_quad["SE_wind"]) / 4
national_thresh_g = national_g.quantile(DUNKELFLAUTE_QUANTILE)
national_low_g = national_g <= national_thresh_g

granularity_results = [{"granularity": "1 region (national)", "n_regions": 1, "p_all_low": 1.0, "p_any_low": 1.0}]

north_thresh_g = df_ns_wind_2024["North_wind"].quantile(DUNKELFLAUTE_QUANTILE)
south_thresh_g = df_ns_wind_2024["South_wind"].quantile(DUNKELFLAUTE_QUANTILE)
north_low_g = df_ns_wind_2024["North_wind"] <= north_thresh_g
south_low_g = df_ns_wind_2024["South_wind"] <= south_thresh_g
p_all_2 = (north_low_g & south_low_g & national_low_g).sum() / national_low_g.sum()
p_any_2 = ((north_low_g | south_low_g) & national_low_g).sum() / national_low_g.sum()
granularity_results.append({"granularity": "2 regions (N/S)", "n_regions": 2, "p_all_low": p_all_2, "p_any_low": p_any_2})

quad_thresh_g = {q: df_quad[f"{q}_wind"].quantile(DUNKELFLAUTE_QUANTILE) for q in QUADRANTS}
quad_low_g = pd.DataFrame({q: df_quad[f"{q}_wind"] <= quad_thresh_g[q] for q in QUADRANTS})
all_low_4 = quad_low_g.all(axis=1)
any_low_4 = quad_low_g.any(axis=1)
p_all_4 = (all_low_4 & national_low_g).sum() / national_low_g.sum()
p_any_4 = (any_low_4 & national_low_g).sum() / national_low_g.sum()
granularity_results.append({"granularity": "4 regions (quadrants)", "n_regions": 4, "p_all_low": p_all_4, "p_any_low": p_any_4})

granularity_df = pd.DataFrame(granularity_results)
print("\n" + "=" * 95)
print("Wind, 2000-2024, conditional on the national aggregate already being in its own bottom decile:")
print(f"{'Granularity':<26}{'P(ALL sub-regions also low)':>30}{'P(ANY sub-region also low)':>30}")
print("=" * 95)
for _, r in granularity_df.iterrows():
    print(f"{r['granularity']:<26}{r['p_all_low']:>29.1%}{r['p_any_low']:>30.1%}")

print("\nStorm cut-out proxy, hours reaching it at each granularity (national mean, each half, each quadrant):")
HUB_SCALING_G = (100 / 10) ** 0.14
cutout_10m_g = 25.0 / HUB_SCALING_G
print(f"  National: {(national_g >= cutout_10m_g).sum()}  North: {(df_ns_wind_2024['North_wind'] >= cutout_10m_g).sum()}  "
      f"South: {(df_ns_wind_2024['South_wind'] >= cutout_10m_g).sum()}  " +
      "  ".join(f"{q}: {(df_quad[f'{q}_wind'] >= cutout_10m_g).sum()}" for q in QUADRANTS))

In [ ]:
# 34. Overheating (low wind, high irradiance, high temperature, daylight): observed rate and two proxy indices
#       PV cell temperature (NOCT model): T_cell = T_ambient + (NOCT - 20) / 800 * G, NOCT = 45 C;
#         derating = -0.4 %/K * max(0, T_cell - 25 C)
#       overhead-line ampacity index relative to the static rating point (35 C, 0.6 m/s perpendicular wind):
#         wind anchors (m/s -> % of baseline): 0 -> -22.5 %, 0.6 -> 0 %, 2 -> +30 %, 5 -> +65 % (interpolated)
#         temperature adjustment: -1.25 %/K relative to 35 C
#         index = 100 * (1 + wind_adj) * (1 + temp_adj)

temp_hi_day_new = df_day_new["temp"].quantile(1 - Q_SCENARIO)

overheating_flag = (df_day_new["wind"] <= wind_lo_new) & (df_day_new["solar"] >= solar_hi_new) & (df_day_new["temp"] >= temp_hi_day_new)
overheating_rate = overheating_flag.mean()
print(f"Low Wind + High Solar + High Temp: {overheating_rate:.3%} of daylight hours "
      f"({int(overheating_flag.sum())} hours in 25 years), {overheating_rate / (Q_SCENARIO ** 3):.2f}x independence baseline")

df_day_oh = df_day_new.copy()
df_day_oh["temp_C"] = df_day_oh["temp"] - 273.15

NOCT = 45.0  # C, typical datasheet value
df_day_oh["t_cell"] = df_day_oh["temp_C"] + (NOCT - 20) / 800 * df_day_oh["solar"]
df_day_oh["pv_derating_pct"] = -0.4 * np.clip(df_day_oh["t_cell"] - 25.0, 0, None)

print("\n--- PV panel overheating (NOCT proxy) ---")
print(f"Typical daylight hour:      T_cell = {df_day_oh['t_cell'].mean():.1f} C, derating = {df_day_oh['pv_derating_pct'].mean():.2f}%")
print(f"During this scenario:       T_cell = {df_day_oh.loc[overheating_flag, 't_cell'].mean():.1f} C, "
      f"derating = {df_day_oh.loc[overheating_flag, 'pv_derating_pct'].mean():.2f}%")
print(f"Worst hour during scenario: T_cell = {df_day_oh.loc[overheating_flag, 't_cell'].max():.1f} C, "
      f"derating = {df_day_oh.loc[overheating_flag, 'pv_derating_pct'].min():.2f}%")

wind_anchors = np.array([0.0, 0.6, 2.0, 5.0])
ampacity_pct_anchors = np.array([-22.5, 0.0, 30.0, 65.0])


def wind_ampacity_adj(w):
    w_clipped = np.clip(w, wind_anchors[0], wind_anchors[-1])
    return np.interp(w_clipped, wind_anchors, ampacity_pct_anchors) / 100.0


df_day_oh["wind_adj"] = wind_ampacity_adj(df_day_oh["wind"].to_numpy())
df_day_oh["temp_adj"] = -0.0125 * (df_day_oh["temp_C"] - 35.0)
df_day_oh["ampacity_index"] = 100 * (1 + df_day_oh["wind_adj"]) * (1 + df_day_oh["temp_adj"])

print("\n--- Overhead line ampacity (proxy index, 100 = German static rating design point) ---")
print(f"Typical daylight hour:      ampacity index = {df_day_oh['ampacity_index'].mean():.1f}")
print(f"During this scenario:       ampacity index = {df_day_oh.loc[overheating_flag, 'ampacity_index'].mean():.1f}")
print(f"Worst hour during scenario: ampacity index = {df_day_oh.loc[overheating_flag, 'ampacity_index'].min():.1f}")
frac_below_static = (df_day_oh.loc[overheating_flag, "ampacity_index"] < 100).mean()
print(f"Fraction of scenario hours below the static rating (index<100): {frac_below_static:.1%}")

corr_oh = df_day_oh[["ampacity_index", "pv_derating_pct"]].corr().iloc[0, 1]
print(f"\nCorrelation, ampacity index vs. PV derating, ALL daylight hours: {corr_oh:+.2f}")
print("(both move together - the same low-wind/high-temp combination pushes PV panels hotter")
print(" AND pushes the line carrying their output closer to its limit, at the same time)")


In [ ]:
# 35. Cold Dunkelflaute and overheating in the ensembles of the model AR + Copula (Section 8)

def sim_rate_arcop(flag_fn):
    rates = []
    for traj in arcop_trajectories.values():
        traj_day = traj.loc[is_daylight_new(traj.index)]
        rates.append(flag_fn(traj_day).mean())
    return np.array(rates)


# Cold Dunkelflaute
real_cold = (df_day_new["wind"] <= wind_lo_new) & (df_day_new["solar"] <= solar_lo_new) & (df_day_new["temp"] <= temp_lo_day_new)
sim_cold_rates = sim_rate_arcop(lambda d: (d["wind"] <= wind_lo_new) & (d["solar"] <= solar_lo_new) & (d["temp"] <= temp_lo_day_new))

print("=" * 90)
print("Cold Dunkelflaute: observed vs. AR+Copula")
print("=" * 90)
print(f"REAL:                          {real_cold.mean():.3%}")
print(f"SIMULATED (N={len(sim_cold_rates)}) mean:   {sim_cold_rates.mean():.3%}")
print(f"SIMULATED 5th-95th percentile: {np.percentile(sim_cold_rates, 5):.3%} - {np.percentile(sim_cold_rates, 95):.3%}")
print(f"Ratio (sim mean / real):       {sim_cold_rates.mean() / real_cold.mean():.2f}x")
inside_cold = np.percentile(sim_cold_rates, 5) <= real_cold.mean() <= np.percentile(sim_cold_rates, 95)
print(f"Real rate inside simulated 5-95% band: {inside_cold}")

# Overheating
sim_oh_rates = sim_rate_arcop(lambda d: (d["wind"] <= wind_lo_new) & (d["solar"] >= solar_hi_new) & (d["temp"] >= temp_hi_day_new))

print("\n" + "=" * 90)
print("Overheating (low wind + high solar + high temp): REAL vs. AR+Copula")
print("=" * 90)
print(f"REAL:                          {overheating_rate:.3%}")
print(f"SIMULATED (N={len(sim_oh_rates)}) mean:   {sim_oh_rates.mean():.3%}")
print(f"SIMULATED 5th-95th percentile: {np.percentile(sim_oh_rates, 5):.3%} - {np.percentile(sim_oh_rates, 95):.3%}")
print(f"Ratio (sim mean / real):       {sim_oh_rates.mean() / overheating_rate:.2f}x")
inside_oh = np.percentile(sim_oh_rates, 5) <= overheating_rate <= np.percentile(sim_oh_rates, 95)
print(f"Real rate inside simulated 5-95% band: {inside_oh}")

model_scenario_df = pd.DataFrame({
    "scenario": ["Cold Dunkelflaute", "Overheating"],
    "real_rate": [real_cold.mean(), overheating_rate],
    "sim_rate_mean": [sim_cold_rates.mean(), sim_oh_rates.mean()],
    "sim_rate_p5": [np.percentile(sim_cold_rates, 5), np.percentile(sim_oh_rates, 5)],
    "sim_rate_p95": [np.percentile(sim_cold_rates, 95), np.percentile(sim_oh_rates, 95)],
    "real_inside_band": [inside_cold, inside_oh],
})
print("\n" + model_scenario_df.round(4).to_string(index=False))


In [ ]:
# 36. Granularity sweep, observed against simulated, 1 / 2 / 4 / 16 regions: national model AR + Copula, the
#     two-region model AR + Spatial (Section 31) and the 4- and 16-region simulations of
#     analysis/simulate_4_16_regions.py; rates of the scenarios of Sections 32-35 and the probabilities that all
#     or any sub-regions flag, given that the national mean flags

import pickle

GRID_DIR = "./data"


def region_flag_36(scenario, df_region, thresh):
    w, s, t = df_region["wind"], df_region["solar"], df_region["temp"]
    if scenario == "Plain Dunkelflaute":
        return (w <= thresh["wind_lo"]) & (s <= thresh["solar_lo"])
    if scenario == "Cold Dunkelflaute":
        return (w <= thresh["wind_lo"]) & (s <= thresh["solar_lo"]) & (t <= thresh["temp_lo"])
    return (w <= thresh["wind_lo"]) & (s >= thresh["solar_hi"]) & (t >= thresh["temp_hi"])


NATIONAL_THRESH_36 = {"wind_lo": wind_lo_new, "solar_lo": solar_lo_new, "solar_hi": solar_hi_new,
                       "temp_lo": temp_lo_day_new, "temp_hi": temp_hi_day_new}


def national_flag_36(scenario, df_nat):
    return region_flag_36(scenario, df_nat, NATIONAL_THRESH_36)


SCENARIOS_36 = ["Plain Dunkelflaute", "Cold Dunkelflaute", "Overheating"]
sweep_rows = []

# 1 region: national AR+Copula (reuses Section 35)
real_plain_nat = ((df_day_new["wind"] <= wind_lo_new) & (df_day_new["solar"] <= solar_lo_new)).mean()
sim_plain_nat = sim_rate_arcop(lambda d: (d["wind"] <= wind_lo_new) & (d["solar"] <= solar_lo_new)).mean()
sweep_rows.append({"granularity": 1, "scenario": "Plain Dunkelflaute", "real_rate": real_plain_nat, "sim_rate": sim_plain_nat})
sweep_rows.append({"granularity": 1, "scenario": "Cold Dunkelflaute", "real_rate": real_cold.mean(), "sim_rate": sim_cold_rates.mean()})
sweep_rows.append({"granularity": 1, "scenario": "Overheating", "real_rate": overheating_rate, "sim_rate": sim_oh_rates.mean()})

# 2 regions: North/South, each already-calibrated spatial copula
real_idx_2 = df_ns_wind.index.intersection(df_ns_temp.index).intersection(df_ns_solar_raw.index)
real_idx_2 = real_idx_2[real_idx_2.year <= 2024]
real_regions_2 = {
    rn: pd.DataFrame({
        "wind": df_ns_wind.loc[real_idx_2, f"{rn}_wind"],
        "temp": df_ns_temp.loc[real_idx_2, f"{rn}_temp"],
        "solar": df_ns_solar_raw.loc[real_idx_2, f"{rn}_solar"],
    })
    for rn in ["North", "South"]
}
day_mask_2 = is_daylight_new(real_idx_2)
real_regions_2_day = {rn: df.loc[day_mask_2] for rn, df in real_regions_2.items()}
real_thresh_2 = {
    rn: {"wind_lo": df["wind"].quantile(Q_SCENARIO), "solar_lo": df["solar"].quantile(Q_SCENARIO), "solar_hi": df["solar"].quantile(1 - Q_SCENARIO),
         "temp_lo": df["temp"].quantile(Q_SCENARIO), "temp_hi": df["temp"].quantile(1 - Q_SCENARIO)}
    for rn, df in real_regions_2_day.items()
}
real_national_2 = pd.DataFrame({
    "wind": (real_regions_2["North"]["wind"] + real_regions_2["South"]["wind"]) / 2,
    "temp": (real_regions_2["North"]["temp"] + real_regions_2["South"]["temp"]) / 2,
    "solar": (real_regions_2["North"]["solar"] + real_regions_2["South"]["solar"]) / 2,
}).loc[day_mask_2]

day_mask_sim_2 = is_daylight_new(target_index_ns)
region_names_2 = ["North", "South"]
sim_rate_2 = {s: [] for s in SCENARIOS_36}
sim_all_2 = {s: [] for s in SCENARIOS_36}
sim_any_2 = {s: [] for s in SCENARIOS_36}
for j in range(N_TRAJECTORIES):
    sim_regions_j = {
        "North": pd.DataFrame({"wind": phys_batch_ns["North_wind"][j], "temp": phys_batch_ns_temp["North_temp"][j], "solar": phys_batch_ns_solar["North_solar"][j]}, index=target_index_ns).loc[day_mask_sim_2],
        "South": pd.DataFrame({"wind": phys_batch_ns["South_wind"][j], "temp": phys_batch_ns_temp["South_temp"][j], "solar": phys_batch_ns_solar["South_solar"][j]}, index=target_index_ns).loc[day_mask_sim_2],
    }
    sim_nat_j = pd.DataFrame({
        "wind": (sim_regions_j["North"]["wind"] + sim_regions_j["South"]["wind"]) / 2,
        "temp": (sim_regions_j["North"]["temp"] + sim_regions_j["South"]["temp"]) / 2,
        "solar": (sim_regions_j["North"]["solar"] + sim_regions_j["South"]["solar"]) / 2,
    })
    for scenario in SCENARIOS_36:
        nat_flag = national_flag_36(scenario, sim_nat_j)
        n_j = nat_flag.sum()
        sim_rate_2[scenario].append(n_j / len(sim_nat_j))
        if n_j > 0:
            flags_j = pd.DataFrame({rn: region_flag_36(scenario, sim_regions_j[rn], real_thresh_2[rn]) for rn in region_names_2})
            sim_all_2[scenario].append((flags_j.all(axis=1) & nat_flag).sum() / n_j)
            sim_any_2[scenario].append((flags_j.any(axis=1) & nat_flag).sum() / n_j)

for scenario in SCENARIOS_36:
    real_nat_flag_2 = national_flag_36(scenario, real_national_2)
    n_real_2 = real_nat_flag_2.sum()
    real_flags_2 = pd.DataFrame({rn: region_flag_36(scenario, real_regions_2_day[rn], real_thresh_2[rn]) for rn in region_names_2})
    real_all_2 = (real_flags_2.all(axis=1) & real_nat_flag_2).sum() / n_real_2
    real_any_2 = (real_flags_2.any(axis=1) & real_nat_flag_2).sum() / n_real_2
    sweep_rows.append({
        "granularity": 2, "scenario": scenario, "real_rate": real_nat_flag_2.mean(), "sim_rate": np.mean(sim_rate_2[scenario]),
        "real_p_all": real_all_2, "sim_p_all": np.mean(sim_all_2[scenario]), "real_p_any": real_any_2, "sim_p_any": np.mean(sim_any_2[scenario]),
    })

# 4 and 16 regions: simulations of analysis/simulate_4_16_regions.py
cells_36 = [f"r{r}c{c}" for r in range(4) for c in range(4)]
quadrant_groups_36 = {
    "Q_NW": [f"r{r}c{c}" for r in [0, 1] for c in [0, 1]], "Q_NE": [f"r{r}c{c}" for r in [0, 1] for c in [2, 3]],
    "Q_SW": [f"r{r}c{c}" for r in [2, 3] for c in [0, 1]], "Q_SE": [f"r{r}c{c}" for r in [2, 3] for c in [2, 3]],
}
grid_wind = pd.read_pickle(f"{GRID_DIR}/grid4x4_wind_2000-2024.pkl")
grid_temp = pd.read_pickle(f"{GRID_DIR}/grid4x4_temp_2000-2024.pkl")
grid_solar = pd.read_pickle(f"{GRID_DIR}/grid4x4_solar_2000-2024.pkl")
grid_idx = grid_wind.index.intersection(grid_temp.index).intersection(grid_solar.index)
grid_wind, grid_temp, grid_solar = grid_wind.loc[grid_idx], grid_temp.loc[grid_idx], grid_solar.loc[grid_idx]
day_mask_grid = is_daylight_new(grid_idx)

for n_reg, region_map in [(4, quadrant_groups_36), (16, {c: [c] for c in cells_36})]:
    region_names_n = list(region_map.keys())
    real_regions_n = {
        rn: pd.DataFrame({
            "wind": grid_wind[[f"{c}_wind" for c in cs]].mean(axis=1),
            "temp": grid_temp[[f"{c}_temp" for c in cs]].mean(axis=1),
            "solar": grid_solar[[f"{c}_solar" for c in cs]].mean(axis=1),
        }).loc[day_mask_grid]
        for rn, cs in region_map.items()
    }
    real_thresh_n = {
        rn: {"wind_lo": df["wind"].quantile(Q_SCENARIO), "solar_lo": df["solar"].quantile(Q_SCENARIO), "solar_hi": df["solar"].quantile(1 - Q_SCENARIO),
             "temp_lo": df["temp"].quantile(Q_SCENARIO), "temp_hi": df["temp"].quantile(1 - Q_SCENARIO)}
        for rn, df in real_regions_n.items()
    }
    real_national_n = pd.DataFrame({
        "wind": grid_wind[[f"{c}_wind" for c in cells_36]].mean(axis=1),
        "temp": grid_temp[[f"{c}_temp" for c in cells_36]].mean(axis=1),
        "solar": grid_solar[[f"{c}_solar" for c in cells_36]].mean(axis=1),
    }).loc[day_mask_grid]

    with open(f"{GRID_DIR}/sim_{n_reg}region_wind.pkl", "rb") as f:
        sim_w = pickle.load(f)
    with open(f"{GRID_DIR}/sim_{n_reg}region_temp.pkl", "rb") as f:
        sim_t = pickle.load(f)
    with open(f"{GRID_DIR}/sim_{n_reg}region_solar.pkl", "rb") as f:
        sim_s = pickle.load(f)
    target_index_n = sim_w["target_index"]
    day_mask_sim_n = is_daylight_new(target_index_n)

    for scenario in SCENARIOS_36:
        real_nat_flag_n = national_flag_36(scenario, real_national_n)
        n_real_n = real_nat_flag_n.sum()
        real_flags_n = pd.DataFrame({rn: region_flag_36(scenario, real_regions_n[rn], real_thresh_n[rn]) for rn in region_names_n})
        real_all_n = (real_flags_n.all(axis=1) & real_nat_flag_n).sum() / n_real_n
        real_any_n = (real_flags_n.any(axis=1) & real_nat_flag_n).sum() / n_real_n

        sim_rate_list, sim_all_list, sim_any_list = [], [], []
        for j in range(N_TRAJECTORIES):
            nat_j = pd.DataFrame({
                "wind": np.mean([sim_w["phys"][rn][j] for rn in region_names_n], axis=0),
                "temp": np.mean([sim_t["phys"][rn][j] for rn in region_names_n], axis=0),
                "solar": np.mean([sim_s["phys"][rn][j] for rn in region_names_n], axis=0),
            }, index=target_index_n).loc[day_mask_sim_n]
            nat_flag_j = national_flag_36(scenario, nat_j)
            n_j = nat_flag_j.sum()
            sim_rate_list.append(n_j / len(nat_j))
            if n_j > 0:
                flags_j = pd.DataFrame({
                    rn: region_flag_36(
                        scenario,
                        pd.DataFrame({"wind": sim_w["phys"][rn][j], "temp": sim_t["phys"][rn][j], "solar": sim_s["phys"][rn][j]}, index=target_index_n).loc[day_mask_sim_n],
                        real_thresh_n[rn],
                    )
                    for rn in region_names_n
                })
                sim_all_list.append((flags_j.all(axis=1) & nat_flag_j).sum() / n_j)
                sim_any_list.append((flags_j.any(axis=1) & nat_flag_j).sum() / n_j)

        sweep_rows.append({
            "granularity": n_reg, "scenario": scenario, "real_rate": real_nat_flag_n.mean(), "sim_rate": np.mean(sim_rate_list),
            "real_p_all": real_all_n, "sim_p_all": np.mean(sim_all_list), "real_p_any": real_any_n, "sim_p_any": np.mean(sim_any_list),
        })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df["ratio"] = sweep_df["sim_rate"] / sweep_df["real_rate"]
print("=" * 100)
print("REAL vs. SIMULATED rate, all four granularities, all three scenarios:")
print("=" * 100)
print(sweep_df[["granularity", "scenario", "real_rate", "sim_rate", "ratio"]].round(4).to_string(index=False))
print("\nP(ALL sub-regions also flag) / P(ANY sub-region also flags), conditional on national flag (2/4/16 only):")
print(sweep_df.loc[sweep_df["granularity"] > 1, ["granularity", "scenario", "real_p_all", "sim_p_all", "real_p_any", "sim_p_any"]].round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 5.0))
colors_36 = {"Plain Dunkelflaute": "#00549F", "Cold Dunkelflaute": "#8EBAE5", "Overheating": "#EB811B"}
for scenario in SCENARIOS_36:
    sub = sweep_df[sweep_df["scenario"] == scenario].sort_values("granularity")
    ax.plot(sub["granularity"], sub["ratio"], marker="o", color=colors_36[scenario], label=scenario)
ax.axhline(1.0, color="#9AA5AB", linestyle="--", linewidth=1, label="Perfect (sim = real)")
ax.set_xscale("log", base=2)
ax.set_xticks([1, 2, 4, 16])
ax.set_xticklabels(["1", "2", "4", "16"])
ax.set_xlabel("Number of regions the model is fit and simulated on")
ax.set_ylabel("Simulated rate / real rate")
ax.set_title("Model fidelity for compound scenarios degrades as spatial resolution increases")
ax.legend(frameon=False, loc="best")
fig.tight_layout()
plt.savefig("real_vs_sim_granularity_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

sweep_df.to_csv(f"{GRID_DIR}/notebook_granularity_sweep_results.csv", index=False)


In [ ]:
# 37. 4-region simulation with an R-vine copula across the regions (analysis/simulate_4region_vine.py), compared
#     with the Student-t version

import pickle

with open("./data/sim_4region_vine_diagnostics.pkl", "rb") as f:
    vine_diag = pickle.load(f)
print("Example fitted structure - wind, winter (every edge is Student-t, but with its OWN degrees of freedom):")
print(vine_diag["wind"]["winter"])

with open(f"{GRID_DIR}/sim_4region_wind.pkl", "rb") as f:
    mt_wind = pickle.load(f)
with open(f"{GRID_DIR}/sim_4region_temp.pkl", "rb") as f:
    mt_temp = pickle.load(f)
with open(f"{GRID_DIR}/sim_4region_solar.pkl", "rb") as f:
    mt_solar = pickle.load(f)
with open(f"{GRID_DIR}/sim_4region_vine_wind.pkl", "rb") as f:
    vine_wind = pickle.load(f)
with open(f"{GRID_DIR}/sim_4region_vine_temp.pkl", "rb") as f:
    vine_temp = pickle.load(f)
with open(f"{GRID_DIR}/sim_4region_vine_solar.pkl", "rb") as f:
    vine_solar = pickle.load(f)

real_regions_4 = {
    rn: pd.DataFrame({
        "wind": grid_wind[[f"{c}_wind" for c in cs]].mean(axis=1),
        "temp": grid_temp[[f"{c}_temp" for c in cs]].mean(axis=1),
        "solar": grid_solar[[f"{c}_solar" for c in cs]].mean(axis=1),
    }).loc[day_mask_grid]
    for rn, cs in quadrant_groups_36.items()
}
real_thresh_4 = {
    rn: {"wind_lo": df["wind"].quantile(Q_SCENARIO), "solar_lo": df["solar"].quantile(Q_SCENARIO), "solar_hi": df["solar"].quantile(1 - Q_SCENARIO),
         "temp_lo": df["temp"].quantile(Q_SCENARIO), "temp_hi": df["temp"].quantile(1 - Q_SCENARIO)}
    for rn, df in real_regions_4.items()
}
real_national_4 = pd.DataFrame({
    "wind": grid_wind[[f"{c}_wind" for c in cells_36]].mean(axis=1),
    "temp": grid_temp[[f"{c}_temp" for c in cells_36]].mean(axis=1),
    "solar": grid_solar[[f"{c}_solar" for c in cells_36]].mean(axis=1),
}).loc[day_mask_grid]
region_names_4 = list(quadrant_groups_36.keys())


def eval_4region_model(sim_w, sim_t, sim_s, label):
    target_index_4 = sim_w["target_index"]
    day_mask_sim_4 = is_daylight_new(target_index_4)
    rows = []
    for scenario in SCENARIOS_36:
        real_nat_flag = national_flag_36(scenario, real_national_4)
        n_real = real_nat_flag.sum()
        real_flags = pd.DataFrame({rn: region_flag_36(scenario, real_regions_4[rn], real_thresh_4[rn]) for rn in region_names_4})
        real_all = (real_flags.all(axis=1) & real_nat_flag).sum() / n_real
        real_any = (real_flags.any(axis=1) & real_nat_flag).sum() / n_real

        sim_rate_list, sim_all_list, sim_any_list = [], [], []
        for j in range(N_TRAJECTORIES):
            nat_j = pd.DataFrame({
                "wind": np.mean([sim_w["phys"][rn][j] for rn in region_names_4], axis=0),
                "temp": np.mean([sim_t["phys"][rn][j] for rn in region_names_4], axis=0),
                "solar": np.mean([sim_s["phys"][rn][j] for rn in region_names_4], axis=0),
            }, index=target_index_4).loc[day_mask_sim_4]
            nat_flag_j = national_flag_36(scenario, nat_j)
            n_j = nat_flag_j.sum()
            sim_rate_list.append(n_j / len(nat_j))
            if n_j > 0:
                flags_j = pd.DataFrame({
                    rn: region_flag_36(scenario,
                                        pd.DataFrame({"wind": sim_w["phys"][rn][j], "temp": sim_t["phys"][rn][j], "solar": sim_s["phys"][rn][j]}, index=target_index_4).loc[day_mask_sim_4],
                                        real_thresh_4[rn])
                    for rn in region_names_4
                })
                sim_all_list.append((flags_j.all(axis=1) & nat_flag_j).sum() / n_j)
                sim_any_list.append((flags_j.any(axis=1) & nat_flag_j).sum() / n_j)
        rows.append({"model": label, "scenario": scenario, "real_rate": real_nat_flag.mean(), "sim_rate": np.mean(sim_rate_list),
                     "ratio": np.mean(sim_rate_list) / real_nat_flag.mean()})
    return rows


vine_vs_mt_rows = eval_4region_model(mt_wind, mt_temp, mt_solar, "multivar-t (shared nu)")
vine_vs_mt_rows += eval_4region_model(vine_wind, vine_temp, vine_solar, "R-vine (per-edge)")
vine_vs_mt_df = pd.DataFrame(vine_vs_mt_rows)
print("\nReal-vs-simulated ratio, 4 regions, multivariate-t vs. R-vine:")
print(vine_vs_mt_df.pivot(index="scenario", columns="model", values="ratio").round(3).to_string())


In [ ]:
# 38. Cross-variable tail dependence of the seasonal AR residuals: conditional probabilities of joint extremes

from scipy.stats import kendalltau

print("Sanity check - reproducing Section 13's bulk-tau finding on these same residuals:")
tau_check_rows = []
for season in SEASONS:
    resid_s = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, variables]
    for a, b in [("wind", "solar"), ("wind", "temp"), ("solar", "temp")]:
        tau, _ = kendalltau(resid_s[a], resid_s[b])
        tau_check_rows.append({"season": season, "pair": f"{a}-{b}", "tau": tau})
tau_check_df = pd.DataFrame(tau_check_rows)
print(tau_check_df.pivot(index="season", columns="pair", values="tau").round(3).to_string())

print("\nTail-conditional probability (residual level), vs. the Q=10% independence baseline:")
print("(Overheating needs: wind low->solar high, wind low->temp high, solar high->temp high)")
crossvar_tail_rows = []
for season in SEASONS:
    resid_s = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, variables]
    checks = [
        ("wind low -> solar high", resid_s["wind"] <= resid_s["wind"].quantile(Q_SCENARIO), resid_s["solar"], resid_s["solar"].quantile(1 - Q_SCENARIO), "ge"),
        ("wind low -> temp high", resid_s["wind"] <= resid_s["wind"].quantile(Q_SCENARIO), resid_s["temp"], resid_s["temp"].quantile(1 - Q_SCENARIO), "ge"),
        ("solar high -> temp high", resid_s["solar"] >= resid_s["solar"].quantile(1 - Q_SCENARIO), resid_s["temp"], resid_s["temp"].quantile(1 - Q_SCENARIO), "ge"),
        ("wind low -> solar low", resid_s["wind"] <= resid_s["wind"].quantile(Q_SCENARIO), resid_s["solar"], resid_s["solar"].quantile(Q_SCENARIO), "le"),
    ]
    for label, mask_a, series_b, thresh_b, direction in checks:
        hit_b = (series_b >= thresh_b) if direction == "ge" else (series_b <= thresh_b)
        p = hit_b[mask_a].mean()
        crossvar_tail_rows.append({"season": season, "relationship": label, "ratio_vs_indep": p / Q_SCENARIO})

crossvar_tail_df = pd.DataFrame(crossvar_tail_rows)
print(crossvar_tail_df.pivot(index="season", columns="relationship", values="ratio_vs_indep").round(2).to_string())


In [ ]:
# 39. Cross-variable C-vine (temperature as root, Gumbel pair copulas) calibrated by bisection to the observed
#     conditional probabilities of joint extremes (tree 1: wind-temperature and irradiance-temperature; tree 2
#     against the triple-joint rate); simulation with the AR reconstruction of Sections 7-8; overheating rate
#     against the observed rate and the independent draws

EPS_CV = 1e-6


def gumbel_implied_prob(theta, q, n_mc=300_000, seed=0):
    bicop = pv.Bicop(family=pv.BicopFamily.gumbel, parameters=np.array([[theta]]))
    uv = bicop.simulate(n=n_mc, seeds=[seed])
    mask = uv[:, 0] >= 1 - q
    return (uv[mask, 1] >= 1 - q).mean() if mask.sum() > 0 else 0.0


def calibrate_gumbel(target_prob, q, lo=1.0, hi=25.0, tol=1e-4, max_iter=40):
    p_lo, p_hi = gumbel_implied_prob(lo, q), gumbel_implied_prob(hi, q)
    if target_prob <= p_lo:
        return lo, p_lo
    if target_prob >= p_hi:
        return hi, p_hi
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        p_mid = gumbel_implied_prob(mid, q)
        if abs(p_mid - target_prob) < tol:
            return mid, p_mid
        if p_mid < target_prob:
            lo = mid
        else:
            hi = mid
    return mid, p_mid


# Step B: calibrate tree-1 (wind-temp, solar-temp), per season
print("Calibrating tree-1 (wind-temp, solar-temp) per season against the measured target...")
pairwise_theta = {}
for season in SEASONS:
    resid_s = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, variables]
    wind_lo_r, temp_hi_r, solar_hi_r = resid_s["wind"].quantile(Q_SCENARIO), resid_s["temp"].quantile(1 - Q_SCENARIO), resid_s["solar"].quantile(1 - Q_SCENARIO)
    wind_low_mask = resid_s["wind"] <= wind_lo_r
    target_wt = (resid_s.loc[wind_low_mask, "temp"] >= temp_hi_r).mean()
    target_st = (resid_s.loc[resid_s["solar"] >= solar_hi_r, "temp"] >= temp_hi_r).mean()
    theta_wt, _ = calibrate_gumbel(target_wt, Q_SCENARIO)
    theta_st, _ = calibrate_gumbel(target_st, Q_SCENARIO)
    pairwise_theta[season] = {"wt": theta_wt, "st": theta_st}
    print(f"  {season}: wind-temp target={target_wt/Q_SCENARIO:.2f}x theta={theta_wt:.3f}   "
          f"solar-temp target={target_st/Q_SCENARIO:.2f}x theta={theta_st:.3f}")

# tree-2 (wind-solar | temp), calibrated against the measured TRIPLE-joint rate
print("\nCalibrating tree-2 (wind-solar | temp) against the measured triple-joint rate...")


def simulate_triple_rate(theta_wt, theta_st, theta2, q, n=400_000, seed=0):
    bicop_wt = pv.Bicop(family=pv.BicopFamily.gumbel, parameters=np.array([[theta_wt]]))
    bicop_st = pv.Bicop(family=pv.BicopFamily.gumbel, parameters=np.array([[theta_st]]))
    bicop_t2 = pv.Bicop(family=pv.BicopFamily.gumbel, parameters=np.array([[theta2]]))
    rng = np.random.default_rng(seed)
    w_temp = rng.uniform(size=n)
    w_wind = rng.uniform(size=n)
    w_solar = rng.uniform(size=n)
    u_wind = 1 - np.asarray(bicop_wt.hinv2(np.column_stack([w_wind, w_temp]))).ravel()
    v = np.asarray(bicop_t2.hinv2(np.column_stack([w_solar, w_wind]))).ravel()
    u_solar = np.asarray(bicop_st.hinv2(np.column_stack([v, w_temp]))).ravel()
    return ((u_wind <= q) & (u_solar >= 1 - q) & (w_temp >= 1 - q)).mean()


tree2_theta = {}
for season in SEASONS:
    resid_s = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, variables]
    wind_lo_r, temp_hi_r, solar_hi_r = resid_s["wind"].quantile(Q_SCENARIO), resid_s["temp"].quantile(1 - Q_SCENARIO), resid_s["solar"].quantile(1 - Q_SCENARIO)
    triple_target = ((resid_s["wind"] <= wind_lo_r) & (resid_s["solar"] >= solar_hi_r) & (resid_s["temp"] >= temp_hi_r)).mean()

    theta_wt, theta_st = pairwise_theta[season]["wt"], pairwise_theta[season]["st"]
    lo, hi = 1.0, 30.0
    p_lo = simulate_triple_rate(theta_wt, theta_st, lo, Q_SCENARIO)
    if triple_target <= p_lo:
        theta2 = lo
    else:
        for _ in range(20):
            mid = (lo + hi) / 2
            p_mid = simulate_triple_rate(theta_wt, theta_st, mid, Q_SCENARIO)
            if p_mid < triple_target:
                lo = mid
            else:
                hi = mid
        theta2 = (lo + hi) / 2
    tree2_theta[season] = theta2
    print(f"  {season}: triple target={triple_target/Q_SCENARIO**3:.2f}x independence, "
          f"tree-1-only implied={p_lo/Q_SCENARIO**3:.2f}x -> theta2={theta2:.3f}"
          f"{' (independence floor - tree-1 alone already explains it)' if theta2 <= 1.001 else ''}")

# simulate national wind/temp/solar jointly via the calibrated C-vine
print("\nSimulating national wind/temp/solar via the calibrated cross-variable C-vine...")
season_labels_target_cv = np.array([get_season(m) for m in target_index.month])
resid_sim_batch_cv = {col: np.empty((N_TRAJECTORIES, len(target_index))) for col in variables}
season_seed_offsets_cv = {"winter": 0, "spring": 1000, "summer": 2000, "autumn": 3000}
for season in SEASONS:
    mask = season_labels_target_cv == season
    n_hours = int(mask.sum())
    resid_s = residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == season, variables]
    n_pool = len(resid_s)
    sorted_pool = {col: np.sort(resid_s[col].to_numpy()) for col in variables}
    bicop_wt = pv.Bicop(family=pv.BicopFamily.gumbel, parameters=np.array([[pairwise_theta[season]["wt"]]]))
    bicop_st = pv.Bicop(family=pv.BicopFamily.gumbel, parameters=np.array([[pairwise_theta[season]["st"]]]))
    bicop_t2 = pv.Bicop(family=pv.BicopFamily.gumbel, parameters=np.array([[tree2_theta[season]]]))

    for j, seed in enumerate(seeds):
        rng = np.random.default_rng(int(seed) + season_seed_offsets_cv[season] + 555)
        u_temp = rng.uniform(size=n_hours)
        w_wind = rng.uniform(size=n_hours)
        w_solar = rng.uniform(size=n_hours)
        u_wind = 1 - np.asarray(bicop_wt.hinv2(np.column_stack([w_wind, u_temp]))).ravel()
        v = np.asarray(bicop_t2.hinv2(np.column_stack([w_solar, w_wind]))).ravel()
        u_solar = np.asarray(bicop_st.hinv2(np.column_stack([v, u_temp]))).ravel()

        u_wind = np.clip(u_wind, EPS_CV, 1 - EPS_CV)
        u_temp_c = np.clip(u_temp, EPS_CV, 1 - EPS_CV)
        u_solar = np.clip(u_solar, EPS_CV, 1 - EPS_CV)
        idx_wind = np.clip((u_wind * n_pool).astype(int), 0, n_pool - 1)
        idx_temp = np.clip((u_temp_c * n_pool).astype(int), 0, n_pool - 1)
        idx_solar = np.clip((u_solar * n_pool).astype(int), 0, n_pool - 1)
        resid_sim_batch_cv["wind"][j, mask] = sorted_pool["wind"][idx_wind]
        resid_sim_batch_cv["temp"][j, mask] = sorted_pool["temp"][idx_temp]
        resid_sim_batch_cv["solar"][j, mask] = sorted_pool["solar"][idx_solar]

phys_cv = reconstruct_simulation_seasonal_batch(resid_sim_batch_cv, models_seasonal, target_mean_batch, target_index, df_anom, variables, get_season)
print("Cross-variable C-vine trajectories reconstructed.")

# evaluate against real data and against the independent-draw baseline
day_mask_target_cv = is_daylight_new(target_index)
cv_results = []
for scenario_name in ["Plain Dunkelflaute", "Cold Dunkelflaute", "Overheating"]:
    if scenario_name == "Plain Dunkelflaute":
        flag_fn = lambda d: (d["wind"] <= wind_lo_new) & (d["solar"] <= solar_lo_new)
    elif scenario_name == "Cold Dunkelflaute":
        flag_fn = lambda d: (d["wind"] <= wind_lo_new) & (d["solar"] <= solar_lo_new) & (d["temp"] <= temp_lo_day_new)
    else:
        flag_fn = lambda d: (d["wind"] <= wind_lo_new) & (d["solar"] >= solar_hi_new) & (d["temp"] >= temp_hi_day_new)

    real_rate = flag_fn(df_day_new).mean()
    indep_rates = sim_rate_arcop(flag_fn)
    cv_rates = []
    for j in range(N_TRAJECTORIES):
        traj = pd.DataFrame({c: phys_cv[c][j] for c in variables}, index=target_index).loc[day_mask_target_cv]
        cv_rates.append(flag_fn(traj).mean())
    cv_rates = np.array(cv_rates)
    cv_results.append({"scenario": scenario_name, "real_rate": real_rate, "indep_sim_mean": indep_rates.mean(),
                        "indep_ratio": indep_rates.mean() / real_rate, "cvine_sim_mean": cv_rates.mean(),
                        "cvine_ratio": cv_rates.mean() / real_rate,
                        "real_inside_cvine_band": np.percentile(cv_rates, 5) <= real_rate <= np.percentile(cv_rates, 95)})

cv_results_df = pd.DataFrame(cv_results)
print("\n" + "=" * 100)
print("Real vs. simulated, independent draws (production) vs. calibrated cross-variable C-vine:")
print("=" * 100)
print(cv_results_df.round(4).to_string(index=False))


In [ ]:
# 40. Pairwise joint extremes of the overheating scenario in summer, in the reconstructed trajectories

summer_mask_target = target_index.month.isin([6, 7, 8])
df_day_summer_new = df_day_new[df_day_new.index.month.isin([6, 7, 8])]


def pair_excess(df, col_a, thresh_a, dir_a, col_b, thresh_b, dir_b):
    mask_a = (df[col_a] <= thresh_a) if dir_a == "le" else (df[col_a] >= thresh_a)
    hit_b = (df[col_b] <= thresh_b) if dir_b == "le" else (df[col_b] >= thresh_b)
    p_indep = mask_a.mean() * hit_b.mean()
    return (mask_a & hit_b).mean() / p_indep if p_indep > 0 else float("nan")


pair_checks = [
    ("wind_lo <-> solar_hi", "wind", wind_lo_new, "le", "solar", solar_hi_new, "ge"),
    ("wind_lo <-> temp_hi", "wind", wind_lo_new, "le", "temp", temp_hi_day_new, "ge"),
    ("solar_hi <-> temp_hi", "solar", solar_hi_new, "ge", "temp", temp_hi_day_new, "ge"),
]

print("Within summer, at the scenario's year-round thresholds - excess over independence (1.0x = none):")
print(f"{'Pairwise link':<24}{'REAL':>10}{'Cross-var C-vine (simulated)':>32}")
gap_rows = []
for label, ca, ta, da, cb, tb, db in pair_checks:
    real_excess = pair_excess(df_day_summer_new, ca, ta, da, cb, tb, db)
    sim_excess_list = []
    for j in range(N_TRAJECTORIES):
        traj = pd.DataFrame({c: phys_cv[c][j] for c in variables}, index=target_index).loc[day_mask_target_cv & summer_mask_target]
        sim_excess_list.append(pair_excess(traj, ca, ta, da, cb, tb, db))
    sim_excess = np.nanmean(sim_excess_list)
    print(f"{label:<24}{real_excess:>9.2f}x{sim_excess:>31.2f}x")
    gap_rows.append({"link": label, "real_excess": real_excess, "sim_excess": sim_excess})

gap_df = pd.DataFrame(gap_rows)


In [ ]:
# 41. Persistence of Dunkelflaute episodes: runs of consecutive days with at least one Dunkelflaute hour,
#     observed and in the three models; the flags of every model are cached in data/dunkelflaute_flags_by_arm.pkl

def _dunkelflaute_flag_fast(df, day_mask):
    sub = df.loc[day_mask]
    return (sub["wind"] <= wind_thresh) & (sub["solar"] <= solar_thresh)


def _day_spells(flag):
    """Run lengths in consecutive CALENDAR DAYS carrying >=1 flagged hour.
    np.unique returns a datetime64 ndarray, so it is wrapped back into a
    DatetimeIndex to get a TimedeltaIndex from the diff."""
    days = pd.DatetimeIndex(np.unique(flag.index[flag.to_numpy()].normalize()))
    if len(days) == 0:
        return np.array([]), 0
    gaps = np.asarray((days[1:] - days[:-1]).total_seconds()) != 86400.0
    breaks = np.concatenate([[0], np.where(gaps)[0] + 1, [len(days)]])
    return np.diff(breaks), len(days)


def _summarise_spells(spells, n_flagged_days, n_years):
    return {
        "df days/yr": n_flagged_days / n_years,
        "spells/yr": len(spells) / n_years,
        "mean spell (d)": spells.mean() if len(spells) else np.nan,
        "share >=2d": (spells >= 2).mean() if len(spells) else np.nan,
        "share >=3d": (spells >= 3).mean() if len(spells) else np.nan,
        "share >=5d": (spells >= 5).mean() if len(spells) else np.nan,
        "max spell (d)": spells.max() if len(spells) else 0,
    }


real_day_mask = is_daylight(df_train.index)
real_flag = _dunkelflaute_flag_fast(df_train[variables], real_day_mask)
n_years_real = df_train.index.year.nunique()
_sp, _nd = _day_spells(real_flag)
rows = {"REAL (2000-2024)": _summarise_spells(_sp, _nd, n_years_real)}

flag_cache = {"real": real_flag}
traj_day_mask = is_daylight(target_index)  # all trajectories share this calendar
for label, trajs in [("AR", ar_trajectories), ("VAR", var_trajectories), ("AR+Copula", arcop_trajectories)]:
    spells_all, days_total, flags = [], 0, []
    for traj in trajs.values():
        f = _dunkelflaute_flag_fast(traj[variables], traj_day_mask)
        sp, nd = _day_spells(f)
        spells_all.append(sp)
        days_total += nd
        flags.append(f.to_numpy())
    spells_all = np.concatenate([a for a in spells_all if len(a)]) if any(len(a) for a in spells_all) else np.array([])
    rows[label] = _summarise_spells(spells_all, days_total, len(trajs))
    flag_cache[label] = np.vstack(flags)

episode_df = pd.DataFrame(rows).T
print("Dunkelflaute persistence: consecutive days carrying at least one dunkelflaute hour")
print(episode_df.round(3).to_string())

real_row = episode_df.loc["REAL (2000-2024)"]
diff = (episode_df.drop(index="REAL (2000-2024)") - real_row).abs()
print("\nAbsolute distance from real (smaller is better):")
print(diff.round(3).to_string())
print("\nClosest arm per statistic:")
for stat, arm in diff.idxmin().items():
    print(f"  {stat:<16} -> {arm}")

import pickle as _pickle
with open("./data/dunkelflaute_flags_by_arm.pkl", "wb") as _f:
    _pickle.dump({"flag_cache": flag_cache,
                   "real_index": real_flag.index,                       # daylight hours, 2000-2024
                   "traj_index": target_index[traj_day_mask],           # daylight hours, sim calendar
                   "wind_thresh": wind_thresh, "solar_thresh": solar_thresh}, _f)
print("\nCached per-arm dunkelflaute flags to data/dunkelflaute_flags_by_arm.pkl")
print("(so this comparison can be re-cut without re-running the notebook)")


In [ ]:
# 42. Variant D: wind in Box-Cox space with lambda = 0.5 and the residuals of all three variables drawn as joint
#     24-h blocks (no temporal copula of irradiance); rates and coverage against the other models

WIND_LAMBDA = 0.5
BLOCK_HOURS = 24


def _fwd_wind(x, lam=WIND_LAMBDA):
    return (x ** lam - 1.0) / lam


def _inv_wind(y, lam=WIND_LAMBDA):
    return np.clip(lam * y + 1.0, 1e-9, None) ** (1.0 / lam)


# transform, climatology, trend, anomalies (own pipeline: wind differs)
_cols_d = list(variables)
df_d = df_train[_cols_d].copy()
df_d["wind"] = _fwd_wind(df_d["wind"].to_numpy())
df_d["solar"] = np.log(df_d["solar"] + SOLAR_LOG_EPS)

_tc_d = fit_linear_trend_ns(df_d, _cols_d, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
_det_d = df_d[_cols_d] - trend_value_ns(df_d.index, _tc_d, _cols_d)
_clim_d = build_smoothed_climatology_ns(_det_d, _cols_d, window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
_mean_d = map_climatology_ns(df_d.index, _clim_d, _cols_d) + trend_value_ns(df_d.index, _tc_d, _cols_d)
df_anom_d = df_d[_cols_d] - _mean_d

_models_d, _resid_d = {}, {}
for _s in SEASONS:
    _bid = season_block_ids(df_anom_d.index, _s)
    _models_d[_s] = {c: fit_ar_model_seasonal_one(y=df_anom_d[c], block_id=_bid, col=c, maxlag=AR_MAX_LAG)
                     for c in _cols_d}
    # align the three variables onto common timestamps so a JOINT block
    # draw takes all three from the same real consecutive hours
    _resid_d[_s] = pd.DataFrame({c: _models_d[_s][c]["resid"] for c in _cols_d}).dropna()
print("Arm D AR orders:", {s: [len(_models_d[s][c]["lags"]) for c in _cols_d] for s in SEASONS})

# target mean for the simulation calendar, incl. the same year-bootstrap
_sim_mean_d = map_climatology_ns(sim_calendar_mean.index, _clim_d, _cols_d) + \
              trend_value_ns(sim_calendar_mean.index, _tc_d, _cols_d)
_ybo_d = df_anom_d.groupby(df_anom_d.index.year).mean()
_T_d = len(target_index)
_seeds_d = [BASE_SEED + r for r in range(N_TRAJECTORIES)]
_tmb_d = {c: np.empty((N_TRAJECTORIES, _T_d)) for c in _cols_d}
for _j, _seed in enumerate(_seeds_d):
    _rng = np.random.default_rng(_seed + 1_000_000)
    _yr = int(_rng.choice(_ybo_d.index.to_numpy()))
    for c in _cols_d:
        _tmb_d[c][_j] = _sim_mean_d[c].to_numpy() + _ybo_d.loc[_yr, c]

# joint 24h block residual draws
_slab_d = np.array([get_season(m) for m in target_index.month])
_resid_sim_d = {c: np.empty((N_TRAJECTORIES, _T_d)) for c in _cols_d}
for _s in SEASONS:
    _msk = _slab_d == _s
    _n = int(_msk.sum())
    _pool = _resid_d[_s]
    _arr = {c: _pool[c].to_numpy() for c in _cols_d}
    _np = len(_pool)
    _nb = int(np.ceil(_n / BLOCK_HOURS))
    for _j, _seed in enumerate(_seeds_d):
        _rng = np.random.default_rng(int(_seed) + hash(_s) % 9999 + 4242)
        _starts = _rng.integers(0, max(_np - BLOCK_HOURS, 1), size=_nb)
        _idx = np.clip(np.concatenate([np.arange(b, b + BLOCK_HOURS) for b in _starts])[:_n], 0, _np - 1)
        for c in _cols_d:
            _resid_sim_d[c][_j, _msk] = _arr[c][_idx]

# AR reconstruction, then back-transform
_maxlag_d = max(max(_models_d[s][c]["lags"], default=0) for s in SEASONS for c in _cols_d)
phys_batch_d = {}
for c in _cols_d:
    _hist = df_anom_d[c].dropna().to_numpy(dtype=float)[-_maxlag_d:]
    _sim = np.empty((N_TRAJECTORIES, _maxlag_d + _T_d), dtype=float)
    _sim[:, :_maxlag_d] = _hist[np.newaxis, :]
    for _t in range(_T_d):
        _i = _t + _maxlag_d
        _m = _models_d[_slab_d[_t]][c]
        _p = _m["params"]
        _val = np.full(N_TRAJECTORIES, float(_p["const"]))
        for _L in _m["lags"]:
            _val += float(_p[f"{c}.L{_L}"]) * _sim[:, _i - _L]
        _sim[:, _i] = _val + _resid_sim_d[c][:, _t]
    _ph = _sim[:, _maxlag_d:] + _tmb_d[c]
    if c == "solar":
        _ph = np.clip(np.exp(_ph) - SOLAR_LOG_EPS, 0, None)
    elif c == "wind":
        _ph = _inv_wind(_ph)
    phys_batch_d[c] = _ph
print("Arm D reconstructed:", {c: (round(float(phys_batch_d[c].mean()), 2),
                                    round(float(phys_batch_d[c].min()), 2)) for c in _cols_d})

# dunkelflaute marginals / rate / multiplier, against the other arms
_dmask_d = is_daylight(target_index)
_w_d, _s_d = phys_batch_d["wind"][:, _dmask_d], phys_batch_d["solar"][:, _dmask_d]
_rd_d = df_train.loc[is_daylight(df_train.index)]
_rw, _rs = _rd_d["wind"].to_numpy(), _rd_d["solar"].to_numpy()

# reference values of the model AR + Copula (Sections 12-14), recomputed
_pw_ = np.vstack([t["wind"].to_numpy() for t in arcop_trajectories.values()])[:, is_daylight(target_index)]
_ps_ = np.vstack([t["solar"].to_numpy() for t in arcop_trajectories.values()])[:, is_daylight(target_index)]
_prod_pw = float((_pw_ <= wind_thresh).mean())
_prod_ps = float((_ps_ <= solar_thresh).mean())
_prod_rate = float(((_pw_ <= wind_thresh) & (_ps_ <= solar_thresh)).mean())
_prod_mult = _prod_rate / (_prod_pw * _prod_ps)


def _mult(w, s):
    pw, ps = (w <= wind_thresh).mean(), (s <= solar_thresh).mean()
    return ((w <= wind_thresh) & (s <= solar_thresh)).mean() / (pw * ps), pw, ps

_rm, _rpw, _rps = _mult(_rw, _rs)
_dm, _dpw, _dps = _mult(_w_d, _s_d)
arm_d_summary = pd.DataFrame([
    {"metric": "P(low wind)", "real": _rpw, "AR+Copula": _prod_pw, "Arm D": _dpw},
    {"metric": "P(low solar)", "real": _rps, "AR+Copula": _prod_ps, "Arm D": _dps},
    {"metric": "dunkelflaute rate", "real": ((_rw <= wind_thresh) & (_rs <= solar_thresh)).mean(),
     "AR+Copula": _prod_rate, "Arm D": ((_w_d <= wind_thresh) & (_s_d <= solar_thresh)).mean()},
    {"metric": "dependence multiplier", "real": _rm, "AR+Copula": _prod_mult, "Arm D": _dm},
])
print("\nDunkelflaute metrics (AR+Copula column from Sections 12-14):")
print(arm_d_summary.round(4).to_string(index=False))

# coverage of the extreme years, mirroring Section 11/31 exactly
_daily_d = {}
for c in _cols_d:
    _df_nat = pd.DataFrame(phys_batch_d[c].T, index=target_index)
    _dly = _df_nat.resample("D").mean()
    _daily_d[c] = _dly.to_numpy().T
    _day_index_d = _dly.index
if not (len(_day_index_d) == len(day_index) and (_day_index_d.dayofyear == day_index.dayofyear).all()):
    print("WARNING: Arm D day index differs from Section 11's - coverage not directly comparable")
_doy_d = _day_index_d.dayofyear

_cov_rows_d = []
for _yl, _ydf in extreme_years_to_plot.items():
    _yd = _ydf.resample("D").mean()
    _ydoy = _yd.index.dayofyear
    for c in _cols_d:
        _stack = _daily_d[c]
        _lo = np.interp(_ydoy, _doy_d, np.percentile(_stack, 5, axis=0))
        _hi = np.interp(_ydoy, _doy_d, np.percentile(_stack, 95, axis=0))
        _rv = _yd[c].to_numpy()
        _cov_rows_d.append({"extreme_year": _yl, "variable": c, "model": "Arm D",
                            "coverage": ((_rv >= _lo) & (_rv <= _hi)).mean()})
arm_d_coverage = pd.DataFrame(_cov_rows_d)
print("\nCoverage of extreme years, Arm D vs the three existing arms:")
_all_cov = pd.concat([combined_coverage_df, arm_d_coverage], ignore_index=True)
print(_all_cov.pivot_table(index="variable", columns="model", values="coverage").round(3).to_string())


In [ ]:
# 43. Variant E: wind in Box-Cox space with lambda = 0.5, residual mechanism of the model AR + Copula unchanged
#     (bootstrap per variable, hour-of-day scale of wind, temporal copula of irradiance); temperature and
#     irradiance fits reused from Section 7

WIND_LAMBDA_E = 0.5


def _fwd_wind_e(x, lam=WIND_LAMBDA_E):
    return (x ** lam - 1.0) / lam


def _inv_wind_e(y, lam=WIND_LAMBDA_E):
    return np.clip(lam * y + 1.0, 1e-9, None) ** (1.0 / lam)


# wind-only pipeline: transform, trend, climatology, anomalies
_df_e = df_train[["wind"]].copy()
_df_e["wind"] = _fwd_wind_e(_df_e["wind"].to_numpy())

_tc_e = fit_linear_trend_ns(_df_e, ["wind"], window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
_det_e = _df_e[["wind"]] - trend_value_ns(_df_e.index, _tc_e, ["wind"])
_clim_e = build_smoothed_climatology_ns(_det_e, ["wind"], window_days=CLIMATOLOGY_SMOOTHING_WINDOW_DAYS)
_mean_e = map_climatology_ns(_df_e.index, _clim_e, ["wind"]) + trend_value_ns(_df_e.index, _tc_e, ["wind"])
_anom_e_wind = (_df_e[["wind"]] - _mean_e)["wind"]

# wind AR fit, same estimator/criterion as production
_models_e_wind = {}
_wind_pool_e = {}
for _s in SEASONS:
    _bid = season_block_ids(_anom_e_wind.index, _s)
    _fit = fit_ar_model_seasonal_one(y=_anom_e_wind, block_id=_bid, col="wind", maxlag=AR_MAX_LAG)
    _models_e_wind[_s] = _fit
    _wind_pool_e[_s] = _fit["resid"].dropna().to_numpy()
print("Arm E wind AR orders:", {s: len(_models_e_wind[s]["lags"]) for s in SEASONS},
      " (Arm C wind:", {s: len(models_seasonal[s]["wind"]["lags"]) for s in SEASONS}, ")")

# per-variable model table: wind from Arm E, temp/solar from production
_models_e = {s: {"wind": _models_e_wind[s],
                 "temp": models_seasonal[s]["temp"],
                 "solar": models_seasonal[s]["solar"]} for s in SEASONS}

# target mean: wind from Arm E's climatology, temp/solar from production
_sim_mean_e_wind = (map_climatology_ns(sim_calendar_mean.index, _clim_e, ["wind"])
                    + trend_value_ns(sim_calendar_mean.index, _tc_e, ["wind"]))["wind"].to_numpy()
_ybo_e_wind = _anom_e_wind.groupby(_anom_e_wind.index.year).mean()

_T_e = len(target_index)
_seeds_e = list(seeds)
_tmb_e = {}
for _col in variables:
    _tmb_e[_col] = np.empty((N_TRAJECTORIES, _T_e))
for _j, _seed in enumerate(_seeds_e):
    # same year-bootstrap draw as production (same seed stream) so the
    # low-frequency component is shared and only wind's transform differs
    _rng_y = np.random.default_rng(_seed + 1_000_000)
    _yr = int(_rng_y.choice(_ybo_e_wind.index.to_numpy()))
    _tmb_e["wind"][_j] = _sim_mean_e_wind + float(_ybo_e_wind.loc[_yr])
for _col in ("temp", "solar"):
    _tmb_e[_col] = target_mean_batch[_col]

# hour-of-day wind scale, recomputed on Arm E's anomaly space
_anom_e_df = pd.DataFrame({"wind": _anom_e_wind})
_wind_hour_scale_raw_e = compute_wind_hour_scale(_anom_e_df, SEASONS, get_season)
_slab_e = np.array([get_season(m) for m in target_index.month])
_hour_e = target_index.hour.to_numpy()
_wind_scale_e = np.empty(_T_e)
for _s in SEASONS:
    _m = _slab_e == _s
    _raw = _wind_hour_scale_raw_e[_s][_hour_e[_m]]
    _wind_scale_e[_m] = 1.0 + WIND_HOUR_SCALE_STRENGTH[_s] * (_raw - 1.0)

# residual simulation as in Section 8 (same loop order, generators and seeds); only the wind pool differs
# (Box-Cox residuals of this variant)
_pool_temp_e = {s: residuals_seasonal_ar.loc[residuals_seasonal_ar["season"] == s, "temp"].dropna().to_numpy()
                for s in SEASONS}
_resid_e = {c: np.empty((N_TRAJECTORIES, _T_e)) for c in variables}
for _j, _seed in enumerate(_seeds_e):
    _rng_b = np.random.default_rng(_seed)
    for _col, _pools in (("wind", _wind_pool_e), ("temp", _pool_temp_e)):
        for _s in SEASONS:
            _m = _slab_e == _s
            _resid_e[_col][_j, _m] = _rng_b.choice(_pools[_s], size=int(_m.sum()), replace=True)
    _resid_e["wind"][_j] = _resid_e["wind"][_j] * _wind_scale_e
    _resid_e["solar"][_j] = simulate_solar_temporal_sequence(
        solar_temporal_copulas=solar_temporal_copulas, solar_pool_by_season=solar_pool_by_season,
        season_labels=_slab_e, seed=_seed + 5_000_000,
    )

# AR reconstruction (inline: wind needs the Box-Cox inverse, not clip)
_train_hist_e = {"wind": _anom_e_wind.dropna().to_numpy(dtype=float),
                 "temp": df_anom["temp"].dropna().to_numpy(dtype=float),
                 "solar": df_anom["solar"].dropna().to_numpy(dtype=float)}
phys_batch_e = {}
for _col in variables:
    _maxlag = max(max(_models_e[s][_col]["lags"], default=0) for s in SEASONS)
    _step_const = np.empty(_T_e)
    _step_coefs = []
    for _t in range(_T_e):
        _mdl = _models_e[_slab_e[_t]][_col]
        _p = _mdl["params"]
        _step_const[_t] = float(_p["const"])
        _step_coefs.append({L: float(_p[f"{_col}.L{L}"]) for L in _mdl["lags"]})

    _sim = np.empty((N_TRAJECTORIES, _maxlag + _T_e), dtype=float)
    if _maxlag > 0:
        _sim[:, :_maxlag] = _train_hist_e[_col][-_maxlag:][np.newaxis, :]
    for _t in range(_T_e):
        _i = _t + _maxlag
        _val = np.full(N_TRAJECTORIES, _step_const[_t])
        for _L, _c in _step_coefs[_t].items():
            _val += _c * _sim[:, _i - _L]
        _sim[:, _i] = _val + _resid_e[_col][:, _t]

    _ph = _sim[:, _maxlag:] + _tmb_e[_col]
    if _col == "solar":
        _ph = np.clip(np.exp(_ph) - SOLAR_LOG_EPS, 0, None)
    elif _col == "wind":
        _ph = _inv_wind_e(_ph)
    phys_batch_e[_col] = _ph

print("Arm E reconstructed:", {c: (round(float(phys_batch_e[c].mean()), 2),
                                   round(float(phys_batch_e[c].min()), 3)) for c in variables})

# dunkelflaute marginals / rate / multiplier
_dmask_e = is_daylight(target_index)
_w_e, _s_e = phys_batch_e["wind"][:, _dmask_e], phys_batch_e["solar"][:, _dmask_e]
_rd_e = df_train.loc[is_daylight(df_train.index)]
_rw_e, _rs_e = _rd_e["wind"].to_numpy(), _rd_e["solar"].to_numpy()


# reference values of the model AR + Copula (Sections 12-14), recomputed
_pw_ = np.vstack([t["wind"].to_numpy() for t in arcop_trajectories.values()])[:, is_daylight(target_index)]
_ps_ = np.vstack([t["solar"].to_numpy() for t in arcop_trajectories.values()])[:, is_daylight(target_index)]
_prod_pw = float((_pw_ <= wind_thresh).mean())
_prod_ps = float((_ps_ <= solar_thresh).mean())
_prod_rate = float(((_pw_ <= wind_thresh) & (_ps_ <= solar_thresh)).mean())
_prod_mult = _prod_rate / (_prod_pw * _prod_ps)


def _mult_e(w, s):
    pw = (w <= wind_thresh).mean()
    ps = (s <= solar_thresh).mean()
    return ((w <= wind_thresh) & (s <= solar_thresh)).mean() / (pw * ps), pw, ps


_rm_e, _rpw_e, _rps_e = _mult_e(_rw_e, _rs_e)
_em_e, _epw_e, _eps_e = _mult_e(_w_e, _s_e)
arm_e_summary = pd.DataFrame([
    {"metric": "P(low wind)", "real": _rpw_e, "AR+Copula": _prod_pw, "Arm D": 0.0953, "Arm E": _epw_e},
    {"metric": "P(low solar)", "real": _rps_e, "AR+Copula": _prod_ps, "Arm D": 0.1007, "Arm E": _eps_e},
    {"metric": "dunkelflaute rate", "real": ((_rw_e <= wind_thresh) & (_rs_e <= solar_thresh)).mean(),
     "AR+Copula": _prod_rate, "Arm D": 0.0131,
     "Arm E": ((_w_e <= wind_thresh) & (_s_e <= solar_thresh)).mean()},
    {"metric": "dependence multiplier", "real": _rm_e, "AR+Copula": _prod_mult, "Arm D": 1.3662, "Arm E": _em_e},
])
print("\nDunkelflaute metrics:")
print(arm_e_summary.round(4).to_string(index=False))

# shape of the simulated wind distribution
from scipy.stats import skew as _skew_e
print("\nWind marginal shape (daylight hours):")
print(f"  {'':22}{'real':>10}{'Arm C':>10}{'Arm E':>10}")
_wc_e = np.vstack([t["wind"].to_numpy() for t in arcop_trajectories.values()])[:, _dmask_e]
for _lbl, _fn in [("skew", lambda x: _skew_e(np.ravel(x))), ("min", lambda x: float(np.min(x))),
                  ("frac < 0.5 m/s", lambda x: float((x < 0.5).mean()))]:
    print(f"  {_lbl:22}{_fn(_rw_e):>10.4f}{_fn(_wc_e):>10.4f}{_fn(_w_e):>10.4f}")

# hour-of-day amplitude of the simulated wind against the observed one
print("\nHour-of-day wind std amplitude (max/mean of std by hour), by season:")
print(f"  {'season':10}{'real':>10}{'Arm C':>10}{'Arm E':>10}")
_rs_season_e = np.array([get_season(m) for m in _rd_e.index.month])
_sim_season_e = np.array([get_season(m) for m in target_index[_dmask_e].month])
_hr_real_e = _rd_e.index.hour.to_numpy()
_hr_sim_e = target_index[_dmask_e].hour.to_numpy()
for _s in SEASONS:
    def _amp(vals, hrs):
        sd = pd.Series(np.ravel(vals)).groupby(np.tile(hrs, vals.shape[0]) if vals.ndim > 1 else hrs).std()
        return float(sd.max() / sd.mean())
    _mr = _rs_season_e == _s
    _ms = _sim_season_e == _s
    print(f"  {_s:10}{_amp(_rw_e[_mr], _hr_real_e[_mr]):>10.3f}"
          f"{_amp(_wc_e[:, _ms], _hr_sim_e[_ms]):>10.3f}{_amp(_w_e[:, _ms], _hr_sim_e[_ms]):>10.3f}")

# coverage of the extreme years, mirroring Section 11/31/42 exactly
_daily_e = {}
for _col in variables:
    _dly = pd.DataFrame(phys_batch_e[_col].T, index=target_index).resample("D").mean()
    _daily_e[_col] = _dly.to_numpy().T
    _day_index_e = _dly.index
if not (len(_day_index_e) == len(day_index) and (_day_index_e.dayofyear == day_index.dayofyear).all()):
    print("WARNING: Arm E day index differs from Section 11's - coverage not directly comparable")
_doy_e = _day_index_e.dayofyear

_cov_rows_e = []
for _yl, _ydf in extreme_years_to_plot.items():
    _yd = _ydf.resample("D").mean()
    _ydoy = _yd.index.dayofyear
    for _col in variables:
        _stack = _daily_e[_col]
        _lo = np.interp(_ydoy, _doy_e, np.percentile(_stack, 5, axis=0))
        _hi = np.interp(_ydoy, _doy_e, np.percentile(_stack, 95, axis=0))
        _rv = _yd[_col].to_numpy()
        _cov_rows_e.append({"extreme_year": _yl, "variable": _col, "model": "Arm E",
                            "coverage": ((_rv >= _lo) & (_rv <= _hi)).mean()})
arm_e_coverage = pd.DataFrame(_cov_rows_e)
print("\nCoverage of extreme years, all arms:")
_all_cov_e = pd.concat([combined_coverage_df, arm_d_coverage, arm_e_coverage], ignore_index=True)
print(_all_cov_e.pivot_table(index="variable", columns="model", values="coverage").round(3).to_string())

print("\nPer-year solar coverage (the result Arm D lost):")
print(arm_e_coverage[arm_e_coverage.variable == "solar"].set_index("extreme_year")["coverage"].round(3).to_string())

with open("./data/arm_e_phys_batch.pkl", "wb") as _f:
    pickle.dump({"phys_batch": phys_batch_e, "target_index": target_index,
                 "summary": arm_e_summary, "coverage": arm_e_coverage}, _f)
print("\nCached to data/arm_e_phys_batch.pkl")


In [ ]:
# 44. Model VAR + Copula: seasonal VAR across the three variables (its cross-lags carry the wind-temperature
#     coupling); R-vine copula on the innovations with empirical margins, irradiance at the root so that it can
#     be drawn sequentially; lag-1 temporal copula of irradiance with its parameter scaled by 2.5

import pyvinecopulib as pv
from scipy.stats import rankdata as _rankdata

VARCOP_SHRINK = 2.5
VARCOP_ORDER_GRID = [1, 2, 3, 6, 12, 24, 36, 48, 60, 72]
VARCOP_FAMILIES = [pv.BicopFamily.gaussian, pv.BicopFamily.student, pv.BicopFamily.clayton,
                    pv.BicopFamily.gumbel, pv.BicopFamily.frank, pv.BicopFamily.bb1,
                    pv.BicopFamily.joe]
_k2 = len(variables)
_SOL = variables.index("solar")
assert _SOL == _k2 - 1, "solar must be last so it lands in the vine's root position"
_EPS2 = 1e-6


def _svar_design(Y, block_id, p):
    n = len(Y)
    cols = []
    for lag in range(1, p + 1):
        sv = np.roll(Y, lag, axis=0)
        sb = np.roll(block_id, lag)
        ok = (block_id == sb) & (block_id != -1) & (np.arange(n) >= lag)
        cols.append(np.where(ok[:, None], sv, np.nan))
    X = np.column_stack([np.ones(n)] + cols)
    valid = (block_id != -1) & ~np.isnan(X).any(axis=1)
    return X[valid], Y[valid], valid


print("Section 44 (VAR + Copula): fitting seasonal VAR + innovation vines...")
_Y2 = df_anom[variables].to_numpy(dtype=float)
_fits2 = {}
for _s in SEASONS:
    _bid2 = season_block_ids(df_anom.index, _s)
    _best = None
    for _p in VARCOP_ORDER_GRID:
        _X, _y, _valid = _svar_design(_Y2, _bid2, _p)
        if len(_y) < 10 * (_k2 * _p + 1):
            continue
        _beta, *_ = np.linalg.lstsq(_X, _y, rcond=None)
        _res = _y - _X @ _beta
        _, _ld = np.linalg.slogdet(np.cov(_res.T, bias=True))
        _bic = len(_y) * _ld + (_k2 * _k2 * _p) * np.log(len(_y))
        if _best is None or _bic < _best["bic"]:
            _best = {"p": _p, "beta": _beta, "bic": _bic, "resid": _res,
                     "index": df_anom.index[_valid]}
    _f = _fits2[_s] = _best
    _R = _f["resid"]
    _n = len(_R)
    _U = np.asfortranarray(np.column_stack([
        np.clip(_rankdata(_R[:, i], method="average") / (_n + 1), _EPS2, 1 - _EPS2)
        for i in range(_k2)]))
    _f["vine"] = pv.Vinecop.from_data(data=_U, controls=pv.FitControlsVinecop(
        family_set=VARCOP_FAMILIES, selection_criterion="bic"))
    _f["margins"] = [np.sort(_R[:, i]) for i in range(_k2)]
    _f["grid"] = np.linspace(0.0, 1.0, _n)
    _idx2 = _f["index"]
    _gap = np.asarray((_idx2[1:] - _idx2[:-1]).total_seconds()) == 3600.0
    _us = _U[:, _SOL]
    _bc = pv.Bicop()
    _bc.select(data=np.asfortranarray(np.column_stack([_us[:-1][_gap], _us[1:][_gap]])),
               controls=pv.FitControlsBicop(family_set=VARCOP_FAMILIES, selection_criterion="bic"))
    _par = np.asarray(_bc.parameters).astype(float) * VARCOP_SHRINK
    _lb = np.asarray(_bc.parameters_lower_bounds).reshape(_par.shape)
    _ub = np.asarray(_bc.parameters_upper_bounds).reshape(_par.shape)
    _f["bc"] = pv.Bicop(family=_bc.family, rotation=_bc.rotation,
                        parameters=np.clip(_par, _lb, _ub))
    print(f"  {_s:8} VAR order={_f['p']:>3}  solar lag-1 tau (base)={_bc.tau:+.3f}")

# simulate
_slab2 = np.array([get_season(m) for m in target_index.month])
_T2 = len(target_index)
_maxp2 = max(_fits2[s]["p"] for s in SEASONS)

_usol = np.empty((N_TRAJECTORIES, _T2))
_wd = np.stack([np.random.default_rng(sd + 11_000_000).uniform(size=_T2) for sd in seeds])
_prev = None
for _t in range(_T2):
    _s = _slab2[_t]
    if _s != _prev:
        _usol[:, _t] = _wd[:, _t]
    else:
        _pair = np.column_stack([_wd[:, _t], _usol[:, _t - 1]])
        _usol[:, _t] = np.asarray(_fits2[_s]["bc"].hinv2(np.asfortranarray(_pair))).ravel()
    _prev = _s
_usol = np.clip(_usol, _EPS2, 1 - _EPS2)

_innov2 = np.empty((N_TRAJECTORIES, _T2, _k2))
_rng22 = np.random.default_rng(BASE_SEED + 12_000_000)
for _s in SEASONS:
    _m = _slab2 == _s
    _nm = int(_m.sum())
    _W = np.empty((N_TRAJECTORIES * _nm, _k2))
    _W[:, :_SOL] = _rng22.uniform(size=(N_TRAJECTORIES * _nm, _SOL))
    _W[:, _SOL] = _usol[:, _m].reshape(-1)
    _Uo = np.asarray(_fits2[_s]["vine"].inverse_rosenblatt(np.asfortranarray(_W)))
    assert np.allclose(_Uo[:, _SOL], _W[:, _SOL]), "solar did not pass through the vine root"
    for _i in range(_k2):
        _v = np.interp(np.clip(_Uo[:, _i], _EPS2, 1 - _EPS2),
                       _fits2[_s]["grid"], _fits2[_s]["margins"][_i])
        _innov2[:, _m, _i] = _v.reshape(N_TRAJECTORIES, _nm)

_sim2 = np.empty((N_TRAJECTORIES, _maxp2 + _T2, _k2))
_sim2[:, :_maxp2] = df_anom[variables].dropna().to_numpy(dtype=float)[-_maxp2:][None, :, :]
for _t in range(_T2):
    _i = _t + _maxp2
    _f = _fits2[_slab2[_t]]
    _X = np.concatenate([np.ones((N_TRAJECTORIES, 1))]
                        + [_sim2[:, _i - lag] for lag in range(1, _f["p"] + 1)], axis=1)
    _sim2[:, _i] = _X @ _f["beta"] + _innov2[:, _t]

phys_batch_varcop = {}
for _i, _c in enumerate(variables):
    _ph = _sim2[:, _maxp2:, _i] + target_mean_batch[_c]
    phys_batch_varcop[_c] = (np.clip(np.exp(_ph) - SOLAR_LOG_EPS, 0, None) if _c == "solar"
                         else wind_inv(_ph) if _c == "wind" else _ph)
varcop_trajectories = {j: pd.DataFrame({c: phys_batch_varcop[c][j] for c in variables},
                                        index=target_index) for j in range(N_TRAJECTORIES)}
print("VAR + Copula reconstructed:", {c: (round(float(phys_batch_varcop[c].mean()), 2),
                                      round(float(phys_batch_varcop[c].min()), 3)) for c in variables})

ARMS_ALL = {"AR": ar_trajectories, "VAR": var_trajectories,
           "AR+Copula": arcop_trajectories, "VAR + Copula": varcop_trajectories}
_dm2 = is_daylight(target_index)
_rd2 = df_train.loc[is_daylight(df_train.index)]
_rw2, _rs2 = _rd2["wind"].to_numpy(), _rd2["solar"].to_numpy()

print("\n" + "=" * 92)
print("A. DUNKELFLAUTE RATE AND MARGINALS")
print("=" * 92)
_rows = []
_pw_r, _ps_r = (_rw2 <= wind_thresh).mean(), (_rs2 <= solar_thresh).mean()
_rate_r = ((_rw2 <= wind_thresh) & (_rs2 <= solar_thresh)).mean()
_rows.append({"arm": "REAL", "P(low wind)": _pw_r, "P(low solar)": _ps_r,
              "rate": _rate_r, "multiplier": _rate_r / (_pw_r * _ps_r)})
for _a, _tr in ARMS_ALL.items():
    _w = np.vstack([t["wind"].to_numpy() for t in _tr.values()])[:, _dm2]
    _s_ = np.vstack([t["solar"].to_numpy() for t in _tr.values()])[:, _dm2]
    _pw, _ps = (_w <= wind_thresh).mean(), (_s_ <= solar_thresh).mean()
    _rt = ((_w <= wind_thresh) & (_s_ <= solar_thresh)).mean()
    _rows.append({"arm": _a, "P(low wind)": _pw, "P(low solar)": _ps,
                  "rate": _rt, "multiplier": _rt / (_pw * _ps)})
varcop_rate_df = pd.DataFrame(_rows).set_index("arm")
print(varcop_rate_df.round(4).to_string())
print(f"\nrate error vs real: " + "  ".join(
    f"{a}={100*(varcop_rate_df.loc[a,'rate']/_rate_r-1):+.0f}%" for a in ARMS_ALL))

print("\n" + "=" * 92)
print("B. COVERAGE OF THE EXTREME YEARS   (nominal for a 5-95% band is 0.90)")
print("=" * 92)
_cov_rows = []
for _a, _tr in ARMS_ALL.items():
    for _c in variables:
        _dly = pd.DataFrame(np.vstack([t[_c].to_numpy() for t in _tr.values()]).T,
                            index=target_index).resample("D").mean()
        _arr, _didx2 = _dly.to_numpy().T, _dly.index
        _cs = []
        for _lbl, _ydf in extreme_years_to_plot.items():
            _yd = _ydf.resample("D").mean()
            _lo = np.interp(_yd.index.dayofyear, _didx2.dayofyear, np.percentile(_arr, 5, axis=0))
            _hi = np.interp(_yd.index.dayofyear, _didx2.dayofyear, np.percentile(_arr, 95, axis=0))
            _rv = _yd[_c].to_numpy()
            _cs.append(((_rv >= _lo) & (_rv <= _hi)).mean())
        _cov_rows.append({"arm": _a, "variable": _c, "coverage": float(np.mean(_cs))})
varcop_coverage_df = pd.DataFrame(_cov_rows)
print(varcop_coverage_df.pivot_table(index="variable", columns="arm", values="coverage").round(3).to_string())

print("\n" + "=" * 92)
print("C. DUNKELFLAUTE EPISODE PERSISTENCE  (same day-spell definition as Section 41)")
print("=" * 92)


def _day_spells_44(flag):
    days = pd.DatetimeIndex(np.unique(flag.index[flag.to_numpy()].normalize()))
    if len(days) == 0:
        return np.array([]), 0
    gaps = np.asarray((days[1:] - days[:-1]).total_seconds()) != 86400.0
    breaks = np.concatenate([[0], np.where(gaps)[0] + 1, [len(days)]])
    return np.diff(breaks), len(days)


def _spell_stats(spells, ndays, nyears):
    if len(spells) == 0:
        return dict.fromkeys(["df days/yr", "spells/yr", "mean spell (d)",
                              "share >=2d", "share >=3d", "share >=5d", "max spell (d)"], 0.0)
    return {"df days/yr": ndays / nyears, "spells/yr": len(spells) / nyears,
            "mean spell (d)": spells.mean(), "share >=2d": (spells >= 2).mean(),
            "share >=3d": (spells >= 3).mean(), "share >=5d": (spells >= 5).mean(),
            "max spell (d)": float(spells.max())}


_rd_idx = df_train.index[is_daylight(df_train.index)]
_real_flag = pd.Series((_rw2 <= wind_thresh) & (_rs2 <= solar_thresh), index=_rd_idx)
_nyr_r = len(np.unique(_rd_idx.year))
_sp, _nd = _day_spells_44(_real_flag)
_prows = [dict(arm="REAL (2000-2024)", **_spell_stats(_sp, _nd, _nyr_r))]
_tidx = target_index[_dm2]
_nyr_s = max(len(np.unique(_tidx.year)), 1)
for _a, _tr in ARMS_ALL.items():
    _w = np.vstack([t["wind"].to_numpy() for t in _tr.values()])[:, _dm2]
    _s_ = np.vstack([t["solar"].to_numpy() for t in _tr.values()])[:, _dm2]
    _fl = (_w <= wind_thresh) & (_s_ <= solar_thresh)
    _acc = {}
    for _j in range(_fl.shape[0]):
        _st = _spell_stats(*_day_spells_44(pd.Series(_fl[_j], index=_tidx)), _nyr_s)
        for _kk, _vv in _st.items():
            _acc.setdefault(_kk, []).append(_vv)
    _prows.append(dict(arm=_a, **{k: float(np.mean(v)) for k, v in _acc.items()}))
varcop_persistence_df = pd.DataFrame(_prows).set_index("arm")
print(varcop_persistence_df.round(3).to_string())
_err = (varcop_persistence_df.drop(index="REAL (2000-2024)")
        - varcop_persistence_df.loc["REAL (2000-2024)"]).abs()
print("\nabsolute distance from real (smaller is better):")
print(_err.round(3).to_string())
print("\nclosest arm per statistic:")
for _c in _err.columns:
    print(f"  {_c:18} -> {_err[_c].idxmin()}")
print(f"\nVAR + Copula closest on {sum(1 for c in _err.columns if _err[c].idxmin()=='VAR + Copula')}"
      f" of {len(_err.columns)} statistics")

print("\n" + "=" * 92)
print("D. WIND-TEMPERATURE COUPLING (daily anomaly correlation) - the cross-variable coupling target")
print("=" * 92)
_clim2 = {c: df_train.groupby([df_train.index.month, df_train.index.hour])[c].mean() for c in variables}


def _an2(idx, vals, c):
    return vals - np.array([_clim2[c].get(k, np.nan) for k in zip(idx.month, idx.hour)])


def _blk2(x, kk):
    n = (len(x) // kk) * kk
    return x[:n].reshape(-1, kk).mean(axis=1) if kk > 1 else x


from scipy.stats import pearsonr as _pr
_rwa = _an2(df_train.index, df_train["wind"].to_numpy(), "wind")
_rta = _an2(df_train.index, df_train["temp"].to_numpy(), "temp")
_rsea = np.array([get_season(m) for m in df_train.index.month])
print(f"{'season':9}{'REAL':>10}" + "".join(f"{a:>13}" for a in ARMS_ALL))
for _s in SEASONS:
    _rm = _rsea == _s
    _sm = _slab2 == _s
    _line = f"{_s:9}{_pr(_blk2(_rwa[_rm],24), _blk2(_rta[_rm],24))[0]:>+10.3f}"
    for _a, _tr in ARMS_ALL.items():
        _vals = []
        for _j in range(min(40, N_TRAJECTORIES)):
            _wa = _an2(target_index, _tr[_j]["wind"].to_numpy(), "wind")[_sm]
            _ta = _an2(target_index, _tr[_j]["temp"].to_numpy(), "temp")[_sm]
            _vals.append(_pr(_blk2(_wa, 24), _blk2(_ta, 24))[0])
        _line += f"{np.mean(_vals):>+13.3f}"
    print(_line)

with open("./data/varcop_section44.pkl", "wb") as _f2:
    pickle.dump({"phys": phys_batch_varcop, "rate": varcop_rate_df,
                 "coverage": varcop_coverage_df, "persistence": varcop_persistence_df,
                 "shrink": VARCOP_SHRINK,
                 "orders": {s: _fits2[s]["p"] for s in SEASONS}}, _f2)
print("\nCached to data/varcop_section44.pkl")
